# Gestor de Variables – SESNA


## Módulo 1: Instalación y configuración

In [1]:
# ========================================================================
# MÓDULO 1: CONFIGURACIÓN, INSTALACIÓN DE DEPENDENCIAS, IMPORTS Y LOGGING
# ========================================================================

# ------------------------------------------------------------------
# 1. INSTALACIÓN INTELIGENTE DE DEPENDENCIAS (con caché)
# ------------------------------------------------------------------

import os
import time
import subprocess
import sys

def ensure_dependencies(force_reinstall=False, cache_file='.deps_installed', expire_seconds=86400):
    required_packages = [
        'flask-compress',
        'pyngrok',
        'supabase',
        'scikit-learn',
        'pandas',
        'numpy',
        'flask-login',
    ]

    if not force_reinstall and os.path.exists(cache_file):
        with open(cache_file, 'r') as f:
            last_install = float(f.read().strip())
        if time.time() - last_install < expire_seconds:
            print(f"✅ Dependencias ya instaladas (última instalación hace menos de {expire_seconds//3600} horas).")
            return

    missing = []
    for pkg in required_packages:
        import_name = pkg.replace('-', '_')
        try:
            __import__(import_name)
        except ImportError:
            missing.append(pkg)

    if not missing and not force_reinstall:
        print("✅ Todas las dependencias ya están instaladas.")
        with open(cache_file, 'w') as f:
            f.write(str(time.time()))
        return

    to_install = required_packages if force_reinstall else missing
    if to_install:
        print(f"📦 Instalando dependencias: {', '.join(to_install)} ...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + to_install)
        print("✅ Instalación completada.")
        with open(cache_file, 'w') as f:
            f.write(str(time.time()))
    else:
        print("✅ Todas las dependencias ya están instaladas.")

ensure_dependencies()

# ------------------------------------------------------------------
# 2. IMPORTS
# ------------------------------------------------------------------

import csv
import datetime
import functools
import hashlib
import importlib
import io
import json
import logging
import os
import pickle
import re
import shutil
import socket
import tempfile
import threading
import time
import traceback
import unicodedata
import uuid
import zipfile
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple, Union

import numpy as np
import pandas as pd
import concurrent.futures
from difflib import SequenceMatcher
from flask import Flask, jsonify, make_response, render_template_string, request, redirect, abort
from flask_compress import Compress
from flask_login import LoginManager, UserMixin, login_user, logout_user, login_required, current_user
from werkzeug.security import generate_password_hash, check_password_hash
from google.colab import userdata
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from supabase import create_client, Client
from functools import partial

# ------------------------------------------------------------------
# 3. CONFIGURACIÓN DE LOGGING
# ------------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# ------------------------------------------------------------------
# 4. INICIALIZACIÓN DE SUPABASE (SOLO ANON KEY - SEGURO)
# ------------------------------------------------------------------

supabase = None
try:
    SUPABASE_URL = userdata.get('SUPABASE_URL').strip()
    SUPABASE_KEY = userdata.get('SUPABASE_KEY').strip()
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    logger.info("✅ Cliente de Supabase inicializado correctamente (anon key).")
except Exception as e:
    logger.error(f"❌ Error al inicializar Supabase: {e}")
    raise RuntimeError("No se pudo conectar a Supabase. Verifica tus credenciales.") from e

# ------------------------------------------------------------------
# 5. CONFIGURACIÓN DE AUTENTICACIÓN (Flask-Login) con ROLES
# ------------------------------------------------------------------

DISABLE_AUTH = os.environ.get('DISABLE_AUTH', 'false').lower() == 'true'

class User(UserMixin):
    def __init__(self, id, username, password_hash, rol='user'):
        self.id = id
        self.username = username
        self.password_hash = password_hash
        self.rol = rol

    @staticmethod
    def get(user_id):
        try:
            res = supabase.from_('usuarios').select('*').eq('id', user_id).execute()
            if res.data:
                u = res.data[0]
                return User(u['id'], u['username'], u['password_hash'], u.get('rol', 'user'))
        except:
            pass
        return None

    @staticmethod
    def find_by_username(username):
        try:
            res = supabase.from_('usuarios').select('*').eq('username', username).execute()
            if res.data:
                u = res.data[0]
                return User(u['id'], u['username'], u['password_hash'], u.get('rol', 'user'))
        except:
            pass
        return None

    def check_password(self, password):
        return check_password_hash(self.password_hash, password)

    def is_admin(self):
        return self.rol == 'admin'

# Crear usuario administrador por defecto si no existe ninguno
def ensure_admin_user():
    if DISABLE_AUTH:
        logger.info("Autenticación deshabilitada por variable de entorno.")
        return
    try:
        res = supabase.from_('usuarios').select('count', count='exact').execute()
        if res.count == 0:
            hashed = generate_password_hash('admin123')
            supabase.from_('usuarios').insert({
                'username': 'admin',
                'password_hash': hashed,
                'rol': 'admin'
            }).execute()
            logger.info("✅ Usuario administrador creado: admin / admin123 (cambia la contraseña).")
    except Exception as e:
        logger.error(f"❌ Error al verificar/crear usuario admin: {e}")

ensure_admin_user()


✅ Dependencias ya instaladas (última instalación hace menos de 24 horas).


## Módulo 2: Constantes y keywords

In [2]:
# ============================================================
# MÓDULO 2: CONSTANTES Y KEYWORDS
# ============================================================

COLUMN_DISPLAY_NAMES: List[str] = [
    "ID", "Proceso", "Eje", "Tema", "Nombre",
    "Institución", "Cobertura", "Periodicidad", "Liga Web",
    "Fuente", "Año", "Estado", "Valor"
]

EXPECTED_COLUMNS: List[str] = [
    'id', 'proceso', 'eje', 'tema', 'nombre',
    'institucion', 'cobertura', 'periodicidad', 'liga_web',
    'fuente', 'año', 'estado', 'valor'
]

DYNAMIC_OPTIONS_COLUMNS: List[str] = [
    "proceso", "eje", "tema",
    "institucion", "cobertura", "periodicidad",
    "estado"
]

CODIGO_ESTADO_A_NOMBRE: Dict[str, str] = {
    '01': 'Aguascalientes', '02': 'Baja California', '03': 'Baja California Sur',
    '04': 'Campeche', '05': 'Coahuila de Zaragoza', '06': 'Colima',
    '07': 'Chiapas', '08': 'Chihuahua', '09': 'Ciudad de México',
    '10': 'Durango', '11': 'Guanajuato', '12': 'Guerrero',
    '13': 'Hidalgo', '14': 'Jalisco', '15': 'México',
    '16': 'Michoacán de Ocampo', '17': 'Morelos', '18': 'Nayarit',
    '19': 'Nuevo León', '20': 'Oaxaca', '21': 'Puebla',
    '22': 'Querétaro', '23': 'Quintana Roo', '24': 'San Luis Potosí',
    '25': 'Sinaloa', '26': 'Sonora', '27': 'Tabasco',
    '28': 'Tamaulipas', '29': 'Tlaxcala', '30': 'Veracruz de Ignacio de la Llave',
    '31': 'Yucatán', '32': 'Zacatecas'
}

KEYWORDS: Dict[str, Dict[str, List[str]]] = {
    'proceso': {
        'A. Prevención': [
            'existencia', 'mecanismos', 'cobertura', 'plan', 'programa',
            'profesionalización', 'capacitación', 'servicio civil', 'reglas',
            'lineamientos', 'diseño', 'atributos', 'calidad', 'indicadores',
            'objetivos', 'metas', 'participación ciudadana', 'contraloría social',
            'educación', 'campañas', 'comunicaciones', 'código de ética',
            'declaración patrimonial', 'evaluación de control y confianza',
            'personal', 'recursos humanos'
        ],
        'B. Detección': [
            'percibe', 'percepción', 'frecuencia', 'prevalencia', 'incidencia',
            'experimentaron', 'usuarios', 'unidades económicas', 'denuncias recibidas',
            'carpetas de investigación', 'averiguaciones previas', 'quejas recibidas',
            'actos de corrupción reportados', 'conocimiento por terceros',
            'inconformidades', 'observaciones', 'investigación iniciada', 'víctimas',
            'imputados', 'inculpados', 'causas penales ingresadas'
        ],
        'C. Sanción': [
            'sanciones', 'condenatorias', 'absolutorias', 'multas', 'inhabilitación',
            'destitución', 'amonestación', 'resarcitoria', 'pliego de observaciones',
            'procedimientos de responsabilidad', 'sentencias', 'fincamiento',
            'sanciones económicas', 'revocación', 'castigo', 'ejecución de sentencia',
            'cumplimiento de órdenes'
        ],
        'D. Fiscalización y control de recursos': [
            'auditoría', 'fiscalización', 'control interno', 'observaciones',
            'presupuesto', 'recursos', 'contratos', 'monto', 'donativos', 'compras',
            'obra pública', 'licitaciones', 'convocatorias', 'adjudicaciones',
            'proveedores', 'contratistas', 'cuenta pública', 'armonización contable',
            'estado de situación financiera', 'ingresos', 'egresos', 'fideicomisos'
        ]
    },
    'eje': {
        '1. Combatir la corrupción y la impunidad': [
            'quejas', 'denuncias', 'faltas administrativas', 'justicia', 'delitos',
            'cohecho', 'peculado', 'enriquecimiento', 'abuso de autoridad',
            'ejercicio indebido', 'procuración', 'impartición', 'ministerio público',
            'causas penales', 'sentencias', 'sanciones', 'responsabilidad administrativa',
            'investigación de servidores', 'tribunales', 'jueces', 'magistrados',
            'fiscalía', 'averiguación previa', 'carpeta de investigación'
        ],
        '2. Combatir la arbitrariedad y el abuso de poder': [
            'profesionalización', 'servicio civil', 'carrera', 'capacitación',
            'evaluación de desempeño', 'declaración patrimonial', 'planes anticorrupción',
            'programas sociales', 'procesos institucionales', 'indicadores',
            'reglas de operación', 'riesgos', 'auditoría', 'fiscalización',
            'control interno', 'obra pública', 'contrataciones', 'adquisiciones',
            'arrendamientos', 'proveedores', 'contratistas', 'testigos sociales',
            'planeación', 'programación', 'presupuestación', 'seguimiento', 'evaluación',
            'marco jurídico', 'transparencia', 'datos abiertos', 'rendición de cuentas'
        ],
        '3. Promover la mejora de la gestión y los puntos de contacto gobierno-sociedad': [
            'trámites', 'servicios públicos', 'puntos de contacto', 'ciudadanía',
            'iniciativa privada', 'licitaciones', 'convocatorias', 'permisos',
            'pagos', 'predial', 'tenencia', 'atención ciudadana',
            'solicitudes de información', 'oficina de transparencia',
            'módulos de orientación', 'infracciones', 'licencias de conducir',
            'registro civil', 'afiliación', 'consulta médica', 'inscripción escolar',
            'compras del gobierno', 'contratación pública', 'MIPYMES',
            'competencia económica', 'prácticas monopólicas'
        ],
        '4. Involucrar a la sociedad y el sector privado': [
            'participación ciudadana', 'contraloría social', 'vigilancia',
            'colaboración', 'cocreación', 'educación', 'campañas anticorrupción',
            'comunicación', 'integridad empresarial', 'código de ética', 'compliance',
            'denuncias corporativas', 'testigos sociales', 'observadores',
            'organizaciones civiles', 'ONG', 'sindicatos', 'medios de comunicación',
            'redes sociales', 'parlamento abierto', 'gobierno abierto',
            'consulta ciudadana', 'presupuesto participativo'
        ]
    },
    'tema': {
        '1.1. Prevención, detección, denuncia, investigación, substanciación y sanción de faltas administrativas': [
            'quejas', 'denuncias', 'oficina de contraloría', 'procedimientos de responsabilidad',
            'declaración patrimonial', 'conflictos de interés', 'sistema informático para quejas',
            'buzón de quejas', 'atención ciudadana', 'faltas administrativas',
            'sanciones administrativas', 'amonestación', 'suspensión', 'destitución',
            'inhabilitación', 'responsabilidad resarcitoria'
        ],
        '1.2. Procuración e impartición de justicia en materia de delitos por hechos de corrupción': [
            'delitos', 'cohecho', 'peculado', 'enriquecimiento ilícito', 'tráfico de influencias',
            'abuso de autoridad', 'ejercicio indebido', 'carpetas de investigación',
            'averiguaciones previas', 'causas penales', 'sentencias condenatorias',
            'sentencias absolutorias', 'jueces', 'magistrados', 'ministerio público',
            'fiscalía', 'imputados', 'víctimas'
        ],
        '2.1. Profesionalización e integridad en el servicio público': [
            'servicio civil', 'carrera', 'capacitación', 'evaluación de control y confianza',
            'desempeño', 'competencias', 'personal', 'licenciatura', 'recursos humanos',
            'profesionalización', 'integridad', 'código de ética', 'declaración patrimonial',
            'evaluación de desempeño'
        ],
        '2.2. Procesos institucionales': [
            'plan anticorrupción', 'programas sociales', 'riesgos', 'indicadores',
            'metas', 'cobertura', 'desempeño', 'presupuesto', 'reglas de operación',
            'lineamientos', 'diseño', 'calidad', 'cumplimiento de metas', 'cuadrante',
            'targeting', 'instrumentos de medición', 'panel de control', 'seguimiento',
            'evaluación de diseño', 'consistencia y resultados'
        ],
        '2.3. Auditoría y fiscalización': [
            'auditoría', 'fiscalización', 'control interno', 'observaciones', 'revisión',
            'órgano de control', 'visitaduría', 'entidad de fiscalización',
            'armonización contable', 'ejercicio y control', 'seguimiento', 'evaluación',
            'indicadores de resultados', 'marco jurídico', 'planeación', 'programación',
            'presupuestación'
        ],
        '3.1. Puntos de contacto gobierno-ciudadanía: trámites, servicios y programas públicos': [
            'trámites', 'servicios', 'solicitudes de información', 'atención ciudadana',
            'pagos', 'predial', 'tenencia', 'permisos', 'construcción', 'licencias',
            'usuarios', 'confianza en información', 'módulos de orientación',
            'oficina de transparencia', 'solicitudes de acceso a la información',
            'datos abiertos', 'quejas de servicios', 'contacto con autoridades'
        ],
        '3.2. Puntos de contacto gobierno-iniciativa privada': [
            'contrataciones', 'licitaciones', 'obra pública', 'proveedores',
            'contratistas', 'MIPYMES', 'testigos sociales', 'convocatorias',
            'adquisiciones', 'arrendamientos', 'servicios relacionados',
            'prácticas monopólicas', 'competencia', 'registro de contratistas',
            'sistema electrónico de contrataciones', 'invitación restringida',
            'adjudicación directa'
        ],
        '4.1. Participación ciudadana: vigilancia, colaboración y cocreación': [
            'contraloría social', 'órganos de participación', 'consulta ciudadana',
            'vigilancia', 'presupuesto participativo', 'parlamento abierto',
            'gobierno abierto', 'mecanismos de participación', 'comités de contraloría',
            'consejos ciudadanos', 'rendición de cuentas', 'espacios de participación'
        ],
        '4.2. Corresponsabilidad e integridad empresarial': [
            'empresa', 'integridad', 'código de ética', 'compliance', 'proveedores',
            'denuncias corporativas', 'programa de integridad', 'anticorrupción',
            'políticas de sobornos', 'conflicto de intereses', 'transparencia corporativa'
        ],
        '4.3. Educación y comunicación para el control de la corrupción': [
            'educación', 'campañas', 'conocimiento cívico', 'estudiantes',
            'comunicación', 'redes sociales', 'medios de comunicación', 'divulgación',
            'sensibilización', 'normalización de la corrupción', 'cultura de la legalidad',
            'formación'
        ]
    }
}

## Módulo 3: Funciones auxiliares básicas

In [3]:
# ============================================================
# MÓDULO 3: FUNCIONES AUXILIARES BÁSICAS
# ============================================================

# Cache TTL para datos de Supabase
CACHE_TTL: int = 60
_cache_timestamp: float = 0
_existing_data_cache: Optional[List[Dict]] = None
_existing_ids_cache: Optional[Set[str]] = None
_MAX_ID_CACHE: Optional[int] = None

# ------------------------------------------------------------------
# Funciones de limpieza y caché
# ------------------------------------------------------------------

@functools.lru_cache(maxsize=1024)
def remover_acentos_cached(s: str) -> str:
    """Elimina acentos de una cadena usando caché.

    Args:
        s: Cadena de entrada.

    Returns:
        Cadena sin acentos.
    """
    if not s:
        return ""
    return "".join(c for c in unicodedata.normalize('NFKD', str(s))
                   if not unicodedata.combining(c))


def remover_acentos(s: str) -> str:
    """Wrapper de remover_acentos_cached."""
    return remover_acentos_cached(s)


def safe_clean(v: Any) -> str:
    """Limpia un valor convirtiéndolo a cadena y eliminando saltos de línea.

    Args:
        v: Valor a limpiar (puede ser None, NaN, numérico o cadena).

    Returns:
        Cadena limpia sin saltos de línea ni espacios extra.
    """
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    s = str(v).replace('\n', ' ').replace('\r', ' ').strip()
    s = re.sub(r'\s+', ' ', s)
    return s


def _clean_data_for_json(data: Any) -> Any:
    """Convierte datos a tipos serializables por JSON.

    Args:
        data: Diccionario, lista o valor nativo de Python/Pandas/Numpy.

    Returns:
        Datos con tipos nativos de Python.
    """
    if isinstance(data, dict):
        return {k: _clean_data_for_json(v) for k, v in data.items()}
    if isinstance(data, list):
        return [_clean_data_for_json(elem) for elem in data]
    if isinstance(data, (np.integer, np.int64, np.int32)):
        return int(data)
    if isinstance(data, (np.floating, np.float64, np.float32)):
        return float(data)
    if isinstance(data, np.bool_):
        return bool(data)
    if pd.isna(data):
        return None
    return data


def _refresh_caches(force: bool = False) -> None:
    """Refresca las cachés de Supabase si han expirado o se fuerza.

    Args:
        force: Si es True, actualiza aunque no haya expirado.
    """
    global _existing_data_cache, _existing_ids_cache, _cache_timestamp
    now = time.time()
    if not force and _existing_data_cache is not None and (now - _cache_timestamp) < CACHE_TTL:
        return

    try:
        res = supabase.from_('variables').select('*').execute()
        if res.data:
            _existing_data_cache = res.data
            _existing_ids_cache = {str(item['id']) for item in res.data}
        else:
            _existing_data_cache = []
            _existing_ids_cache = set()
        _cache_timestamp = now
    except Exception as e:
        logger.error(f"Error al refrescar caches: {e}")
        _existing_data_cache = []
        _existing_ids_cache = set()
        _cache_timestamp = 0


def _get_all_existing_ids() -> Set[str]:
    """Obtiene el conjunto de IDs existentes en la tabla 'variables'.

    Returns:
        Conjunto de IDs (strings).
    """
    global _existing_ids_cache
    if _existing_ids_cache is None:
        _refresh_caches()
    return _existing_ids_cache or set()


def _get_all_variables_data() -> List[Dict]:
    """Obtiene todos los datos de la tabla 'variables' normalizados.

    Returns:
        Lista de diccionarios con los datos normalizados.
    """
    global _existing_data_cache
    if _existing_data_cache is None:
        _refresh_caches()
    if not _existing_data_cache:
        return []
    data = []
    for item in _existing_data_cache:
        row = {}
        for k, v in item.items():
            if k in ['año', 'valor']:
                v = normalizar_numero_texto(v, k)
            row[k] = safe_clean(v)
        data.append(row)
    return data

def _compute_row_hash_vectorized(df: pd.DataFrame) -> pd.Series:
    """
    Calcula el hash (_get_non_id_hash) para todas las filas de forma vectorizada,
    incluyendo parent_id si existe en el DataFrame.
    """
    cols = [c for c in EXPECTED_COLUMNS if c != 'id']
    # Añadir parent_id si está presente
    if 'parent_id' in df.columns and 'parent_id' not in cols:
        cols.append('parent_id')
    # Asegurar que todas las columnas existan
    for c in cols:
        if c not in df.columns:
            df[c] = ''

    # Normalización previa de columnas numéricas y especiales
    for col in ['version', 'año']:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: normalizar_numero_texto(x, col))
    if 'valor' in df.columns:
        def norm_valor(v):
            temp = normalizar_numero_texto(v, 'valor')
            if temp.upper() in VALORES_ESPECIALES:
                return temp
            try:
                return f"{float(temp):.1f}"
            except:
                return ''
        df['valor'] = df['valor'].apply(norm_valor)
    # Normalizar parent_id (convertir a string, manejar None)
    if 'parent_id' in df.columns:
        df['parent_id'] = df['parent_id'].apply(lambda x: str(x) if pd.notna(x) else None)

    # Aplicar safe_clean a todas las columnas (vectorizado con str)
    for c in cols:
        df[c] = df[c].astype(str).str.replace('\n', ' ', regex=False).str.replace('\r', ' ', regex=False).str.strip()
        df[c] = df[c].str.replace(r'\s+', ' ', regex=True)

    # Generar tupla por fila (operación rápida en C)
    return df[cols].apply(tuple, axis=1)

def _get_non_id_hash(row_dict: Dict) -> Tuple:
    """Calcula un hash (tupla) de una fila excluyendo el campo 'id', pero incluyendo 'parent_id'."""
    # Definir las claves relevantes: todas las columnas esperadas + parent_id (si existe)
    relevant_keys = [k for k in EXPECTED_COLUMNS if k != 'id']
    if 'parent_id' in row_dict:
        relevant_keys.append('parent_id')
    # Ordenar para consistencia
    relevant_keys = sorted(relevant_keys)
    values = []
    for key in relevant_keys:
        val = row_dict.get(key, '')
        if pd.isna(val):
            val = ''
        if key in ['version', 'año']:
            val = normalizar_numero_texto(val, key)
        elif key == 'valor':
            temp = normalizar_numero_texto(val, key)
            if temp.upper() in VALORES_ESPECIALES:
                val = temp
            else:
                try:
                    num = float(temp)
                    val = f"{num:.1f}"
                except ValueError:
                    val = ''
        elif key == 'parent_id':
            # Asegurar que parent_id se incluya como string o None
            val = str(val) if val is not None else None
        val = safe_clean(val)
        values.append(val)
    return tuple(values)

_id_generation_lock = threading.Lock()
def generar_siguiente_id(used_ids: Optional[Set[str]] = None) -> str:
    """Genera el siguiente ID único con formato 'A-XXXXX' (thread-safe)."""
    global _MAX_ID_CACHE
    with _id_generation_lock:
        if _MAX_ID_CACHE is None:
            try:
                res = supabase.from_('variables').select('id').limit(100000).execute()
                ids = [item['id'] for item in res.data if item['id'] and item['id'].startswith('A-')]
                nums = [int(re.search(r'A-(\d+)', id_str).group(1)) for id_str in ids
                        if re.search(r'A-(\d+)', id_str)]
                _MAX_ID_CACHE = max(nums) if nums else 0
            except Exception:
                _MAX_ID_CACHE = 0

        new_num = _MAX_ID_CACHE + 1
        new_id = f"A-{new_num:05d}"
        while used_ids and new_id in used_ids:
            new_num += 1
            new_id = f"A-{new_num:05d}"
        _MAX_ID_CACHE = new_num
        return new_id


def _normalize_col_for_matching(col_name: str) -> str:
    """Normaliza un nombre de columna para comparación.

    Args:
        col_name: Nombre de columna.

    Returns:
        Nombre normalizado (sin acentos, minúsculas, guiones bajos).
    """
    return remover_acentos(str(col_name).lower().replace(" ", "_")).replace("-", "_")


def _find_best_column_match(
    uploaded_col_normalized: str,
    available_expected_cols_normalized: List[str],
    threshold: float = 0.8
) -> Optional[str]:
    """Encuentra la mejor coincidencia entre un nombre subido y los esperados.

    Args:
        uploaded_col_normalized: Nombre normalizado de la columna subida.
        available_expected_cols_normalized: Lista de nombres esperados normalizados.
        threshold: Umbral de similitud (0-1).

    Returns:
        Mejor coincidencia o None.
    """
    best_match, highest_score = None, threshold
    for expected_col_norm in available_expected_cols_normalized:
        score = SequenceMatcher(None, uploaded_col_normalized, expected_col_norm).ratio()
        if score > highest_score:
            highest_score, best_match = score, expected_col_norm
    return best_match


_catalog_cache: Dict[str, List[str]] = {}


def _get_catalog_options(column_key: str) -> List[str]:
    """Obtiene las opciones de un catálogo (tabla) de Supabase con caché.

    Args:
        column_key: Nombre de la tabla de catálogo.

    Returns:
        Lista de nombres de opciones.
    """
    if column_key in _catalog_cache:
        return _catalog_cache[column_key]
    try:
        res = supabase.from_(column_key).select('name').execute()
        options = sorted({r['name'] for r in res.data if r.get('name')}) if res.data else []
        _catalog_cache[column_key] = options
        return options
    except Exception as e:
        logger.error(f"Error al obtener opciones para '{column_key}': {e}")
        return []


def _find_best_value_match(
    input_value: str,
    available_options: List[str],
    threshold: float = 0.8
) -> Tuple[Optional[str], float]:
    """Encuentra la mejor coincidencia para un valor dentro de una lista.

    Args:
        input_value: Valor de entrada.
        available_options: Lista de opciones disponibles.
        threshold: Umbral de similitud (0-1).

    Returns:
        Tupla (mejor_opción, puntuación) o (None, 0.0).
    """
    if not input_value or not available_options:
        return None, 0.0
    input_norm = remover_acentos(str(input_value).lower())
    best_match_val, highest_score = None, 0.0
    for option_original in available_options:
        option_norm = remover_acentos(str(option_original).lower())
        score = SequenceMatcher(None, input_norm, option_norm).ratio()
        if score > highest_score:
            highest_score, best_match_val = score, option_original
    return (best_match_val, highest_score) if highest_score >= threshold else (None, highest_score)


def _find_disaggregation_catalog(col_name: str, dict_dir: Union[str, Path]) -> Optional[Dict[str, str]]:
    """
    Busca un archivo CSV de catálogo para una columna de desagregación.

    Args:
        col_name: Nombre de la columna (ej. 'cve_ent').
        dict_dir: Directorio donde buscar catálogos (string o Path).

    Returns:
        Diccionario {clave: descripción} o None.
    """
    # Convertir a Path si es string
    if isinstance(dict_dir, str):
        dict_dir = Path(dict_dir)

    if not col_name or not dict_dir.is_dir():
        logger.warning(f"❌ col_name vacío o {dict_dir} no es directorio")
        return None

    col_name_lower = col_name.lower()
    logger.info(f"🔍 Buscando catálogo para '{col_name}' en {dict_dir}")

    for f in dict_dir.glob('*.csv'):
        base = f.stem
        if base.lower().startswith(col_name_lower):
            logger.info(f"📄 Archivo candidato: {f}")
            try:
                df_cat = pd.read_csv(f, encoding='utf-8-sig')
            except UnicodeDecodeError:
                try:
                    df_cat = pd.read_csv(f, encoding='latin-1')
                except Exception as e:
                    logger.warning(f"⚠️ No se pudo leer {f}: {e}")
                    continue
            if len(df_cat.columns) >= 2:
                key_col = df_cat.columns[0]
                desc_col = df_cat.columns[1]
                catalog = {}
                for _, row in df_cat.iterrows():
                    k = str(row[key_col]).strip()
                    v = str(row[desc_col]).strip()
                    catalog[k] = v
                logger.info(f"✅ Catálogo cargado: {f.name} -> {len(catalog)} entradas.")
                return catalog
            logger.warning(f"⚠️ El archivo {f.name} tiene menos de 2 columnas.")
    logger.warning(f"❌ No se encontró catálogo para '{col_name}' en {dict_dir}")
    return None

## Módulo 4: Normalización de números y catálogos

In [4]:
# ============================================================
# MÓDULO 4: NORMALIZACIÓN DE NÚMEROS Y CATÁLOGOS
# ============================================================

VALORES_ESPECIALES: Set[str] = {"NSS", "NA", "No aplica"}


def normalizar_numero_texto(
    valor: Any,
    columna: Optional[str] = None,
    for_display: bool = False
) -> Union[str, float]:
    """Normaliza un valor según la columna, respetando valores especiales.

    Args:
        valor: Valor a normalizar.
        columna: Nombre de la columna ('año', 'version', 'valor').
        for_display: Si es True, devuelve el valor sin formatear (para mostrar).

    Returns:
        Valor normalizado (str o número).
    """
    if valor is None:
        return ""
    if isinstance(valor, str):
        stripped = valor.strip()
        if stripped.upper() in VALORES_ESPECIALES:
            return stripped
        try:
            num = float(stripped.replace(',', ''))
            if columna in ('año', 'version'):
                return str(int(num))
            if columna == 'valor':
                return f"{num:.1f}" if not for_display else str(num)
            return stripped
        except ValueError:
            return stripped
    if isinstance(valor, (int, float)) and not pd.isna(valor):
        if columna in ('año', 'version'):
            return str(int(valor))
        if columna == 'valor':
            return f"{float(valor):.1f}" if not for_display else str(valor)
        return str(valor)
    try:
        return str(valor)
    except Exception:
        return ""


def safe_normalize(val: Any, col_name: str) -> str:
    """Envuelve normalizar_numero_texto manejando series y arrays."""
    if isinstance(val, pd.Series):
        val = val.iloc[0] if len(val) > 0 else ''
    if isinstance(val, (list, np.ndarray)):
        val = val[0] if len(val) > 0 else ''
    if pd.isna(val):
        val = ''
    try:
        return normalizar_numero_texto(val, col_name)
    except Exception:
        return str(val) if val is not None else ''


def normalizar_valor_con_catalogo(valor: str, tabla: str, threshold: float = 0.5) -> str:
    """Normaliza un valor usando el catálogo correspondiente (inserta si no existe).

    Args:
        valor: Valor a normalizar.
        tabla: Nombre de la tabla de catálogo.
        threshold: Umbral de similitud.

    Returns:
        Valor normalizado (canónico) o el original.
    """
    if not valor or not isinstance(valor, str):
        return valor
    valor_limpio = safe_clean(valor)
    if not valor_limpio or valor_limpio == "No aplica":
        return valor_limpio

    sinonimos = {
        'institucion': {
            'inegi': 'Instituto Nacional de Estadística y Geografía',
            'secretaria de educacion': 'Secretaría de Educación Pública',
            'sep': 'Secretaría de Educación Pública',
        }
    }
    if tabla in sinonimos:
        valor_lower = valor_limpio.lower()
        for clave, canonico in sinonimos[tabla].items():
            if clave in valor_lower:
                return canonico

    opciones = _get_catalog_options(tabla)
    if not opciones:
        try:
            res = supabase.from_(tabla).insert({"name": valor_limpio}).execute()
            if res.data:
                _catalog_cache[tabla] = sorted(_catalog_cache.get(tabla, []) + [valor_limpio])
                return valor_limpio
        except Exception:
            pass
        return valor

    valor_norm = remover_acentos(valor_limpio.lower().strip())
    for opt in opciones:
        if remover_acentos(opt.lower().strip()) == valor_norm:
            return opt

    match, score = _find_best_value_match(valor_limpio, opciones, threshold)
    if match:
        return match

    try:
        res = supabase.from_(tabla).insert({"name": valor_limpio}).execute()
        if res.data:
            _catalog_cache[tabla] = sorted(_catalog_cache.get(tabla, []) + [valor_limpio])
            return valor_limpio
    except Exception as e:
        logger.error(f"Error al insertar '{valor_limpio}' en tabla '{tabla}': {e}")
    return valor


def normalizar_categoria(
    valor: str,
    tipo: str,
    debug: bool = False,
    threshold: float = 0.8
) -> Tuple[str, bool]:
    """Normaliza una categoría (proceso, eje, tema) usando keywords y difusa.

    Args:
        valor: Valor de entrada.
        tipo: 'proceso', 'eje' o 'tema'.
        debug: Si es True, imprime información de depuración.
        threshold: Umbral de similitud.

    Returns:
        Tupla (valor_normalizado, bool_indica_si_fue_normalizado).
    """
    if not valor:
        return valor, False
    valor_limpio = safe_clean(valor)
    if not valor_limpio:
        return valor, False

    if tipo == 'proceso':
        opciones = list(KEYWORDS['proceso'].keys())
        palabras_por_opcion = KEYWORDS['proceso']
    elif tipo == 'eje':
        opciones = list(KEYWORDS['eje'].keys())
        palabras_por_opcion = KEYWORDS['eje']
    elif tipo == 'tema':
        opciones = list(KEYWORDS['tema'].keys())
        palabras_por_opcion = KEYWORDS['tema']
    else:
        return valor, False

    valor_norm = remover_acentos(valor_limpio.lower().strip())

    for opt in opciones:
        if remover_acentos(opt.lower().strip()) == valor_norm:
            if debug:
                logger.info(f"✅ Normalizado {tipo}: '{valor}' -> '{opt}'")
            return opt, True

    def texto_contiene_palabra_clave(texto: str, lista_palabras: List[str]) -> bool:
        texto_norm = remover_acentos(texto.lower())
        for palabra in lista_palabras:
            if remover_acentos(palabra.lower()) in texto_norm:
                return True
        return False

    for categoria, palabras in palabras_por_opcion.items():
        if texto_contiene_palabra_clave(valor_limpio, palabras):
            if debug:
                logger.info(f"✅ Normalizado {tipo} por keyword: '{valor}' -> '{categoria}'")
            return categoria, True

    best_match = None
    best_score = 0.0
    for opt in opciones:
        score = SequenceMatcher(
            None,
            remover_acentos(valor_limpio.lower()),
            remover_acentos(opt.lower())
        ).ratio()
        if score > best_score:
            best_score = score
            best_match = opt
    if best_score >= threshold:
        if debug:
            logger.info(f"⚠️ Normalizado {tipo} (difuso): '{valor}' -> '{best_match}' (score {best_score:.2f})")
        return best_match, True

    if debug:
        logger.info(f"❌ No se pudo normalizar {tipo}: '{valor}'")
    return valor, False


def to_numeric_clean(series: pd.Series, log_messages: Optional[List[str]] = None) -> pd.Series:
    """Convierte una serie a numérico, tratando NSS/NA como NaN.

    Args:
        series: Serie a convertir.
        log_messages: Lista para acumular logs.

    Returns:
        Serie numérica (con NaN para no numéricos).
    """
    if log_messages is None:
        log_messages = []
    if series.dtype == object:
        cleaned = series.astype(str).str.strip()

        def clean_number(val: str) -> float:
            if not val or val == '':
                return np.nan
            val = val.replace(' ', '')
            if val.upper() in VALORES_ESPECIALES:
                return np.nan
            try:
                return float(val)
            except ValueError:
                return np.nan

        cleaned = cleaned.apply(clean_number)
        numeric = pd.to_numeric(cleaned, errors='coerce')
        nan_count = numeric.isna().sum()
        if nan_count > 0:
            log_messages.append(
                f"⚠️ {nan_count} valores no numéricos en la columna (ej: {series.head(3).tolist()})"
            )
        return numeric
    return pd.to_numeric(series, errors='coerce')

## Módulo 5: Procesamiento de CSV

In [5]:
# ============================================================
# MÓDULO 5: PROCESAMIENTO DE CSV (VERSIÓN OPTIMIZADA CON VECTORIZACIÓN)
# ============================================================

import functools
from collections import defaultdict

# --- Caches para normalización ---
@functools.lru_cache(maxsize=2048)
def _cached_normalizar_categoria(valor: str, tipo: str, threshold: float = 0.8) -> Tuple[str, bool]:
    """Versión cacheada de normalizar_categoria."""
    return normalizar_categoria(valor, tipo, debug=False, threshold=threshold)

@functools.lru_cache(maxsize=2048)
def _cached_normalizar_valor_con_catalogo(valor: str, tabla: str, threshold: float = 0.5) -> str:
    """Versión cacheada de normalizar_valor_con_catalogo."""
    return normalizar_valor_con_catalogo(valor, tabla, threshold=threshold)


@functools.lru_cache(maxsize=1024)
def predict_categories(texto: str) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    """
    Predice proceso, eje y tema usando ML o fallback por palabras clave.
    Con caché para strings de longitud razonable.
    """
    if not texto:
        return None, None, None

    # Si los modelos ML están cargados, usarlos
    if _models_loaded:
        texto_limpio = clean_text(texto)
        if texto_limpio:
            prob_proc = _model_proceso.predict_proba([texto_limpio])[0]
            max_prob_proc = max(prob_proc)
            proc_pred = _model_proceso.classes_[prob_proc.argmax()] if max_prob_proc >= UMBRAL_CONFIANZA else None

            prob_eje = _model_eje.predict_proba([texto_limpio])[0]
            max_prob_eje = max(prob_eje)
            eje_pred = _model_eje.classes_[prob_eje.argmax()] if max_prob_eje >= UMBRAL_CONFIANZA else None

            prob_tema = _model_tema.predict_proba([texto_limpio])[0]
            max_prob_tema = max(prob_tema)
            tema_pred = _model_tema.classes_[prob_tema.argmax()] if max_prob_tema >= UMBRAL_CONFIANZA else None

            # Verificar consistencia eje-tema
            if eje_pred and tema_pred:
                eje_num_match = re.search(r'Eje\s*(\d+)', eje_pred)
                tema_num_match = re.search(r'^(\d+)\.', tema_pred)
                if eje_num_match and tema_num_match:
                    if eje_num_match.group(1) != tema_num_match.group(1):
                        tema_pred = None

            if proc_pred and eje_pred and tema_pred:
                return proc_pred, eje_pred, tema_pred

    # Fallback por palabras clave
    return predecir_por_palabras_clave(texto)

def _process_disaggregated_data(df, catalog_map, id_col, agg_cols, existing_hashes, nombre_base,
                                global_metadata, log_messages=None, debug=False):
    if log_messages is None:
        log_messages = []
    if global_metadata is None:
        global_metadata = {}
    messages = []
    all_rows = []

    # --- Asegurar nombre_base no vacío ---
    if not nombre_base or nombre_base.strip() == '':
        nombre_base = f"Agregación de {id_col}" if id_col else "Variable agregada"
        if debug:
            messages.append(f"⚠️ nombre_base vacío, se usará '{nombre_base}'")

    # --- Predecir categorías una sola vez para el nombre_base ---
    try:
        proc, eje, tema = predict_categories(nombre_base)
    except Exception as e:
        messages.append(f"Error al predecir categorías para nombre_base '{nombre_base}': {e}")
        proc, eje, tema = None, None, None

    # --- Asegurar valores por defecto SIEMPRE ---
    if not proc:
        proc = list(KEYWORDS['proceso'].keys())[0]
    if not eje:
        eje = list(KEYWORDS['eje'].keys())[0]
    if not tema:
        tema = list(KEYWORDS['tema'].keys())[0]

    # --- Asegurar consistencia tema-eje ---
    eje_num = re.search(r'Eje\s*(\d+)', eje)
    tema_num = re.search(r'^(\d+)\.', tema)
    if eje_num and tema_num and eje_num.group(1) != tema_num.group(1):
        posibles_temas = [t for t in KEYWORDS['tema'].keys() if t.startswith(eje_num.group(1) + '.')]
        if posibles_temas:
            tema = posibles_temas[0]
            if debug:
                messages.append(f"🔄 Tema ajustado a '{tema}' para coincidir con eje '{eje}'")

    # --- Base común para todas las filas (padre e hijos) ---
    base_fila = {
        'proceso': global_metadata.get('proceso') or proc,
        'eje': global_metadata.get('eje') or eje,
        'tema': global_metadata.get('tema') or tema,
        'estado': global_metadata.get('estado') or 'No aplica',
        'institucion': global_metadata.get('institucion') or '',
        'cobertura': global_metadata.get('cobertura') or '',
        'periodicidad': global_metadata.get('periodicidad') or '',
        'liga_web': global_metadata.get('liga_web') or '',
        'fuente': global_metadata.get('fuente') or '',
        'año': global_metadata.get('año') or '',
        'parent_id': None,
    }

    # Asegurar que año sea string
    if base_fila['año']:
        try:
            base_fila['año'] = str(int(float(base_fila['año'])))
        except:
            pass

    # Para cada columna de agregación
    for agg_col in agg_cols:
        try:
            # Convertir a numérico (coerce errores)
            numeric_series = pd.to_numeric(df[agg_col], errors='coerce')
            total_sum = numeric_series.sum() if not numeric_series.isna().all() else 0

            # --- PADRE ---
            padre_data = base_fila.copy()
            padre_data['nombre'] = f"{nombre_base} - Total"
            padre_data['valor'] = total_sum
            padre_hash = _get_non_id_hash(padre_data)
            padre_existente = existing_hashes.get(padre_hash)

            # --- HIJOS (agrupados por id_col) ---
            grouped = df.groupby(id_col)[agg_col].sum()  # Serie: índice = categoría, valor = suma
            hijos_status = []

            for key, val in grouped.items():
                key_str = str(key).strip()
                # Obtener descripción del catálogo
                desc = catalog_map.get(key_str)
                if desc is None:
                    # Búsqueda por coincidencia exacta sin espacios
                    for k, v in catalog_map.items():
                        if k.strip() == key_str:
                            desc = v
                            break
                if desc is None:
                    desc = f"Categoría {key_str}" if key_str.isdigit() else key_str

                hijo_data = base_fila.copy()
                nombre_hijo = f"{nombre_base} - {desc}".strip()
                if not nombre_hijo or nombre_hijo == '-':
                    nombre_hijo = f"{nombre_base} (sin categoría)"
                hijo_data['nombre'] = nombre_hijo
                hijo_data['valor'] = val
                hijo_data['parent_id'] = padre_existente.get('id') if padre_existente else None
                if 'id' in hijo_data:
                    del hijo_data['id']

                hijo_hash = _get_non_id_hash(hijo_data)
                if hijo_hash in existing_hashes:
                    matching = existing_hashes[hijo_hash]
                    row_status = {
                        'original_csv_index': -1,
                        'data': hijo_data,
                        '_is_duplicate': True,
                        '_supabase_matching_id': matching.get('id'),
                        '_hash': hijo_hash,
                        '_is_child': True,
                        '_parent_hash': padre_hash,
                        '_parent_id': matching.get('parent_id') if matching.get('parent_id') is not None else None
                    }
                else:
                    row_status = {
                        'original_csv_index': -1,
                        'data': hijo_data,
                        '_is_duplicate': False,
                        '_supabase_matching_id': None,
                        '_hash': hijo_hash,
                        '_is_child': True,
                        '_parent_hash': padre_hash,
                        '_parent_id': None
                    }
                hijos_status.append(row_status)

            # Agregar padre (si no existe) y luego los hijos
            if padre_existente is None:
                padre_row_status = {
                    'original_csv_index': -1,
                    'data': padre_data,
                    '_is_duplicate': False,
                    '_supabase_matching_id': None,
                    '_hash': padre_hash,
                    '_is_parent': True,
                    '_children_hashes': [h['_hash'] for h in hijos_status]
                }
                all_rows.append(padre_row_status)
                all_rows.extend(hijos_status)
            else:
                # Padre existente: asignar su id a los hijos
                for h in hijos_status:
                    h['_parent_id'] = padre_existente['id']
                all_rows.extend(hijos_status)

        except Exception as e:
            messages.append(f"Error procesando columna de agregación '{agg_col}': {e}")

    if debug:
        messages.append(f"🔹 Desagregación: generados {len(all_rows)} total (padres+hijos) para columnas {agg_cols}")
        if all_rows:
            for r in all_rows:
                if r.get('_is_child'):
                    messages.append(f"   Ejemplo hijo: {r['data']}")
                    break

    return all_rows, messages

def _predict_categories_vectorized(df: pd.DataFrame, col_nombre: str = 'nombre') -> pd.DataFrame:
    """
    Predice proceso, eje y tema para todas las filas de una vez,
    usando un caché por nombre único.
    """
    # Obtener nombres únicos no vacíos
    nombres_unicos = df[col_nombre].dropna().unique()
    nombres_unicos = [n for n in nombres_unicos if str(n).strip()]

    # Crear un mapa nombre -> (proceso, eje, tema)
    pred_cache = {}
    for nombre in nombres_unicos:
        proc, eje, tema = predict_categories(str(nombre))
        pred_cache[nombre] = (proc, eje, tema)

    def asignar_categoria(row, idx_col):
        nombre = row[col_nombre]
        if pd.isna(nombre) or str(nombre).strip() == '':
            return None
        return pred_cache.get(nombre, (None, None, None))[idx_col]

    # Asignar cada columna solo si está vacía
    for col, idx in [('proceso', 0), ('eje', 1), ('tema', 2)]:
        if col in df.columns:
            mask = df[col].isna() | (df[col] == '')
            df.loc[mask, col] = df.loc[mask].apply(lambda row: asignar_categoria(row, idx), axis=1)

    # Rellenar con valores por defecto si quedaron vacíos
    for col in ['proceso', 'eje', 'tema']:
        if col in df.columns:
            default = list(KEYWORDS[col].keys())[0]
            df[col] = df[col].fillna(default)

    return df


def _normalize_catalog_values_vectorized(df: pd.DataFrame, catalog_columns: List[str]) -> pd.DataFrame:
    """
    Normaliza los valores de catálogo de todo el DataFrame usando cache y operaciones vectorizadas.
    """
    for col in catalog_columns:
        if col not in df.columns:
            continue
        # Obtener valores únicos no nulos
        unique_vals = df[col].dropna().unique()
        unique_vals = [v for v in unique_vals if str(v).strip()]
        if not unique_vals:
            continue

        # Crear mapa valor -> normalizado
        norm_map = {}
        for val in unique_vals:
            if col in ['institucion', 'cobertura', 'periodicidad']:  # tipo_de_dato eliminado
                norm_val = _cached_normalizar_valor_con_catalogo(str(val), col, 0.8)
            else:  # proceso, eje, tema
                norm_val, _ = _cached_normalizar_categoria(str(val), col, 0.8)
            norm_map[val] = norm_val

        # Aplicar el mapa
        df[col] = df[col].map(norm_map).fillna(df[col])

    return df


def _compute_hashes_vectorized(df: pd.DataFrame) -> pd.Series:
    """
    Calcula el hash (_get_non_id_hash) para todas las filas de forma vectorizada.
    """
    cols = [c for c in EXPECTED_COLUMNS if c != 'id']
    for c in cols:
        if c not in df.columns:
            df[c] = ''

    def row_hash(row):
        vals = []
        for key in cols:
            val = row.get(key, '')
            if pd.isna(val):
                val = ''
            if key in ['año']:  # version eliminado
                val = normalizar_numero_texto(val, key)
            elif key == 'valor':
                temp = normalizar_numero_texto(val, key)
                if temp.upper() in VALORES_ESPECIALES:
                    val = temp
                else:
                    try:
                        num = float(temp)
                        val = f"{num:.1f}"
                    except:
                        val = ''
            val = safe_clean(val)
            vals.append(val)
        return tuple(vals)

    return df.apply(row_hash, axis=1)


# --- El resto de funciones auxiliares (normalizar columns, mapear, etc.) ---

def _normalize_columns(df: pd.DataFrame, log_messages: List[str]) -> pd.DataFrame:
    """Renombra y normaliza columnas del CSV."""
    year_aliases = ['año', 'ano', 'anio', 'Año', 'AÑO', 'anho']
    for col in df.columns:
        col_clean = remover_acentos(str(col).lower().strip().replace(' ', '_'))
        if col_clean in year_aliases or col_clean == 'año':
            if col != 'año':
                df.rename(columns={col: 'año'}, inplace=True)
                log_messages.append(f"Columna '{col}' renombrada a 'año' por alias.")
            break

    estado_aliases = ['entidad', 'estado', 'entidad_federativa', 'estado_federativo']
    found_estado = False
    for col in df.columns:
        col_clean = remover_acentos(str(col).lower().strip().replace(' ', '_'))
        if col_clean in estado_aliases:
            if col != 'estado':
                df.rename(columns={col: 'estado'}, inplace=True)
                log_messages.append(f"Columna '{col}' renombrada a 'estado' por alias.")
            found_estado = True
            break
    if not found_estado:
        df['estado'] = pd.NA
        log_messages.append("No se encontró columna de estado, se crea vacía.")
    return df


def _map_columns_to_expected(df: pd.DataFrame, log_messages: List[str]) -> pd.DataFrame:
    """Mapea columnas del CSV a EXPECTED_COLUMNS, conservando columnas adicionales necesarias."""
    all_known_normalized_keys = set(EXPECTED_COLUMNS)
    for display_name in COLUMN_DISPLAY_NAMES:
        all_known_normalized_keys.add(_normalize_col_for_matching(display_name))

    column_rename_map = {}
    processed_target_cols = set()
    for original_col in df.columns:
        normalized_original_col = _normalize_col_for_matching(original_col)
        target_col_name = None
        if normalized_original_col in EXPECTED_COLUMNS:
            target_col_name = normalized_original_col
            if original_col != target_col_name:
                log_messages.append(
                    f"Columna renombrada: '{original_col}' a '{target_col_name}' (normalización)."
                )
        else:
            fuzzy_match = _find_best_column_match(
                normalized_original_col, list(all_known_normalized_keys)
            )
            if fuzzy_match and fuzzy_match in EXPECTED_COLUMNS:
                target_col_name = fuzzy_match
                log_messages.append(
                    f"Columna renombrada: '{original_col}' a '{target_col_name}' (sugerencia)."
                )
        if target_col_name:
            if target_col_name not in processed_target_cols:
                column_rename_map[original_col] = target_col_name
                processed_target_cols.add(target_col_name)
            else:
                log_messages.append(
                    f"Columna '{original_col}' ignorada (ya mapeada a '{target_col_name}')."
                )
        else:
            column_rename_map[original_col] = original_col
    df.rename(columns=column_rename_map, inplace=True)

    # ⚠️ CAMBIO: No eliminar columnas que empiecen con '_' ni 'parent_id'
    extra_cols = [
        col for col in df.columns
        if col not in EXPECTED_COLUMNS
        and col != 'id'
        and not col.startswith('_')
        and col != 'parent_id'
    ]
    if extra_cols:
        df.drop(columns=extra_cols, inplace=True)
        log_messages.append(f"Columnas ignoradas: {', '.join(extra_cols)}")

    # Asegurar que las columnas esperadas existan
    missing_cols = [col for col in EXPECTED_COLUMNS if col not in df.columns]
    for col in missing_cols:
        df[col] = pd.NA
        log_messages.append(f"Columna añadida (faltante): '{col}'")

    # Devolver todas las columnas (incluyendo parent_id y las que empiezan con '_')
    return df


def _build_existing_hashes_for_csv(
    existing_supabase_data_full: List[Dict],
    log_messages: List[str]
) -> Dict[Tuple, Dict]:
    """Construye índice de hashes usando vectorización, incluyendo parent_id."""
    if not existing_supabase_data_full:
        return {}

    try:
        df = pd.DataFrame(existing_supabase_data_full)
        # Normalizar catálogos y categorías
        for col in ['institucion', 'cobertura', 'periodicidad']:
            if col in df.columns:
                df[col] = df[col].apply(lambda v: normalizar_valor_con_catalogo(v, col, 0.5) if v else v)
        for col in ['proceso', 'eje', 'tema']:
            if col in df.columns:
                df[col] = df[col].apply(lambda v: normalizar_categoria(v, col, False, 0.8)[0] if v else v)
        # Asegurar parent_id
        if 'parent_id' not in df.columns:
            df['parent_id'] = None
        # Calcular hashes vectorizados (incluye parent_id)
        hash_series = _compute_row_hash_vectorized(df)
        existing_hashes = {}
        for idx, row in df.iterrows():
            h = hash_series[idx]
            if h not in existing_hashes:
                existing_hashes[h] = row.to_dict()
        return existing_hashes
    except Exception as e:
        log_messages.append(f"Error al normalizar datos existentes: {e}")
        return {}

def _process_csv_data(
    df_input: pd.DataFrame,
    existing_supabase_data_full: List[Dict],
    log_messages: Optional[List[str]] = None,
    debug: bool = False
) -> Tuple[List[Dict], List[str]]:
    if log_messages is None:
        log_messages = []
    messages = []
    df = df_input.copy()

    # 1. Normalización de columnas y mapeo
    df = _normalize_columns(df, messages)
    df = _map_columns_to_expected(df, messages)  # esta función ya conserva parent_id

    # 2. Normalizar año
    if 'año' in df.columns:
        df['año'] = df['año'].apply(lambda x: safe_normalize(x, 'año'))
        df['año'] = df['año'].apply(lambda x: str(int(float(x))) if pd.notna(x) and str(x).strip() else '')

    # 3. Predicción de categorías (vectorizada)
    df = _predict_categories_vectorized(df)

    # 4. Normalización de catálogos
    catalog_cols = ['institucion', 'cobertura', 'periodicidad', 'proceso', 'eje', 'tema']
    df = _normalize_catalog_values_vectorized(df, catalog_cols)

    # 5. Construir hashes de datos existentes (incluye parent_id)
    existing_hashes = _build_existing_hashes_for_csv(existing_supabase_data_full, messages)

    # 6. Calcular hashes del CSV de forma vectorizada (incluye parent_id)
    df['_hash'] = _compute_row_hash_vectorized(df)

    # 7. Determinar duplicados y asignar IDs
    existing_ids_from_db = _get_all_existing_ids()
    temp_ids = set(existing_ids_from_db)

    df['_is_duplicate'] = df['_hash'].isin(existing_hashes.keys())
    df['_supabase_matching_id'] = df['_hash'].apply(lambda h: existing_hashes.get(h, {}).get('id') if h in existing_hashes else None)

    def generate_id(row):
        if not row['_is_duplicate']:
            return generar_siguiente_id(temp_ids)
        return row.get('id', None)

    df['id'] = df.apply(generate_id, axis=1)

    # 8. Convertir a lista de diccionarios con metadatos
    records = df.to_dict('records')
    all_rows = []
    for idx, row_dict in enumerate(records):
        data_dict = {k: v for k, v in row_dict.items() if k not in ['_hash', '_is_duplicate', '_supabase_matching_id']}
        row_status = {
            'original_csv_index': idx,
            'data': data_dict,
            '_is_duplicate': row_dict.get('_is_duplicate', False),
            '_supabase_matching_id': row_dict.get('_supabase_matching_id'),
            '_hash': row_dict.get('_hash')
        }
        # Conservar flags de jerarquía si existen (vienen de process_data_folders_with_return)
        if '_is_parent' in row_dict:
            row_status['_is_parent'] = row_dict['_is_parent']
        if '_is_child' in row_dict:
            row_status['_is_child'] = row_dict['_is_child']
        if '_parent_hash' in row_dict:
            row_status['_parent_hash'] = row_dict['_parent_hash']
        if '_parent_id' in row_dict:
            row_status['_parent_id'] = row_dict['_parent_id']
        all_rows.append(row_status)

    return all_rows, messages

## Módulo 6: Procesamiento de carpetas – parte 1 (auxiliares)

In [6]:
# ============================================================
# MÓDULO 6: PROCESAMIENTO DE CARPETAS – AUXILIARES
# ============================================================
def normalize_string(s):
    if not isinstance(s, str):
        return ""
    s = remover_acentos(s)
    s = s.lower()
    s = re.sub(r'[^a-z0-9\s]', '', s)
    s = re.sub(r'\s+', '_', s).strip('_')
    return s

def parse_global_metadatos(metadatos_text, log_messages=None):
    """
    Extrae metadatos de un archivo de texto (formato similar a DCAT).
    """
    if log_messages is None:
        log_messages = []
    global_meta = {
        'proceso': '', 'eje': '', 'tema': '', 'nombre': '',
        'institucion': '', 'cobertura': '', 'periodicidad': '', 'liga_web': '',
        'fuente': '', 'año': ''
    }

    title_match = re.search(r'Title:\s*(.+)', metadatos_text, re.IGNORECASE)
    if title_match:
        global_meta['fuente'] = title_match.group(1).strip()
        year_match = re.search(r'\b(19|20)\d{2}\b', global_meta['fuente'])
        if year_match:
            global_meta['año'] = year_match.group(0)

    if not global_meta['año']:
        year_match = re.search(r'\b(19|20)\d{2}\b', metadatos_text)
        if year_match:
            global_meta['año'] = year_match.group(0)
            log_messages.append(f"🔍 Año extraído de metadatos: {global_meta['año']}")

    if not global_meta['año']:
        temporal_match = re.search(r'Temporal:\s*(\d{4})-\d{2}-\d{2}', metadatos_text, re.IGNORECASE)
        if temporal_match:
            global_meta['año'] = temporal_match.group(1)
            log_messages.append(f"🔍 Año extraído de Temporal: {global_meta['año']}")

    publisher_match = re.search(r'Publisher:\s*(.+)', metadatos_text, re.IGNORECASE)
    if publisher_match:
        institucion = publisher_match.group(1).strip()
        institucion = re.sub(r'[,.]$', '', institucion).strip()
        if institucion:
            global_meta['institucion'] = institucion
            log_messages.append(f"🏛️ Institución extraída de Publisher: {institucion}")
    else:
        if title_match and 'Censo Nacional de Gobierno Federal' in title_match.group(1):
            global_meta['institucion'] = 'Instituto Nacional de Estadística y Geografía'
            log_messages.append("🏛️ Institución asignada: INEGI (por CNGF en Title)")
        else:
            description_match = re.search(r'Description:\s*(.+)', metadatos_text, re.IGNORECASE)
            if description_match:
                institucion_raw = description_match.group(1).strip()
                institucion_raw = re.sub(r'\b(19|20)\d{2}\b', '', institucion_raw).strip()
                if institucion_raw:
                    global_meta['institucion'] = institucion_raw
                    log_messages.append(f"🏛️ Institución extraída de Description: {institucion_raw}")

    cobertura_found = False
    if title_match:
        title_value = title_match.group(1).strip().lower()
        if any(term in title_value for term in {'federal','federales'}):
            global_meta['cobertura'] = 'Federal'
            cobertura_found = True
        elif any(term in title_value for term in {'estatal','estatales'}):
            global_meta['cobertura'] = 'Estatal'
            cobertura_found = True
        elif any(term in title_value for term in {'municipal','municipales'}):
            global_meta['cobertura'] = 'Municipal'
            cobertura_found = True

    if not cobertura_found:
        spatial_match = re.search(r'Spatial:\s*(.+)', metadatos_text, re.IGNORECASE)
        if spatial_match:
            spatial_value = spatial_match.group(1).strip()
            if 'Estados Unidos Mexicanos' in spatial_value:
                global_meta['cobertura'] = 'Federal'
            else:
                global_meta['cobertura'] = spatial_value

    periodicidad_match = re.search(r'AccrualPeriodicity:\s*(.+)', metadatos_text, re.IGNORECASE)
    if periodicidad_match:
        global_meta['periodicidad'] = periodicidad_match.group(1).strip()

    distribution_match = re.search(r'Distribution:\s*(.+)', metadatos_text, re.IGNORECASE)
    if distribution_match:
        global_meta['liga_web'] = distribution_match.group(1).strip()

    if not global_meta['fuente'] and title_match:
        global_meta['fuente'] = title_match.group(1).strip()

    return global_meta

def _process_section_with_pandas(
    data_lines: List[str],
    col_idx: int,
    desc_idx: int,
    section_description: str,
    target_list: List[Dict],
    log_messages: List[str]
) -> None:
    """
    Procesa un bloque de líneas de una sección del formato antiguo usando Pandas.
    Es mucho más rápido que csv.reader para bloques grandes.
    """
    if not data_lines:
        return
    try:
        block = '\n'.join(data_lines)
        df_block = pd.read_csv(
            io.StringIO(block),
            header=None,
            dtype=str,
            low_memory=False,
            encoding='utf-8-sig'
        )
        max_cols = max(col_idx, desc_idx, 7) + 1
        while df_block.shape[1] < max_cols:
            df_block[f'col_{df_block.shape[1]}'] = pd.NA

        row_count = 0
        for idx, row in df_block.iterrows():
            col_name = str(row[col_idx]).strip() if col_idx < len(row) and pd.notna(row[col_idx]) else ''
            if not col_name or col_name.isdigit() or 'NSS' in col_name or 'ND' in col_name:
                continue

            col_desc = str(row[desc_idx]).strip() if desc_idx < len(row) and pd.notna(row[desc_idx]) else ''
            if not col_desc and section_description:
                col_desc = section_description

            unnamed_col_8 = str(row[7]).strip() if len(row) > 7 and pd.notna(row[7]) else ''

            row_dict = {
                'Nombre de la columna': col_name,
                'Descripcion': col_desc,
                '_seccion_desc': section_description,
                'UNNAMED_COL_8': unnamed_col_8
            }
            target_list.append(row_dict)
            row_count += 1

        if row_count == 0:
            log_messages.append(f"   ⚠️ No se encontraron filas de datos válidas en esta sección.")
    except Exception as e:
        log_messages.append(f"   ❌ Error al procesar datos con Pandas: {e}")

def _process_section_data(data_lines, col_idx, desc_idx, section_description, target_list, log_messages):
    """
    Procesa las líneas de datos de una sección del formato antiguo.
    """
    if not data_lines:
        return
    try:
        reader = csv.reader(data_lines, delimiter=',', quotechar='"')
        row_count = 0
        for row in reader:
            if not row or len(row) <= max(col_idx, desc_idx):
                continue
            col_name = row[col_idx].strip() if col_idx < len(row) else ''
            if not col_name or col_name.isdigit() or 'NSS' in col_name or 'ND' in col_name:
                continue
            col_desc = row[desc_idx].strip() if desc_idx < len(row) else ''
            if not col_desc and section_description:
                col_desc = section_description
            unnamed_col_8 = row[7] if len(row) > 7 else ''
            row_dict = {
                'Nombre de la columna': col_name,
                'Descripcion': col_desc,
                '_seccion_desc': section_description,
                'UNNAMED_COL_8': unnamed_col_8
            }
            target_list.append(row_dict)
            row_count += 1
        if row_count == 0:
            log_messages.append(f"   ⚠️ No se encontraron filas de datos válidas en esta sección.")
    except Exception as e:
        log_messages.append(f"   ❌ Error al procesar datos: {e}")

def parse_dictionary_csv(file_path: str, format_type: str = 'old', log_messages: Optional[List[str]] = None):
    """
    Parsea un archivo CSV de diccionario en formato antiguo (múltiples secciones)
    o nuevo (una tabla con columnas 'COLUMNA' y 'DESCRIPCION').

    Args:
        file_path: Ruta al archivo CSV.
        format_type: 'old' o 'new'.
        log_messages: Lista para acumular mensajes de depuración.

    Returns:
        - Para formato 'new': (lista_de_definiciones, None)
          donde cada definición es un dict con 'Nombre de la columna' y 'Descripcion'.
        - Para formato 'old': (diccionario_secciones, None)
          donde cada clave es el nombre del archivo .dbf y el valor es una lista de definiciones.
    """
    if log_messages is None:
        log_messages = []

    if format_type not in ['old', 'new']:
        raise ValueError(f"Tipo de formato desconocido: {format_type}")

    # ---- Lectura del archivo ----
    lines = None
    try:
        with open(file_path, 'r', encoding='utf-8-sig') as f:
            lines = f.readlines()
        log_messages.append(f"📄 Archivo leído con UTF-8: {file_path} ({len(lines)} líneas)")
    except UnicodeDecodeError:
        try:
            with open(file_path, 'r', encoding='latin-1') as f:
                lines = f.readlines()
            log_messages.append(f"📄 Archivo leído con Latin-1: {file_path} ({len(lines)} líneas)")
        except Exception as e:
            log_messages.append(f"❌ Error al leer archivo con Latin-1: {e}")
            return [] if format_type == 'new' else {}, None
    except Exception as e:
        log_messages.append(f"❌ Error al leer archivo '{file_path}': {e}")
        return [] if format_type == 'new' else {}, None

    if not lines:
        log_messages.append("⚠️ El archivo está vacío.")
        return [] if format_type == 'new' else {}, None

    # ============================================================
    #  FORMATO NUEVO (una sola tabla con encabezado)
    # ============================================================
    if format_type == 'new':
        log_messages.append("🔍 Procesando formato nuevo (encabezado único).")
        try:
            reader = csv.reader(lines, delimiter=',', quotechar='"')
            header = next(reader, None)
            if not header:
                log_messages.append("⚠️ No se encontró encabezado en formato nuevo.")
                return [], None

            col_idx = None
            desc_idx = None
            for i, h in enumerate(header):
                h_norm = remover_acentos(h.strip().lower()).replace(' ', '_')
                if h_norm in ['columna', 'nombre_de_la_columna', 'campo', 'variable']:
                    col_idx = i
                elif h_norm in ['descripcion', 'descripción', 'desc', 'definicion', 'definición']:
                    desc_idx = i

            if col_idx is None or desc_idx is None:
                log_messages.append(f"⚠️ No se encontraron columnas 'Columna' y 'Descripción' en el encabezado: {header}")
                return [], None

            definitions = []
            row_count = 0
            for row in reader:
                if len(row) > max(col_idx, desc_idx):
                    col_name = row[col_idx].strip()
                    desc = row[desc_idx].strip()
                    if col_name:
                        definitions.append({
                            'Nombre de la columna': col_name,
                            'Descripcion': desc
                        })
                        row_count += 1
            log_messages.append(f"✅ Formato nuevo: {len(definitions)} definiciones encontradas (filas leídas: {row_count}).")
            return definitions, None

        except Exception as e:
            log_messages.append(f"❌ Error parsing formato nuevo: {e}")
            return [], None

    # ============================================================
    #  FORMATO ANTIGUO (múltiples secciones) – OPTIMIZADO CON PANDAS
    # ============================================================
    else:
        log_messages.append("🔍 Procesando formato antiguo (múltiples secciones) con Pandas.")
        all_data_file_column_definitions = {}
        current_section = None
        current_description = ""
        data_lines = []
        header_found = False
        col_idx = 1
        desc_idx = 2

        i = 0
        while i < len(lines):
            line = lines[i].strip()
            clean_line = line.strip('"')

            # Detectar nueva sección
            if 'Archivo:' in clean_line and '.dbf' in clean_line:
                # Guardar sección anterior si existe
                if current_section is not None and data_lines:
                    all_data_file_column_definitions[current_section] = []
                    _process_section_with_pandas(
                        data_lines, col_idx, desc_idx, current_description,
                        all_data_file_column_definitions[current_section],
                        log_messages
                    )
                    data_lines = []
                    log_messages.append(f"✅ Sección '{current_section}' procesada (Pandas).")

                match = re.search(r'Archivo:\s*([^\s.]+)\.dbf', clean_line, re.IGNORECASE)
                if match:
                    current_section = match.group(1).lower()
                    log_messages.append(f"🔎 Sección detectada: '{current_section}'")
                    desc_match = re.search(r'\((.*?)\)', clean_line)
                    if desc_match:
                        current_description = desc_match.group(1).strip()
                    else:
                        after_dbf = re.sub(r'.*\.dbf', '', clean_line).strip()
                        current_description = after_dbf.strip(' ,') if after_dbf else ""
                    log_messages.append(f"   Descripción: '{current_description[:60]}{'...' if len(current_description)>60 else ''}'")
                    header_found = False
                    data_lines = []
                    i += 1
                    continue

            # Si estamos dentro de una sección
            if current_section is not None:
                # Detectar encabezado para obtener índices
                if not header_found and ('Núm.de campo' in line or 'Nombre de la columna' in line):
                    header_found = True
                    try:
                        header_parts = next(csv.reader([line], delimiter=',', quotechar='"'))
                        for idx, h in enumerate(header_parts):
                            h_norm = remover_acentos(h.strip().lower()).replace(' ', '_')
                            if h_norm in ['nombre_de_la_columna', 'nombre de la columna', 'columna']:
                                col_idx = idx
                            elif h_norm in ['descripcion', 'descripción', 'desc']:
                                desc_idx = idx
                        log_messages.append(f"   Encabezado: columna={col_idx}, descripción={desc_idx}")
                    except:
                        log_messages.append(f"   ⚠️ Error parseando encabezado, usando índices por defecto (col=1, desc=2)")
                    i += 1
                    continue

                # Acumular líneas de datos (excluyendo cabeceras de censo, etc.)
                if line and not line.startswith('"Censo') and not line.startswith('Censo') and line != ',' * 10:
                    if line.replace(',', '').strip():
                        data_lines.append(line)
                i += 1
            else:
                i += 1

        # Guardar la última sección
        if current_section is not None and data_lines:
            all_data_file_column_definitions[current_section] = []
            _process_section_with_pandas(
                data_lines, col_idx, desc_idx, current_description,
                all_data_file_column_definitions[current_section],
                log_messages
            )
            log_messages.append(f"✅ Sección '{current_section}' procesada (Pandas).")

        # Limpiar secciones vacías
        to_remove = [k for k, v in all_data_file_column_definitions.items() if not v]
        for k in to_remove:
            del all_data_file_column_definitions[k]

        log_messages.append(f"✅ Total de secciones procesadas: {len(all_data_file_column_definitions)}")
        for key, defs in all_data_file_column_definitions.items():
            log_messages.append(f"   - {key}: {len(defs)} definiciones")

        if not all_data_file_column_definitions:
            log_messages.append("⚠️ No se encontraron definiciones. Verifica que el archivo tenga secciones con 'Archivo: nombre.dbf'.")

        return all_data_file_column_definitions, None

def infer_consecutive_descriptions(column_defs):
    processed_column_defs = []
    grouped_defs = {}

    for col_def in column_defs:
        col_name = col_def.get('Nombre de la columna', '')
        match = re.match(r'([a-zA-Z_]+?)(\d+)$', col_name)
        if match:
            base_name = match.group(1)
            suffix = int(match.group(2))
            if base_name not in grouped_defs:
                grouped_defs[base_name] = []
            grouped_defs[base_name].append((suffix, col_def))
        else:
            processed_column_defs.append(col_def)

    for base_name, definitions in grouped_defs.items():
        definitions.sort(key=lambda x: x[0])
        common_base_description_segment = ""

        for suffix, col_def_original in definitions:
            col_def = col_def_original.copy()
            current_raw_description = col_def.get('Descripcion', '').strip()

            differentiator_value = ""
            if 'Opciones' in col_def and col_def['Opciones'].strip():
                differentiator_value = col_def['Opciones'].strip()
            elif 'Categoría' in col_def and col_def['Categoría'].strip():
                differentiator_value = col_def['Categoría'].strip()
            elif 'UNNAMED_COL_8' in col_def and col_def['UNNAMED_COL_8'].strip():
                differentiator_value = col_def['UNNAMED_COL_8'].strip()

            if current_raw_description:
                cleaned_desc_for_current = current_raw_description.rstrip('.,;')
                if differentiator_value:
                    final_description_for_current = f"{cleaned_desc_for_current} {differentiator_value}"
                else:
                    final_description_for_current = cleaned_desc_for_current
                col_def['Descripcion'] = final_description_for_current

                if not common_base_description_segment:
                    common_base_description_segment = cleaned_desc_for_current
            else:
                if common_base_description_segment:
                    if differentiator_value:
                        final_description_for_current = f"{common_base_description_segment} {differentiator_value}"
                    else:
                        final_description_for_current = common_base_description_segment
                    col_def['Descripcion'] = final_description_for_current
                else:
                    col_def['Descripcion'] = ""

            processed_column_defs.append(col_def)

    return processed_column_defs

def _get_output_nombre_value(col_nombre, col_descripcion, col_def, selected_data_file_name, new_format, indice_descriptions):
    final_nombre_value = ""

    def limpiar_descripcion(desc):
        if not desc:
            return desc
        patrones = [
            r'^Contiene variables que caracterizan a la Administración Pública Federal de acuerdo con\s*',
            r'^Contiene variables que caracterizan a la Administración Pública Federal de acuerdo a\s*',
            r'^Contiene variables que caracterizan a la Administración Pública Federal según\s*',
            r'^Contiene variables que caracterizan a la Administración Pública Federal\s*',
            r'^Contiene variables que caracterizan\s*',
            r'^Contiene variables que\s*',
            r'^Variables que caracterizan\s*',
            r'^Descripción:\s*',
            r'^Definición:\s*',
        ]
        desc_limpia = desc
        for patron in patrones:
            desc_limpia = re.sub(patron, '', desc_limpia, flags=re.IGNORECASE)
        desc_limpia = desc_limpia.strip()
        if not desc_limpia:
            return desc
        desc_limpia = re.sub(r'^de acuerdo con\s*', '', desc_limpia, flags=re.IGNORECASE)
        desc_limpia = re.sub(r'^de acuerdo a\s*', '', desc_limpia, flags=re.IGNORECASE)
        desc_limpia = re.sub(r'^según\s*', '', desc_limpia, flags=re.IGNORECASE)
        if desc_limpia:
            desc_limpia = desc_limpia[0].upper() + desc_limpia[1:]
        return desc_limpia.strip()

    if not new_format:
        desc_limpia = limpiar_descripcion(col_descripcion)
        if desc_limpia:
            final_nombre_value = desc_limpia
        else:
            seccion_desc = col_def.get('_seccion_desc', '').strip()
            if seccion_desc:
                seccion_limpia = limpiar_descripcion(seccion_desc)
                final_nombre_value = seccion_limpia if seccion_limpia else seccion_desc
            else:
                final_nombre_value = col_nombre if col_nombre else ""

        unnamed_col_8 = col_def.get('UNNAMED_COL_8', '').strip()
        if unnamed_col_8 and final_nombre_value and not col_descripcion:
            final_nombre_value = f"{final_nombre_value} {unnamed_col_8}"

        if not final_nombre_value:
            final_nombre_value = col_nombre if col_nombre else ""

        if not col_descripcion and not col_def.get('_seccion_desc'):
            final_nombre_value = f"{final_nombre_value} (origen: {selected_data_file_name})"

    else:
        temp_name_without_ext = selected_data_file_name.split('.')[0].lower()
        base_file_name_for_lookup = temp_name_without_ext.split('_')[0] if '_' in temp_name_without_ext else temp_name_without_ext
        indice_content = indice_descriptions.get(base_file_name_for_lookup, '').strip()
        if indice_content:
            if col_descripcion:
                desc_limpia = limpiar_descripcion(col_descripcion)
                final_nombre_value = f"{indice_content}: {desc_limpia if desc_limpia else col_descripcion}"
            else:
                final_nombre_value = indice_content
        else:
            desc_limpia = limpiar_descripcion(col_descripcion)
            final_nombre_value = desc_limpia if desc_limpia else col_descripcion

        if not final_nombre_value:
            final_nombre_value = col_nombre if col_nombre else ""

    if isinstance(final_nombre_value, str):
        final_nombre_value = re.sub(r',\s*durante el a\u00f1o\s*\d{4}\s*\.?\s*', '', final_nombre_value, flags=re.IGNORECASE)
        final_nombre_value = final_nombre_value.strip().strip(',').strip('.')

    if not final_nombre_value or final_nombre_value.strip() == '':
        fallback = col_nombre if col_nombre and col_nombre.strip() else "Sin nombre"
        final_nombre_value = fallback

    return str(final_nombre_value).strip()

## Módulo 7: Procesamiento de carpetas – parte 2 (lógica principal)

In [7]:
# ============================================================
# MÓDULO 7: PROCESAMIENTO DE CARPETAS – LÓGICA PRINCIPAL
# ============================================================

def _is_aggregation_candidate(col_nombre, col_descripcion, matched_df_column, new_format, custom_aggregation_keywords=None):
    """
    Determina si una columna del diccionario es candidata para ser sumada/agregada.
    Ahora acepta 'total' o 'subtotal' en la descripción y nombres de columna.
    """
    if not matched_df_column:
        return False, "no se encontró una columna coincidente en el DataFrame"

    col_nombre_lower = col_nombre.lower()
    col_desc_lower = col_descripcion.lower()

    # Excluir columnas que contengan 'clasificación'
    if 'clasificación' in col_nombre_lower or 'clasificacion' in col_nombre_lower or \
       'clasificación' in col_desc_lower or 'clasificacion' in col_desc_lower:
        return False, "contiene 'clasificación', no se suma"

    # 1. Verificar directamente en la descripción
    if 'total' in col_desc_lower:
        return True, "descripción contiene 'total' (incluye subtotal)"

    # 2. Verificar en el nombre de la columna
    if any(k in col_nombre_lower for k in ['total', 'sum', 'cantidad']):
        return True, "nombre contiene keyword (total/sum/cantidad)"

    # 3. Usar palabras clave con normalización
    keywords_to_check = []
    keywords_for_error_msg = []

    if custom_aggregation_keywords is not None and len(custom_aggregation_keywords) > 0:
        keywords_to_check = [normalize_string(k) for k in custom_aggregation_keywords]
        keywords_for_error_msg = custom_aggregation_keywords
    else:
        if new_format:
            keywords_to_check = ['total', 'totales', 'cantidad', 'sum']
            keywords_for_error_msg = ['total', 'totales', 'cantidad', 'sum']
        else:
            keywords_to_check = [
                normalize_string('cantidad de'),
                normalize_string('cantidad'),
                normalize_string('total'),
                normalize_string('suma'),
                normalize_string('totales')
            ]
            keywords_for_error_msg = ['cantidad de', 'cantidad', 'total', 'suma', 'totales']

    normalized_col_df = normalize_string(matched_df_column)
    normalized_col_nombre = normalize_string(col_nombre)
    normalized_col_descripcion = normalize_string(col_descripcion)

    for term in keywords_to_check:
        if term in normalized_col_nombre or term in normalized_col_descripcion or term in normalized_col_df:
            return True, f"coincidencia normalizada con '{term}'"

    return False, f"no coincide con los criterios de suma ({', '.join(keywords_for_error_msg)})"


def _load_dataframes(data_dir: Path, log_messages: List[str]) -> Dict[str, pd.DataFrame]:
    """
    Carga todos los archivos CSV de la carpeta data_dir en un diccionario.
    Versión optimizada con dtype=str y low_memory=False.
    """
    data_dfs = {}
    all_csv = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
    for file_name in all_csv:
        file_path = os.path.join(data_dir, file_name)
        try:
            try:
                df = pd.read_csv(
                    file_path,
                    encoding='utf-8-sig',
                    dtype=str,
                    low_memory=False
                )
            except UnicodeDecodeError:
                df = pd.read_csv(
                    file_path,
                    encoding='latin-1',
                    dtype=str,
                    low_memory=False
                )
            # Limpiar nombres de columnas
            df.columns = df.columns.str.replace(r'^\ufeff', '', regex=True).str.strip().str.lower()
            data_dfs[file_name] = df
            log_messages.append(f"  ✅ Dataset cargado: {file_name} ({len(df.columns)} cols, {len(df)} filas)")
        except Exception as e:
            log_messages.append(f"  ⚠️ Error al cargar '{file_name}': {e}")
    return data_dfs


def _find_catalog_dir(data_dir: Union[str, Path], dict_dir: Union[str, Path]) -> Path:
    """
    Encuentra la carpeta de catálogos (prioriza 'catalogos' en la raíz, luego en data_dir, luego dict_dir).
    """
    data_dir = Path(data_dir)
    dict_dir = Path(dict_dir)
    base_dir = data_dir.parent
    catalog_dir = base_dir / 'catalogos'
    if not catalog_dir.is_dir():
        catalog_dir = data_dir / 'catalogos'
    if not catalog_dir.is_dir():
        catalog_dir = dict_dir
    return catalog_dir


def _build_existing_hashes(log_messages=None):
    """
    Obtiene los datos existentes de Supabase y construye un diccionario de hashes.
    """
    if log_messages is None:
        log_messages = []
    existing_data = _get_all_variables_data()
    existing_hashes = {}
    for db_row in existing_data:
        norm_row = {}
        for k, v in db_row.items():
            if k in ['año', 'valor']:
                v = normalizar_numero_texto(v, k)
            norm_row[k] = safe_clean(v)
        for col in ['institucion', 'cobertura', 'periodicidad']:  # tipo_de_dato eliminado
            if col in norm_row and norm_row[col]:
                norm_row[col] = normalizar_valor_con_catalogo(norm_row[col], col, threshold=0.5)
        for col in ['proceso', 'eje', 'tema']:
            if col in norm_row and norm_row[col]:
                val_norm, _ = normalizar_categoria(norm_row[col], col, debug=False, threshold=0.8)
                norm_row[col] = val_norm
        if 'año' in norm_row:
            norm_row['año'] = safe_normalize(norm_row.get('año'), 'año')
            norm_row['año'] = str(int(float(norm_row['año']))) if norm_row['año'] and str(norm_row['año']).strip() != '' else ''
        if 'estado' in norm_row and (pd.isna(norm_row['estado']) or str(norm_row['estado']).strip() == ''):
            norm_row['estado'] = 'No aplica'
        h = _get_non_id_hash(norm_row)
        if h not in existing_hashes:
            existing_hashes[h] = norm_row
    return existing_hashes


def _identify_aggregation_columns(
    dict_files: List[str],
    data_dfs: Dict[str, pd.DataFrame],
    catalog_dir: Path,
    dict_dir: Path,
    new_format: bool,
    aggregation_keywords: List[str],
    indice_descriptions: Dict[str, str],
    log_messages: List[str]
) -> Dict[str, List[Tuple[str, str, str]]]:
    """
    Identifica, para cada archivo de datos, qué columnas de agregación (totales) deben procesarse.
    Versión robusta: sin indexación compleja, con conversión explícita a str y manejo de errores.
    """
    agg_info = {}

    # Preprocesar nombres de columnas de DataFrames para comparación (en minúsculas y sin acentos)
    df_columns_lower = {}
    for data_key, df in data_dfs.items():
        df_columns_lower[data_key] = [normalize_string(col) for col in df.columns]

    for dict_file_name in dict_files:
        dict_file_path = os.path.join(dict_dir, dict_file_name)
        try:
            format_type = 'new' if new_format else 'old'
            parsed_dict, _ = parse_dictionary_csv(dict_file_path, format_type=format_type, log_messages=log_messages)
            if not parsed_dict:
                log_messages.append(f"⚠️ No se pudo parsear el diccionario {dict_file_name}")
                continue

            log_messages.append(f"📄 Procesando diccionario: {dict_file_name}")

            if new_format:
                current_defs = infer_consecutive_descriptions(parsed_dict)
                base_name = os.path.splitext(dict_file_name)[0].replace('diccionario_de_datos_', '')
                log_messages.append(f"   Base_name extraído: '{base_name}'")

                # Buscar el dataset asociado por coincidencia de nombre (más flexible)
                matched_key = None
                for data_key in data_dfs.keys():
                    data_key_base = os.path.splitext(data_key)[0]
                    # Comparar ignorando mayúsculas/minúsculas y usando coincidencia parcial
                    if base_name.lower() in data_key_base.lower() or data_key_base.lower() in base_name.lower():
                        matched_key = data_key
                        break

                if not matched_key:
                    log_messages.append(f"   ⚠️ No se encontró dataset para base_name '{base_name}'")
                    continue

                df = data_dfs[matched_key]
                log_messages.append(f"   Asociando diccionario {dict_file_name} con datos {matched_key}")

                # Buscar catálogo de desagregación (si existe)
                first_col = df.columns[0]
                catalog = _find_disaggregation_catalog(first_col, catalog_dir)
                if catalog is None:
                    log_messages.append(f"   ⚠️ No se encontró catálogo para '{first_col}'. Se usará catálogo vacío.")
                    catalog = {}

                # Procesar cada definición
                for col_def in current_defs:
                    # Asegurar que los valores sean strings
                    col_nombre = str(col_def.get('Nombre de la columna', '')).strip()
                    col_desc = str(col_def.get('Descripcion', '')).strip()
                    if not col_nombre:
                        continue

                    # Buscar coincidencia en las columnas del DataFrame
                    matched_col = _find_matching_column(col_nombre, df)
                    if matched_col:
                        is_total, reason = _is_aggregation_candidate(
                            col_nombre, col_desc, matched_col, new_format, aggregation_keywords
                        )
                        if is_total:
                            nombre_base = _get_output_nombre_value(
                                col_nombre, col_desc, col_def, matched_key, new_format, indice_descriptions
                            )
                            agg_info.setdefault(matched_key, []).append((matched_col, col_desc, nombre_base))
                            log_messages.append(f"         ✅ Agregada columna '{matched_col}'")
                    else:
                        log_messages.append(f"      ⚠️ No se encontró columna '{col_nombre}' en {matched_key}")

            else:
                # Formato antiguo (sin cambios, ya que el error ocurre en nuevo)
                for data_file_key, column_defs in parsed_dict.items():
                    matched_key = None
                    for data_key in data_dfs.keys():
                        data_key_base = os.path.splitext(data_key)[0]
                        if data_file_key.lower() in data_key_base.lower() or data_key_base.lower() in data_file_key.lower():
                            matched_key = data_key
                            break
                    if not matched_key:
                        log_messages.append(f"   No se encontró dataset para la sección '{data_file_key}'")
                        continue

                    df = data_dfs[matched_key]
                    first_col = df.columns[0]
                    catalog = _find_disaggregation_catalog(first_col, catalog_dir)
                    if catalog is None:
                        log_messages.append(f"   ⚠️ No se encontró catálogo para '{first_col}'. Se usará catálogo vacío.")
                        catalog = {}

                    inferred_defs = infer_consecutive_descriptions(column_defs)
                    for col_def in inferred_defs:
                        col_nombre = str(col_def.get('Nombre de la columna', '')).strip()
                        col_desc = str(col_def.get('Descripcion', '')).strip()
                        if not col_nombre:
                            continue
                        matched_col = _find_matching_column(col_nombre, df)
                        if matched_col:
                            is_total, reason = _is_aggregation_candidate(
                                col_nombre, col_desc, matched_col, new_format, aggregation_keywords
                            )
                            if is_total:
                                nombre_base = _get_output_nombre_value(
                                    col_nombre, col_desc, col_def, matched_key, new_format, indice_descriptions
                                )
                                agg_info.setdefault(matched_key, []).append((matched_col, col_desc, nombre_base))

        except Exception as e:
            import traceback
            log_messages.append(f"⚠️ Error al leer diccionario '{dict_file_name}': {e}")
            log_messages.append(f"   Traceback: {traceback.format_exc()}")

    for data_file, agg_list in agg_info.items():
        log_messages.append(f"🔹 {data_file}: {len(agg_list)} columnas de agregación identificadas")
    return agg_info

def _process_disaggregation(data_file_name, agg_list, data_dfs, catalog_dir, existing_hashes,
                            global_metadata, log_messages):
    """
    Para un archivo de datos y su lista de columnas de agregación, ejecuta la desagregación.
    Si no hay catálogo, usa uno vacío y nombres genéricos.
    """
    if data_file_name not in data_dfs:
        return []
    df = data_dfs[data_file_name]
    first_col = df.columns[0]
    catalog = _find_disaggregation_catalog(first_col, catalog_dir)
    if catalog is None:
        catalog = {}
        log_messages.append(f"⚠️ No se encontró catálogo para '{first_col}' en '{data_file_name}'. Se usarán nombres genéricos.")

    agg_cols = [item[0] for item in agg_list]
    nombre_base = agg_list[0][2]
    if not nombre_base or nombre_base.strip() == '':
        nombre_base = f"Agregación de {data_file_name}"
        log_messages.append(f"   ⚠️ nombre_base vacío, se usará '{nombre_base}'")

    log_messages.append(f"🔍 Desagregación para '{data_file_name}' usando columna ID '{first_col}', columnas de agregación: {', '.join(agg_cols)}, nombre base: '{nombre_base}'")
    rows, msgs = _process_disaggregated_data(
        df, catalog, first_col, agg_cols, existing_hashes, nombre_base,
        global_metadata,
        log_messages=log_messages, debug=True
    )
    log_messages.extend(msgs)
    log_messages.append(f"✅ Procesado desagregación para '{data_file_name}' (total filas generadas: {len(rows)})")
    return rows


def _process_remaining_dictionaries(dict_files, data_dfs, processed_data_files, global_metadata_dict,
                                    new_format, aggregation_keywords, indice_descriptions,
                                    existing_hashes, dict_dir, log_messages):
    """
    Procesa los diccionarios que no fueron usados en la desagregación, generando filas de totales generales.
    """
    all_results = []
    for dict_file_name in dict_files:
        dict_file_path = os.path.join(dict_dir, dict_file_name)
        try:
            format_type = 'new' if new_format else 'old'
            parsed_dict, _ = parse_dictionary_csv(dict_file_path, format_type=format_type, log_messages=log_messages)
            if not parsed_dict:
                continue

            if new_format:
                current_defs = infer_consecutive_descriptions(parsed_dict)
                for col_def in current_defs:
                    col_nombre = col_def.get('Nombre de la columna', '')
                    col_desc = col_def.get('Descripcion', '')
                    found = False
                    for data_key, df in data_dfs.items():
                        if data_key in processed_data_files:
                            continue
                        matched_col = _find_matching_column(col_nombre, df)
                        if matched_col:
                            is_total, _ = _is_aggregation_candidate(col_nombre, col_desc, matched_col, new_format, aggregation_keywords)
                            if is_total:
                                output_row = _build_output_row(global_metadata_dict, df, matched_col, col_nombre, col_desc, col_def, data_key, new_format, indice_descriptions, "Total general")
                                row_hash = _get_non_id_hash(output_row)
                                if row_hash not in existing_hashes:
                                    all_results.append(output_row)
                                    log_messages.append(f"   ✅ Generada fila total para '{col_nombre}' en '{data_key}'")
                            found = True
                            break
                    if not found:
                        log_messages.append(f"    ⚠️ Columna '{col_nombre}' no procesada (no encontrada en ningún dataset).")
            else:
                for data_file_key, column_defs in parsed_dict.items():
                    selected_data = None
                    for actual_csv in data_dfs.keys():
                        if actual_csv in processed_data_files:
                            continue
                        base = os.path.splitext(actual_csv)[0].lower()
                        if base == data_file_key or actual_csv.startswith(data_file_key + '_') or actual_csv.startswith(data_file_key + '.'):
                            selected_data = actual_csv
                            break
                    if not selected_data:
                        continue
                    df = data_dfs[selected_data]
                    inferred_defs = infer_consecutive_descriptions(column_defs)
                    for col_def in inferred_defs:
                        col_nombre = col_def.get('Nombre de la columna', '')
                        col_desc = col_def.get('Descripcion', '')
                        matched_col = _find_matching_column(col_nombre, df)
                        if matched_col:
                            is_total, _ = _is_aggregation_candidate(col_nombre, col_desc, matched_col, new_format, aggregation_keywords)
                            if is_total:
                                output_row = _build_output_row(global_metadata_dict, df, matched_col, col_nombre, col_desc, col_def, selected_data, new_format, indice_descriptions, "Total general")
                                row_hash = _get_non_id_hash(output_row)
                                if row_hash not in existing_hashes:
                                    all_results.append(output_row)
                                    log_messages.append(f"   ✅ Generada fila total para '{col_nombre}' en '{selected_data}'")
        except Exception as e:
            log_messages.append(f"❌ Error al procesar diccionario '{dict_file_name}': {e}")
    return all_results


def _build_output_row(global_meta, df, col_name, col_nombre, col_desc, col_def, data_file_key, new_format, indice_descriptions, suffix=""):
    numeric_series = to_numeric_clean(df[col_name])
    total_sum = numeric_series.sum() if not numeric_series.isna().all() else None

    output_row = global_meta.copy()

    # Asignar año
    if not output_row.get('año'):
        if output_row.get('fuente'):
            year_match = re.search(r'\b(19|20)\d{2}\b', output_row['fuente'])
            if year_match:
                output_row['año'] = year_match.group(0)
        if not output_row.get('año') and data_file_key:
            year_match = re.search(r'(19|20)\d{2}', data_file_key)
            if year_match:
                output_row['año'] = year_match.group(0)
        if not output_row.get('año'):
            output_row['año'] = ''
    if output_row.get('año'):
        try:
            output_row['año'] = str(int(float(output_row['año'])))
        except:
            pass

    output_row['valor'] = total_sum
    # tipo_de_dato eliminado

    # Obtener nombre con fallback robusto
    nombre_tmp = _get_output_nombre_value(col_nombre, col_desc, col_def, data_file_key, new_format, indice_descriptions)
    if not nombre_tmp or nombre_tmp.strip() == '':
        nombre_tmp = col_nombre if col_nombre else col_desc
        if not nombre_tmp or nombre_tmp.strip() == '':
            nombre_tmp = f"Variable de {data_file_key}" if data_file_key else "Variable sin nombre"
    if suffix:
        nombre_tmp = f"{nombre_tmp} - {suffix}"
    output_row['nombre'] = nombre_tmp
    output_row['estado'] = 'No aplica'

    # Asignar categorías con fallback
    proc, eje, tema = predict_categories(nombre_tmp)
    if not proc:
        proc = list(KEYWORDS['proceso'].keys())[0]
    if not eje:
        eje = list(KEYWORDS['eje'].keys())[0]
    if not tema:
        tema = list(KEYWORDS['tema'].keys())[0]

    # Consistencia eje-tema
    eje_num = re.search(r'Eje\s*(\d+)', eje)
    tema_num = re.search(r'^(\d+)\.', tema)
    if eje_num and tema_num and eje_num.group(1) != tema_num.group(1):
        posibles_temas = [t for t in KEYWORDS['tema'].keys() if t.startswith(eje_num.group(1) + '.')]
        if posibles_temas:
            tema = posibles_temas[0]

    output_row['proceso'] = proc
    output_row['eje'] = eje
    output_row['tema'] = tema

    return output_row

def _find_matching_column(col_name, df):
    """Busca una columna en el DataFrame por coincidencia exacta o difusa."""
    col_name = str(col_name).strip()
    if not col_name:
        return None
    if col_name in df.columns:
        return col_name

    norm_cn = normalize_string(col_name)
    for df_col in df.columns:
        if normalize_string(df_col) == norm_cn:
            return df_col

    best_match = None
    best_score = 0.7
    for df_col in df.columns:
        score = SequenceMatcher(None, norm_cn, normalize_string(df_col)).ratio()
        if score > best_score:
            best_score = score
            best_match = df_col
    return best_match


def _filter_and_build_dataframe(all_results, global_metadata, log_messages):
    messages = []
    if not all_results:
        final_df = pd.DataFrame(columns=EXPECTED_COLUMNS + ['valor', 'parent_id'])
        messages.append("⚠️ No se encontraron filas que cumplan los criterios de agregación.")
        log_messages.extend(messages)
        return final_df, messages, []

    messages.append(f"🔍 Total de filas antes del filtro: {len(all_results)}")
    filtered_row_status = []
    filtered = []

    for row in all_results:
        data_row = row.get('data', row)
        nombre = data_row.get('nombre', '').strip()
        valor = data_row.get('valor')

        if not nombre:
            nombre = f"Variable sin nombre (id: {data_row.get('id', 'desconocido')})"
            data_row['nombre'] = nombre
            messages.append(f"   ⚠️ Nombre vacío, se asignó: '{nombre}'")

        if valor is None:
            continue
        filtered_row_status.append(row)
        filtered.append(data_row)

    messages.append(f"🧹 Filtradas filas con valor None. Quedan {len(filtered)} filas.")

    if not filtered:
        final_df = pd.DataFrame(columns=EXPECTED_COLUMNS + ['valor', 'parent_id'])
        messages.append("⚠️ No se encontraron filas válidas después del filtrado.")
        log_messages.extend(messages)
        return final_df, messages, []

    # Conservar todas las columnas, incluyendo parent_id
    all_cols = set()
    for row in filtered:
        all_cols.update(row.keys())
    for col in EXPECTED_COLUMNS:
        all_cols.add(col)
    all_cols.add('valor')
    if any('parent_id' in row for row in filtered):
        all_cols.add('parent_id')

    final_df = pd.DataFrame(filtered, columns=list(all_cols))
    for col in all_cols:
        if col not in final_df.columns:
            final_df[col] = ''

    # Normalizar año, estado
    if 'año' in final_df.columns:
        final_df['año'] = final_df['año'].apply(lambda x: str(int(float(x))) if pd.notna(x) and str(x).strip() != '' else '')
    if 'estado' in final_df.columns:
        final_df['estado'] = final_df['estado'].apply(lambda x: 'No aplica' if pd.isna(x) or str(x).strip()=='' else x)

    # Normalización vectorizada de catálogos
    catalog_cols = ['institucion', 'cobertura', 'periodicidad', 'estado', 'proceso', 'eje', 'tema']
    existing_catalog_cols = [col for col in catalog_cols if col in final_df.columns]
    if existing_catalog_cols:
        final_df = _normalize_catalog_values_vectorized(final_df, existing_catalog_cols)

    log_messages.extend(messages)
    return final_df, messages, filtered_row_status

def process_data_folders_with_return(data_dir, dict_dir, metadatos_path=None, new_format=False,
                                     indice_filename=None, aggregation_keywords=None,
                                     log_messages=None, parallel=False):
    """
    Procesa carpetas de datos y diccionarios para extraer variables agregadas.
    (Versión refactorizada con funciones auxiliares, admite paralelismo opcional)
    """
    if log_messages is None:
        log_messages = []
    log_messages.append("🚀 ENTRANDO A process_data_folders_with_return")

    if not os.path.isdir(dict_dir) or not os.path.isdir(data_dir):
        log_messages.append("❌ Error: No se encontraron las carpetas 'diccionario_de_datos' o 'conjunto_de_datos'.")
        return pd.DataFrame(columns=EXPECTED_COLUMNS + ['valor']), {}, []

    data_dfs = _load_dataframes(data_dir, log_messages)
    if not data_dfs:
        log_messages.append("❌ No se encontraron archivos CSV válidos en 'conjunto_de_datos'.")
        return pd.DataFrame(columns=EXPECTED_COLUMNS + ['valor']), {}, []

    catalog_dir = _find_catalog_dir(data_dir, dict_dir)
    log_messages.append(f"📁 Ruta de catálogos: {catalog_dir}")

    current_metadatos_text = ""
    if metadatos_path and os.path.isfile(metadatos_path):
        try:
            with open(metadatos_path, 'r', encoding='utf-8') as f:
                current_metadatos_text = f.read()
            log_messages.append(f"📄 Metadatos cargados desde: {metadatos_path}")
        except Exception as e:
            log_messages.append(f"⚠️ Error al leer metadatos: {e}")
    global_metadata_dict = parse_global_metadatos(current_metadatos_text, log_messages)
    log_messages.append("📊 Metadatos globales analizados.")
    log_messages.append(f"   Año extraído de metadatos: {global_metadata_dict.get('año', '')}")

    indice_descriptions = {}
    if new_format and indice_filename:
        indice_file_path_full = os.path.join(data_dir, indice_filename)
        try:
            indice_df = pd.read_csv(indice_file_path_full, encoding='utf-8')
            if 'ARCHIVO' in indice_df.columns and 'CONTENIDO' in indice_df.columns:
                indice_df['ARCHIVO_BASE'] = indice_df['ARCHIVO'].apply(lambda x: x.split('_')[0].lower() if isinstance(x, str) else '')
                indice_descriptions = dict(zip(indice_df['ARCHIVO_BASE'], indice_df['CONTENIDO']))
                log_messages.append(f"📑 Archivo de índice cargado: {len(indice_descriptions)} descripciones.")
            else:
                log_messages.append("⚠️ El índice no tiene columnas 'ARCHIVO' y 'CONTENIDO'.")
        except Exception as e:
            log_messages.append(f"⚠️ Error al cargar índice: {e}")

    existing_hashes = _build_existing_hashes(log_messages)

    if aggregation_keywords is None:
        aggregation_keywords = ['total', 'totales', 'cantidad', 'sum'] if new_format else ['cantidad de', 'total']

    dict_files = [f for f in os.listdir(dict_dir) if f.endswith('.csv')]
    agg_info = _identify_aggregation_columns(
        dict_files, data_dfs, catalog_dir, dict_dir, new_format,
        aggregation_keywords, indice_descriptions, log_messages
    )

    # ======================== FALLBACK AUTOMÁTICO =========================
    if not agg_info:
        log_messages.append("⚠️ No se detectaron columnas de agregación en los diccionarios. Intentando desagregación automática...")
        for data_file, df in data_dfs.items():
            if len(df.columns) < 2:
                continue
            # Usar la primera columna como clave de desagregación
            id_col = df.columns[0]
            # Buscar catálogo para esa columna
            catalog = _find_disaggregation_catalog(id_col, catalog_dir)
            if catalog is None:
                catalog = {}
                log_messages.append(f"   ⚠️ No se encontró catálogo para '{id_col}'. Se usarán nombres genéricos.")
            # Identificar columnas numéricas (excluyendo la primera)
            numeric_cols = []
            for col in df.columns[1:]:
                try:
                    numeric_series = pd.to_numeric(df[col], errors='coerce')
                    if numeric_series.notna().sum() > 0:
                        numeric_cols.append(col)
                except:
                    pass
            if not numeric_cols:
                log_messages.append(f"   ⚠️ No se encontraron columnas numéricas en {data_file} para desagregar.")
                continue
            # Tomar la primera columna numérica como agregación (o todas si se prefiere)
            agg_cols = numeric_cols[:1]  # o numeric_cols para todas
            # Generar nombre base a partir del nombre del archivo
            nombre_base = os.path.splitext(data_file)[0].replace('_', ' ').title()
            log_messages.append(f"   ✅ Desagregación automática para {data_file}: id_col='{id_col}', agg_cols={agg_cols}, nombre_base='{nombre_base}'")
            agg_info[data_file] = [(col, '', nombre_base) for col in agg_cols]
    # ====================================================================

    print(f"🔍 agg_info tiene {len(agg_info)} entradas: {agg_info.keys()}")

    all_results = []
    processed_data_files = set()

    # --- PROCESAMIENTO CON PARALELISMO OPCIONAL ---
    if parallel and len(agg_info) > 1:
        log_messages.append("⚡ Procesando desagregación en paralelo...")
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
            futures = {}
            for data_file, agg_list in agg_info.items():
                future = executor.submit(
                    _process_disaggregation,
                    data_file, agg_list, data_dfs, catalog_dir,
                    existing_hashes, global_metadata_dict, log_messages
                )
                futures[future] = data_file
            for future in concurrent.futures.as_completed(futures):
                try:
                    rows = future.result()
                    all_results.extend(rows)
                    processed_data_files.add(futures[future])
                except Exception as e:
                    log_messages.append(f"❌ Error en procesamiento paralelo: {e}")
    else:
        for data_file, agg_list in agg_info.items():
            rows = _process_disaggregation(data_file, agg_list, data_dfs, catalog_dir,
                                           existing_hashes, global_metadata_dict, log_messages)
            all_results.extend(rows)
            processed_data_files.add(data_file)

    # Procesar diccionarios restantes
    remaining_rows = _process_remaining_dictionaries(
        dict_files, data_dfs, processed_data_files, global_metadata_dict,
        new_format, aggregation_keywords, indice_descriptions,
        existing_hashes, dict_dir, log_messages
    )
    all_results.extend(remaining_rows)

    # Fallback: columnas numéricas si no hay resultados
    if not all_results:
        log_messages.append("⚠️ No se encontraron columnas de totales. Intentando fallback con columnas numéricas...")
        for data_file, df in data_dfs.items():
            if data_file in processed_data_files:
                continue
            for col in df.columns:
                numeric_series = to_numeric_clean(df[col], log_messages)
                if numeric_series.notna().sum() > 0:
                    col_lower = col.lower()
                    if any(key in col_lower for key in ['id', 'clave', 'código', 'codigo', 'nombre', 'descripcion']):
                        continue
                    total_sum = numeric_series.sum()
                    log_messages.append(f"   🔍 Columna numérica encontrada: '{col}' en '{data_file}', suma={total_sum}")
                    output_row = global_metadata_dict.copy()
                    output_row['año'] = _extract_year(data_file, dict_dir, log_messages)
                    output_row['valor'] = total_sum
                    output_row['nombre'] = f"Suma de {col} (fallback)"
                    output_row['estado'] = 'No aplica'
                    row_hash = _get_non_id_hash(output_row)
                    if row_hash not in existing_hashes:
                        fallback_row_status = {
                            'original_csv_index': -1,
                            'data': output_row,
                            '_is_duplicate': False,
                            '_supabase_matching_id': None,
                            '_hash': row_hash,
                        }
                        all_results.append(fallback_row_status)
                        log_messages.append(f"   ✅ Generada fila fallback para columna '{col}'")
        if not all_results:
            log_messages.append("❌ No se encontró ninguna columna numérica válida para generar filas.")

    final_df, filtered_messages, filtered_row_status = _filter_and_build_dataframe(all_results, global_metadata_dict, log_messages)
    log_messages.extend(filtered_messages)

    return final_df, global_metadata_dict, filtered_row_status

def _extract_year(*sources, log_messages=None):
    """
    Extrae el primer año (19xx o 20xx) encontrado en las cadenas proporcionadas.
    """
    for s in sources:
        if not s:
            continue
        match = re.search(r'(19|20)\d{2}', s)
        if match:
            year = match.group(0)
            try:
                year = str(int(float(year)))
            except:
                pass
            if log_messages is not None:
                log_messages.append(f"   Año extraído de: {s[:50]}... -> {year}")
            return year
    return ''

## Módulo 8: Interfaz completa (CSS, HTML, JS)

### CSS

In [8]:
# ============================================================
# MÓDULO 8: INTERFAZ COMPLETA (CSS, HTML, JS)
# ============================================================
CSS = """
:root {
    --primary: #4A5568;
    --primary-bright: #718096;
    --primary-dark: #2D3748;
    --bg: #F8F9FA;
    --panel-bg: #ffffff;
    --text-main: #212529;
    --text-muted: #6C757D;
    --border: #DEE2E6;
    --dark-header: #2C3E50;
    --table-header-bg: #2C3E50;
    --sort-neutral: #6C757D;
    --sort-active: #E72000;
    --selection-gray: #E9ECEF;
    --shadow: 0 2px 4px rgba(0,0,0,0.08);
    --danger: #dc3545;
    --success: #28a745;
}
* { box-sizing: border-box; }
body {
    font-family: 'Inter', sans-serif;
    margin: 0; padding: 0;
    background: var(--bg);
    color: var(--text-main);
    line-height: 1.6;
}
@keyframes fadeInSlideUp {
    from { opacity: 0; transform: translateY(30px); }
    to { opacity: 1; transform: translateY(0); }
}
@keyframes modalIn {
    from { transform: translateY(30px); opacity: 0; }
    to { transform: translateY(0); opacity: 1; }
}
.header-bar {
    background: var(--dark-header);
    color: white;
    padding: 16px 32px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    box-shadow: 0 2px 6px rgba(0,0,0,0.1);
    opacity: 0;
    animation: fadeInSlideUp 0.9s ease-out forwards;
    animation-delay: 0.1s;
}
.header-bar h1 { margin:0; font-size:1.25em; letter-spacing:-0.02em; }
.container {
    margin: 24px auto;
    width: 95%;
    max-width: 2200px;
}
.tool-panel {
    background: var(--panel-bg);
    padding: 16px 20px;
    border-radius: 8px;
    border: 1px solid var(--border);
    box-shadow: var(--shadow);
    margin-bottom: 24px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 12px;
    flex-wrap: wrap;
    opacity: 0;
    animation: fadeInSlideUp 0.9s ease-out forwards;
    animation-delay: 0.6s;
}
.tool-panel .search-wrapper { flex: 1 1 250px; min-width: 180px; }
.tool-panel .search-wrapper input {
    width: 100%;
    padding: 10px 16px;
    border: 1px solid var(--border);
    border-radius: 6px;
    outline: none;
    font-size: 0.95em;
    background: white;
    transition: border-color 0.2s, box-shadow 0.2s;
}
.tool-panel .search-wrapper input:focus {
    border-color: var(--primary);
    box-shadow: 0 0 0 3px rgba(74,85,104,0.15);
}
.tool-panel .button-group { display: flex; gap: 8px; flex-wrap: wrap; }
.btn {
    padding: 8px 18px;
    border-radius: 6px;
    border: 1px solid transparent;
    font-weight: 500;
    font-size: 0.85rem;
    transition: all 0.2s ease;
    box-shadow: 0 1px 3px rgba(0,0,0,0.08);
    letter-spacing: 0.01em;
    cursor: pointer;
    display: inline-flex;
    align-items: center;
    gap: 6px;
    background: white;
    color: var(--text-main);
}
.btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 14px rgba(0,0,0,0.12);
}
.btn-tool { background: var(--sort-active); color: #fff; border-color: var(--sort-active); }
.btn-tool:hover { background: #cc1c00; border-color: #cc1c00; }
.btn-action { background: var(--primary-bright); color: #fff; border-color: var(--primary-bright); }
.btn-action:hover { background: #4a5568; border-color: #4a5568; }
.btn-success { background: var(--success); color: #fff; border-color: var(--success); }
.btn-success:hover { background: #1e7e34; border-color: #1e7e34; }
.btn-outline { background: transparent; color: var(--text-main); border: 1px solid var(--border); }
.btn-outline:hover { background: var(--selection-gray); border-color: var(--primary-bright); }
.btn-danger { background: var(--danger); color: #fff; border-color: var(--danger); }
.btn-danger:hover { background: #bd2130; border-color: #bd2130; }
.btn:disabled { opacity: 0.6; cursor: not-allowed; transform: none !important; }

.table-card {
    background: var(--panel-bg);
    border-radius: 8px;
    border: 1px solid var(--border);
    box-shadow: 0 4px 12px -2px rgba(0,0,0,0.06);
    overflow: hidden;
    position: relative;
    opacity: 0;
    animation: fadeInSlideUp 0.9s ease-out forwards;
    animation-delay: 1.1s;
}
.table-card.drag-over { border: 2px dashed var(--primary); }
.table-card.drag-over::after {
    content: 'Soltar CSV para importar';
    position: absolute; top:0; left:0; width:100%; height:100%;
    background: rgba(255,255,255,0.85);
    z-index:100;
    display:flex; align-items:center; justify-content:center;
    font-weight:600; font-size:1.5em; color:var(--primary);
    pointer-events:none;
}
.table-wrapper { overflow-x: auto; -webkit-overflow-scrolling: touch; }
table { border-collapse: collapse; width: 100%; min-width: 800px; }
th {
    background: var(--table-header-bg);
    color: #F8FAFC !important;
    font-weight: 600;
    font-size: 0.8em;
    text-transform: uppercase;
    letter-spacing: 0.05em;
    border-bottom: 1px solid #334155;
    padding: 10px 14px;
    word-wrap: break-word;
    white-space: normal;
    position: sticky;
    top: 0;
    z-index: 10;
    text-align: center;
    user-select: none;
}
.th-content {
    display: flex;
    justify-content: space-between;
    align-items: center;
    width: 100%;
}
.th-label {
    flex: 1;
    text-align: left;
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
}
.sort-indicators {
    display: flex;
    flex-direction: column;
    align-items: center;
    flex-shrink: 0;
    margin-left: 6px;
    line-height: 0.6;
    font-size: 0.7em;
}
.tri-up, .tri-down {
    color: var(--sort-neutral);
    transition: color 0.1s, transform 0.1s;
    cursor: pointer;
    padding: 1px 3px;
    user-select: none;
}
.tri-up:hover, .tri-down:hover { color: var(--sort-active); transform:scale(1.1); }
th.sorted-asc .tri-up { color: var(--sort-active) !important; text-shadow: 0 0 8px rgba(231,32,0,0.4); }
th.sorted-desc .tri-down { color: var(--sort-active) !important; text-shadow: 0 0 8px rgba(231,32,0,0.4); }

td {
    vertical-align: middle;
    padding: 2px 4px !important;
    min-height: 28px !important;
    box-sizing: border-box;
    word-break: break-word;
    white-space: normal;
    overflow-wrap: break-word;
    text-align: left !important;
}
td:first-child {
    text-align: center !important;
    width: 40px !important;
    min-width: 40px !important;
    max-width: 40px !important;
}
td:first-child input[type="checkbox"] { margin: 0; }

.edit-control, .read-only-input {
    width: 100%;
    min-height: 28px !important;
    padding: 2px 4px !important;
    font-size: 0.8em !important;
    font-family: inherit;
    border: 1px solid transparent;
    background: transparent;
    outline: none;
    line-height: 1.4;
    box-sizing: border-box;
    transition: border-color 0.2s, box-shadow 0.2s;
    color: #212529;
    vertical-align: top;
    text-align: left;
}
textarea.edit-control {
    height: auto;
    resize: none;
    overflow: hidden;
    white-space: pre-wrap;
    word-break: break-word;
}
textarea.edit-control:focus {
    border-color: var(--primary) !important;
    background: white !important;
    box-shadow: 0 0 0 3px rgba(74,85,104,0.15) !important;
}
select.edit-control {
    height: 28px !important;
    padding: 2px 4px !important;
    font-size: 0.8em !important;
    appearance: none;
    -webkit-appearance: none;
    cursor: pointer;
    overflow: visible;
    text-align-last: left;
    width: 100%;
    text-overflow: clip;
    white-space: nowrap;
}
select.edit-control option { white-space: nowrap; overflow: visible; }
.read-only-input { background: transparent; border: none; cursor: default; text-align: left; }
td.cell-focused { box-shadow: 0 4px 12px rgba(0,0,0,0.3); z-index: 2; position: relative; }
tr:hover td { background-color: var(--selection-gray); }
tr.selected td { background-color: var(--selection-gray) !important; }

.row-checkbox {
    width: 40px;
    min-width: 40px;
    max-width: 40px;
    text-align: center;
    vertical-align: middle;
    padding: 0 4px;
}
.row-checkbox input[type="checkbox"] { cursor: pointer; width: 16px; height: 16px; accent-color: var(--primary); }

.custom-file-upload {
    display: inline-block;
    padding: 10px 20px;
    cursor: pointer;
    background: var(--primary-bright);
    color: white !important;
    border-radius: 6px;
    border: none;
    font-weight: 500;
    font-size: 0.9em;
    transition: background 0.2s;
}
.custom-file-upload:hover { background: #4a5568; }
.custom-file-upload input[type="file"] { display: none; }
.file-input-wrapper { margin: 10px 0; }

.modal-overlay {
    position: fixed; top:0; left:0; width:100%; height:100%;
    background: rgba(0,0,0,0.5);
    display: none;
    align-items: center;
    justify-content: center;
    z-index: 1000;
    backdrop-filter: blur(4px);
}
#modalOtraOpcion, #modalSuggestion { z-index: 2000 !important; }
.modal-content {
    background: white;
    padding: 32px 40px;
    border-radius: 12px;
    width: 90%;
    max-width: 825px;
    box-shadow: 0 20px 40px -8px rgba(0,0,0,0.3);
    animation: modalIn 0.3s cubic-bezier(0.16,1,0.3,1);
    border: 1px solid var(--border);
    max-height: 90vh;
    overflow-y: auto;
    position: relative;
}
.modal-header {
    margin-bottom: 20px;
    border-bottom: 1px solid var(--border);
    padding-bottom: 14px;
    position: relative;
}
.modal-header h2 { margin:0; font-size:1.5em; color:var(--text-main); font-weight:600; }
.close-button {
    position: absolute;
    top: 12px;
    right: 16px;
    background: none;
    border: none;
    font-size: 1.6em;
    cursor: pointer;
    color: var(--text-muted);
    transition: color 0.2s;
    line-height: 1;
    padding: 0 4px;
}
.close-button:hover { color: var(--primary-dark); }
.form-group { margin-bottom: 18px; }
.form-group label {
    display: block;
    font-size: 0.8em;
    font-weight: 600;
    color: var(--text-muted);
    margin-bottom: 6px;
    text-transform: uppercase;
    letter-spacing: 0.04em;
}
.form-input {
    width: 100%;
    padding: 10px 14px;
    border: 1px solid var(--border);
    border-radius: 6px;
    font-family: inherit;
    font-size: 1em;
    background: #f8fafa;
    transition: border-color 0.2s, box-shadow 0.2s;
    line-height: 1.5;
    resize: none;
    overflow: hidden;
    color: #212529;
}
textarea.form-input { min-height: 32px; overflow-y:hidden; }
.form-input:focus {
    border-color: var(--primary);
    background: white;
    box-shadow: 0 0 0 3px rgba(74,85,104,0.15);
}
.form-input::placeholder { color: #adb5bd; }
#toast-container {
    position: fixed; bottom:30px; right:30px; z-index:9999;
    display: flex;
    flex-direction: column;
    gap: 8px;
}
.toast {
    background: var(--dark-header);
    color: white;
    padding: 12px 24px;
    border-radius: 6px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.15);
    animation: fadeInSlideUp 0.3s ease-out;
}
#uploadPreviewMessages {
    max-height:120px; overflow-y:auto; margin-bottom:15px; font-size:0.9em;
    color:var(--text-muted); border:1px solid var(--border); padding:10px;
    border-radius:6px; background:#fdfdfd;
}
#uploadPreviewMessages div { margin-bottom:4px; line-height:1.4; color:var(--text-main); padding:2px 0; }
#uploadDataPreview, #zipPreviewTable {
    border-radius: 8px;
    border: 1px solid var(--border);
    overflow: auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.05);
    max-height: 300px;
    margin-bottom: 15px;
}
#uploadDataPreview table, #zipPreviewTable table {
    width: 100%;
    border-collapse: collapse;
    font-size: 0.85rem;
    table-layout: fixed;
}
#uploadDataPreview th, #zipPreviewTable th {
    background: var(--table-header-bg);
    color: #f8fafc;
    font-weight: 600;
    padding: 8px 10px;
    border-bottom: 2px solid #334155;
    text-transform: uppercase;
    letter-spacing: 0.04em;
    position: sticky;
    top: 0;
    z-index: 2;
    text-align: left;
    vertical-align: top;
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
}
#uploadDataPreview td, #zipPreviewTable td {
    padding: 2px 4px !important;
    border-bottom: 1px solid var(--border);
    vertical-align: top;
    text-align: left !important;
    word-break: break-word;
    white-space: normal;
    overflow-wrap: break-word;
}
#uploadDataPreview tbody tr:nth-child(even), #zipPreviewTable tbody tr:nth-child(even) { background: #f8fafc; }
#uploadDataPreview tbody tr:hover, #zipPreviewTable tbody tr:hover { background: var(--selection-gray); }
.duplicate-row-preview { background: #fff5f5 !important; border-left: 4px solid var(--danger); }
.duplicate-row-preview td { background: transparent !important; }

#uploadDataPreview textarea.preview-edit,
#zipPreviewTable textarea.preview-edit {
    width: 100%;
    min-height: 28px;
    padding: 2px 4px;
    border: none !important;
    background: transparent !important;
    font-family: inherit;
    font-size: 0.8em;
    color: #212529;
    outline: none;
    resize: none;
    overflow: hidden;
    white-space: pre-wrap;
    word-break: break-word;
    box-shadow: none;
    cursor: default;
    text-align: left;
    vertical-align: top;
    box-sizing: border-box;
    display: block;
    height: auto;
}
#uploadDataPreview select.preview-edit,
#zipPreviewTable select.preview-edit,
#uploadDataPreview span.preview-edit,
#zipPreviewTable span.preview-edit {
    width: 100%;
    min-height: 28px;
    padding: 2px 4px;
    border: none !important;
    background: transparent !important;
    font-family: inherit;
    font-size: 0.8em;
    color: #212529;
    outline: none;
    resize: none;
    overflow: hidden;
    white-space: pre-wrap;
    word-break: break-word;
    box-shadow: none;
    cursor: default;
    text-align: left;
    vertical-align: top;
    box-sizing: border-box;
    display: block;
}
#uploadDataPreview select.preview-edit,
#zipPreviewTable select.preview-edit {
    appearance: none;
    -webkit-appearance: none;
    cursor: pointer;
    padding-right: 16px;
    background: transparent;
}
#paginationControls {
    display:flex; justify-content:center; align-items:center; gap:10px;
    margin-top:10px; margin-bottom:15px;
}
#paginationControls .btn {
    padding:6px 14px;
    border-radius:6px;
    font-size:0.85em;
    background:white;
    border:1px solid var(--border);
    color:var(--text-main);
}
#paginationControls .btn:disabled {
    background:#f1f3f5;
    color:#adb5bd;
    cursor:not-allowed;
    transform:none;
    box-shadow:none;
}
.table-pagination {
    display:flex; justify-content:center; align-items:center; gap:12px;
    padding:14px 20px;
    background:white;
    border-top:1px solid var(--border);
    border-bottom-left-radius:8px;
    border-bottom-right-radius:8px;
    flex-wrap:wrap;
}
.table-pagination button {
    padding:6px 16px;
    border-radius:6px;
    border:1px solid var(--border);
    background:white;
    cursor:pointer;
    font-weight:500;
    font-size:0.85em;
    transition:all 0.2s;
}
.table-pagination button:hover:not(:disabled) {
    background:var(--selection-gray);
    border-color:var(--primary-bright);
}
.table-pagination button:disabled {
    color:var(--text-muted);
    cursor:not-allowed;
    background:#f8f9fa;
}
.table-pagination span {
    font-size:0.9em;
    color:var(--text-main);
    display:flex;
    align-items:center;
    gap:6px;
}
.table-pagination input[type="number"] {
    width:56px;
    padding:6px 8px;
    border:1px solid var(--border);
    border-radius:6px;
    text-align:center;
    font-size:0.9em;
    background:white;
}
.table-pagination select {
    padding:6px 10px;
    border:1px solid var(--border);
    border-radius:6px;
    font-size:0.9em;
    background:white;
    min-width:65px;
}
.child-row.hidden { display: none; }
.expand-btn {
    cursor: pointer;
    display: inline-block;
    width: 24px;
    height: 24px;
    line-height: 24px;
    text-align: center;
    font-size: 1.2em;
    border-radius: 50%;
    background: #f0f0f0;
    color: #333;
    transition: background 0.2s, transform 0.2s;
    user-select: none;
    box-shadow: 0 1px 3px rgba(0,0,0,0.1);
    display: inline-flex;
    align-items: center;
    justify-content: center;
}
.expand-btn:hover {
    background: #ddd;
    transform: scale(1.1);
}
.expand-btn:active {
    transform: scale(0.9);
}
.child-row td:first-child {
    padding-left: 24px !important;
}
.child-row-preview td:first-child {
    padding-left: 24px !important;
}
.has-new-catalog td { background-color: #fff9e6 !important; }
#historyContainer {
    margin-top: 24px;
    background: var(--panel-bg);
    border-radius: 8px;
    border: 1px solid var(--border);
    padding: 16px 20px;
    box-shadow: var(--shadow);
}
#historyContainer h3 {
    margin-top: 0;
    margin-bottom: 12px;
    color: var(--text-main);
    font-size: 1.1em;
    font-weight: 600;
}
#historyList {
    max-height: 300px;
    overflow-y: auto;
    font-size: 0.9em;
}
#historyList table {
    width: 100%;
    border-collapse: collapse;
    table-layout: fixed;
}
#historyList th, #historyList td {
    padding: 10px 16px;
    border-bottom: 1px solid var(--border);
    text-align: left;
}
#historyList th {
    background: var(--table-header-bg);
    color: #F8FAFC;
    font-weight: 600;
}
#historyList th:nth-child(2), #historyList td:nth-child(2) { min-width: 200px; width: 30%; }
#historyList th:nth-child(1), #historyList td:nth-child(1) { width: 25%; }
#historyList th:nth-child(3), #historyList td:nth-child(3) { width: 15%; }
#historyList th:nth-child(4), #historyList td:nth-child(4) { width: 20%; }
.status-success { color: var(--success); font-weight: 600; }
.status-error { color: var(--danger); font-weight: 600; }
.status-pending { color: #ffc107; font-weight: 600; }
.progress-bar {
    width: 100%;
    height: 20px;
    background: #e9ecef;
    border-radius: 6px;
    overflow: hidden;
    margin: 10px 0;
}
.progress-bar-fill {
    height: 100%;
    background: var(--sort-active);
    width: 0%;
    transition: width 0.5s ease;
}
/* Ocultar filas hijas colapsadas */
.child-row.hidden,
.child-row-preview.hidden {
    display: none;
}
.hidden {
    display: none;
}
@media (max-width: 768px) {
    .tool-panel { flex-direction: column; align-items: stretch; }
    .tool-panel .search-wrapper { flex: 1 1 auto; }
    .tool-panel .button-group { justify-content: center; }
    .modal-content { padding: 24px 20px; }
    .table-pagination { flex-wrap: wrap; gap: 8px; }
}

/* Estilos para dropdown de usuario */
.user-dropdown {
    position: relative;
    display: inline-block;
}
.user-dropdown-menu {
    position: absolute;
    top: calc(100% + 6px);
    right: 0;
    background: white;
    border: 1px solid var(--border);
    border-radius: 8px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.15);
    min-width: 200px;
    z-index: 1000;
    padding: 6px 0;
}
.user-dropdown-menu a {
    display: block;
    padding: 10px 16px;
    color: var(--text-main);
    text-decoration: none;
    transition: background 0.2s;
    cursor: pointer;
    font-size: 0.9rem;
}
.user-dropdown-menu a:hover {
    background: var(--selection-gray);
}
.user-dropdown-menu hr {
    margin: 6px 0;
    border: none;
    border-top: 1px solid var(--border);
}
#userDropdownBtn {
    background: transparent;
    border: 1px solid var(--border);
    color: var(--text-main);
    padding: 8px 14px;
    border-radius: 6px;
    font-size: 0.85rem;
    display: flex;
    align-items: center;
    gap: 6px;
}
#userDropdownBtn:hover {
    background: var(--selection-gray);
}
#userDropdownBtn .arrow {
    font-size: 0.65rem;
    margin-left: 4px;
}


/* Estilos para navegación por vistas */
.view-container {
    display: none;
}
.view-container.active {
    display: block;
}

/* Menú desplegable en cabecera */
.header-menu {
    display: flex;
    align-items: center;
    gap: 8px;
}
.header-menu .dropdown {
    position: relative;
    display: inline-block;
}
.header-menu .dropdown-toggle {
    background: transparent;
    border: 1px solid rgba(255,255,255,0.3);
    color: white;
    padding: 6px 14px;
    border-radius: 6px;
    font-size: 0.9rem;
    cursor: pointer;
    display: flex;
    align-items: center;
    gap: 6px;
    transition: background 0.2s;
}
.header-menu .dropdown-toggle:hover {
    background: rgba(255,255,255,0.1);
}
.header-menu .dropdown-menu {
    position: absolute;
    top: calc(100% + 4px);
    left: 0;
    background: white;
    border: 1px solid var(--border);
    border-radius: 8px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.15);
    min-width: 180px;
    z-index: 1000;
    padding: 6px 0;
}
.header-menu .dropdown-menu a {
    display: block;
    padding: 10px 16px;
    color: var(--text-main);
    text-decoration: none;
    transition: background 0.2s;
    cursor: pointer;
    font-size: 0.9rem;
}
.header-menu .dropdown-menu a:hover {
    background: var(--selection-gray);
}
.header-menu .dropdown-menu a.active {
    background: var(--selection-gray);
    font-weight: 600;
}

/* ===== SIDEBAR ===== */
.sidebar-overlay {
    position: fixed;
    top: 0;
    left: 0;
    width: 100%;
    height: 100%;
    background: rgba(0,0,0,0.3);
    z-index: 9998;
    display: none;
}
.sidebar-overlay.active {
    display: block;
}
.sidebar {
    position: fixed;
    top: 0;
    left: -280px;
    width: 280px;
    height: 100%;
    background: white;
    z-index: 9999;
    box-shadow: 2px 0 12px rgba(0,0,0,0.15);
    transition: left 0.3s ease;
    padding: 20px 0;
    overflow-y: auto;
}
.sidebar.open {
    left: 0;
}
.sidebar .sidebar-header {
    padding: 0 20px 16px 20px;
    border-bottom: 1px solid var(--border);
    font-weight: 600;
    font-size: 1.1rem;
    color: var(--text-main);
}
.sidebar .sidebar-item {
    display: block;
    padding: 14px 24px;
    color: var(--text-main);
    text-decoration: none;
    font-size: 0.95rem;
    border-left: 4px solid transparent;
    transition: background 0.2s, border-color 0.2s;
    cursor: pointer;
}
.sidebar .sidebar-item:hover {
    background: var(--selection-gray);
}
.sidebar .sidebar-item.active {
    background: var(--selection-gray);
    border-left-color: var(--sort-active);
    font-weight: 600;
}
.sidebar .close-sidebar {
    display: block;
    text-align: right;
    padding: 0 20px 12px 20px;
    font-size: 1.5rem;
    cursor: pointer;
    color: var(--text-muted);
}
.sidebar .close-sidebar:hover {
    color: var(--text-main);
}

/* Estilos consistentes para tablas de vistas (Usuarios e Historial) */
.view-table-wrapper {
    background: white;
    border-radius: 8px;
    border: 1px solid var(--border);
    overflow: auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.05);
}
.view-table-wrapper table {
    width: 100%;
    border-collapse: collapse;
    font-size: 0.9rem;
    background: white;
}
.view-table-wrapper th {
    background: var(--table-header-bg);
    color: #f8fafc;
    font-weight: 600;
    padding: 10px 14px;
    text-align: left;
    border-bottom: 2px solid #334155;
    white-space: nowrap;
    position: sticky;
    top: 0;
    z-index: 2;
}
.view-table-wrapper td {
    padding: 10px 14px;
    border-bottom: 1px solid var(--border);
    text-align: left !important;
    vertical-align: middle;
    background: white;
    color: var(--text-main);
}
.view-table-wrapper tr:hover td {
    background: var(--selection-gray);
}
.view-table-wrapper .btn {
    font-size: 0.8rem;
    padding: 4px 12px;
}

/* Mejorar visibilidad del placeholder y texto en inputs */
.tool-panel .search-wrapper input {
    color: var(--text-main); /* texto oscuro */
}
.tool-panel .search-wrapper input::placeholder {
    color: #6c757d; /* gris medio, más visible */
    opacity: 1;
}
/* Para los inputs de login y otros */
input::placeholder,
textarea::placeholder {
    color: #6c757d;
    opacity: 1;
}

/* Botón menú hamburguesa mejorado */
#menuToggleBtn {
    position: relative;
    width: 30px;
    height: 24px;
    background: transparent;
    border: none;
    cursor: pointer;
    padding: 0;
    display: flex;
    flex-direction: column;
    justify-content: space-between;
    align-items: center;
    transition: transform 0.2s ease;
}

#menuToggleBtn:hover {
    transform: scale(1.1);
}

#menuToggleBtn .menu-line {
    display: block;
    width: 28px;
    height: 3px;
    background: white;
    border-radius: 2px;
    transition: all 0.25s ease;
    transform-origin: center;
}

#menuToggleBtn .menu-line:nth-child(1) { transform-origin: top left; }
#menuToggleBtn .menu-line:nth-child(3) { transform-origin: bottom left; }

/* Estado activo (opcional, si quieres que se convierta en X al abrir) */
#menuToggleBtn.active .menu-line:nth-child(1) {
    transform: rotate(45deg) translate(3px, 3px);
}
#menuToggleBtn.active .menu-line:nth-child(2) {
    opacity: 0;
}
#menuToggleBtn.active .menu-line:nth-child(3) {
    transform: rotate(-45deg) translate(3px, -3px);
}



/* Botón de expandir/colapsar todo en el encabezado */
#headerRow th:first-child .expand-all-btn {
    display: inline-flex;
    align-items: center;
    justify-content: center;
    width: 32px;
    height: 32px;
    font-size: 1.4rem;
    line-height: 1;
    border-radius: 50%;
    background: #f1f3f5;
    color: #2c3e50;
    border: 1px solid #dee2e6;
    transition: all 0.2s ease;
    cursor: pointer;
    user-select: none;
    box-shadow: 0 1px 3px rgba(0,0,0,0.06);
}

#headerRow th:first-child .expand-all-btn:hover {
    background: #e9ecef;
    border-color: #adb5bd;
    transform: scale(1.1);
    box-shadow: 0 2px 6px rgba(0,0,0,0.12);
}

#headerRow th:first-child .expand-all-btn:active {
    transform: scale(0.92);
}

/* Opcional: indicador de estado (expandido/colapsado) */
#headerRow th:first-child .expand-all-btn.expanded {
    background: #e9ecef;
    border-color: #6c757d;
}

#headerRow th:first-child {
    width: 40px;
    min-width: 40px;
    max-width: 40px;
}
"""

### HTML

In [9]:
HTML_BODY = """
<!-- BARRA DE CABECERA CON BOTÓN DE MENÚ -->
<div class='header-bar'>
    <div style='display:flex; align-items:center; gap:16px;'>
        <button id="menuToggleBtn" aria-label="Abrir menú">
            <span class="menu-line"></span>
            <span class="menu-line"></span>
            <span class="menu-line"></span>
        </button>
        <h1>Gestor de Variables <span style='font-weight:300; opacity:0.6'>| SESNA</span></h1>
    </div>
    <div style='display:flex; align-items:center; gap:12px;'>
        <span id='currentUsername' style='font-size:0.9rem;'>Usuario</span>
        <div id='status-indicator' style='font-size:0.8em; color:#10B981'>● Sistema Activo</div>
    </div>
</div>

<!-- Sidebar -->
<div class="sidebar-overlay" id="sidebarOverlay"></div>
<div class="sidebar" id="sidebar">
    <div class="close-sidebar" id="closeSidebarBtn">&times;</div>
    <div class="sidebar-header">Navegación</div>
    <a class="sidebar-item active" data-view="variables" onclick="switchView('variables', this)">📊 Variables</a>
    <a class="sidebar-item" data-view="users" onclick="switchView('users', this)">👥 Usuarios</a>
    <a class="sidebar-item" data-view="history" onclick="switchView('history', this)">📋 Cargas</a>
</div>

<!-- ========== VISTA: VARIABLES ========== -->
<div id="view-variables" class="view-container active">
    <!-- Panel de herramientas (igual que antes) -->
    <div class='tool-panel'>
        <div class='search-wrapper'>
            <input type='text' id='searchInput' placeholder='Filtrar variables...' onkeyup='handleSearch()'>
        </div>
        <div class='button-group'>
            <button class='btn btn-tool' onclick='openModal("modalConfirmUpload")' title='Importar CSV'>📤 Importar CSV</button>
            <button class="btn btn-tool" onclick="openModal('modalFolderUpload')">📁 Importar Carpeta</button>
            <button class='btn btn-tool' onclick='downloadCSV()' title='Exportar CSV'>📥 Descargar CSV</button>
            <span style='width:2px; height:30px; background:var(--border); display:inline-block; margin:0 8px;'></span>
            <button class='btn btn-action' onclick='confirmAddRow()' title='Agregar nueva variable'>➕ Nueva Fila</button>
            <button class='btn btn-action' onclick='duplicateSelected()' title='Duplicar filas seleccionadas'>📋 Duplicar Fila</button>
            <button class='btn btn-danger' onclick='deleteSelectedRows()' title='Eliminar filas seleccionadas'>🗑 Eliminar Fila</button>
            <span style='width:2px; height:30px; background:var(--border); display:inline-block; margin:0 8px;'></span>
            <button class='btn btn-tool' onclick='retrainModels()' title='Reentrenar modelos ML'>🔄 Reentrenar Modelos</button>
            <button class='btn btn-outline' onclick='rollbackModels()' title='Restaurar modelos anteriores'>↩️ Rollback Modelos</button>
            <button class='btn btn-danger' onclick='deleteAllData()' title='Eliminar TODOS los datos (¡CUIDADO!)'>🗑️ Eliminar Todos</button>
        </div>
    </div>

    <!-- Tabla principal -->
    <div class='table-card' id='dropZone'>
        <div class='table-wrapper'>
            <table id='dataTable'>
                <tr id='headerRow'>
                    <th style='width:30px; min-width:30px; max-width:30px; text-align:center;'></th>
                    <th style='width:40px; min-width:40px; max-width:40px; text-align:center;'></th>
                </tr>
                <tbody></tbody>
            </table>
        </div>
        <div class='table-pagination'>
            <button onclick='prevPage()' id='prevPageBtnTable'>Anterior</button>
            <span>
                Página <input type='number' id='pageInput' value='1' min='1' onchange='goToPage(this.value)'>
                de <span id='totalPagesSpan'>1</span>
            </span>
            <select id='itemsPerPageSelect' onchange='changeItemsPerPage(this.value)'>
                <option value='10'>10</option>
                <option value='25'>25</option>
                <option value='50'>50</option>
                <option value='100'>100</option>
            </select>
            <span id="pageInfo">Mostrando 0 padres de 0</span>
            <button onclick='nextPage()' id='nextPageBtnTable'>Siguiente</button>
        </div>
    </div>
</div>

<!-- ========== VISTA: USUARIOS ========== -->
<div id="view-users" class="view-container">
    <div style="background: var(--panel-bg); border-radius: 8px; border: 1px solid var(--border); padding: 20px; margin-top: 24px;">
        <h2 style="margin-top:0; color: var(--text-main);">👥 Gestión de Usuarios</h2>
        <!-- Formulario para agregar usuario -->
        <div style='display: flex; gap: 10px; flex-wrap: wrap; margin-bottom: 20px;'>
            <input type='text' id='newUsername' placeholder='Usuario' style='flex: 1; min-width: 120px; padding: 8px 12px; border: 1px solid #ced4da; border-radius: 6px; background: white; color: var(--text-main);'>
            <input type='password' id='newPassword' placeholder='Contraseña (mín 6)' style='flex: 1; min-width: 120px; padding: 8px 12px; border: 1px solid #ced4da; border-radius: 6px; background: white; color: var(--text-main);'>
            <select id='newRol' style='padding: 8px 12px; border: 1px solid #ced4da; border-radius: 6px; background: white; color: var(--text-main);'>
                <option value='user'>Usuario</option>
                <option value='admin'>Administrador</option>
            </select>
            <button class='btn btn-success' onclick='createUser()'>➕ Agregar</button>
        </div>
        <!-- Tabla de usuarios -->
        <div class="view-table-wrapper" style="max-height: 450px; overflow-y: auto;">
            <table>
                <thead>
                    <tr>
                        <th style="width: 30%;">Usuario</th>
                        <th style="width: 20%;">Rol</th>
                        <th style="width: 50%; text-align: center;">Acciones</th>
                    </tr>
                </thead>
                <tbody id="usersTableBody">
                    <tr><td colspan="3" style="text-align: center; padding: 20px; color: var(--text-muted);">Cargando usuarios...</td></tr>
                </tbody>
            </table>
        </div>
    </div>
</div>

<!-- ========== VISTA: HISTORIAL DE CARGAS ========== -->
<div id="view-history" class="view-container">
    <div style="background: var(--panel-bg); border-radius: 8px; border: 1px solid var(--border); padding: 20px; margin-top: 24px;">
        <h2 style="margin-top:0; color: var(--text-main);">📋 Historial de Cargas</h2>
        <div class="view-table-wrapper" style="max-height: 500px; overflow-y: auto;">
            <table>
                <thead>
                    <tr>
                        <th style="width: 25%;">Fecha</th>
                        <th style="width: 35%;">Archivo</th>
                        <th style="width: 15%;">Filas</th>
                        <th style="width: 25%;">Estado</th>
                    </tr>
                </thead>
                <tbody id="historyListBody">
                    <tr><td colspan="4" style="text-align: center; padding: 20px; color: var(--text-muted);">Cargando historial...</td></tr>
                </tbody>
            </table>
        </div>
    </div>
</div>

<!-- MODALES (se mantienen igual, solo asegurar que no se dupliquen) -->
<div class='modal-overlay' id='modalOtraOpcion'>
    <div class='modal-content' style='max-width:450px;'>
        <div class='modal-header'>
            <h2 id='modalOtraTitle'>Nueva Opción</h2>
            <button class='close-button' onclick='handleCloseOtraOpcionModal()'>&times;</button>
        </div>
        <div class='form-group'>
            <label id='modalOtraLabel'>Ingrese el nuevo valor</label>
            <textarea id='otraOpcionInput' class='form-input' oninput='autoResizeModal(this)'></textarea>
        </div>
        <div style='display:flex; gap:12px; margin-top:24px;'>
            <button class='btn btn-outline' style='flex:1;' onclick='handleCloseOtraOpcionModal()'>Cerrar</button>
            <button class='btn btn-action' style='flex:1' id='btnConfirmOtra'>Agregar</button>
        </div>
    </div>
</div>

<div class='modal-overlay' id='modalSuggestion'>
    <div class='modal-content' style='max-width:450px;'>
        <div class='modal-header'>
            <h2>Sugerencia</h2>
            <button class='close-button' onclick='closeModal("modalSuggestion")'>&times;</button>
        </div>
        <p id='suggestionMessage'></p>
        <div style='display:flex; gap:12px; margin-top:24px;'>
            <button class='btn btn-outline' style='flex:1;' id='btnDeclineSuggestion'>Usar mi valor</button>
            <button class='btn btn-action' style='flex:1' id='btnAcceptSuggestion'>Usar sugerencia</button>
        </div>
    </div>
</div>

<div class='modal-overlay' id='modalConfirmUpload'>
    <div class='modal-content'>
        <div class='modal-header'>
            <h2>Cargar CSV</h2>
            <button class='close-button' onclick='closeModal("modalConfirmUpload")'>&times;</button>
        </div>
        <div>
            <p><strong>Instrucciones:</strong> Selecciona un archivo CSV con el siguiente formato:</p>
            <ul>
                <li>Debe contener las columnas: <code>ID, Proceso, Eje, Tema, Nombre, Institución, Cobertura, Periodicidad, Liga Web, Fuente, Año, Estado, Valor</code>.</li>
                <li>La columna <code>Año</code> debe tener el año correspondiente (ej. 2018).</li>
                <li>El sistema detectará automáticamente duplicados y te permitirá sobrescribirlos.</li>
            </ul>
            <div class='file-input-wrapper'>
                <label class='custom-file-upload'>
                    <input type='file' id='csvFileInputModal' accept='.csv' onchange='handleFileUploadModal(this)'>
                    Seleccionar archivo CSV
                </label>
            </div>
            <div id='uploadProgress' style='display:none; margin-top:10px;'>
                <div class='progress-bar'>
                    <div id='uploadProgressFill' class='progress-bar-fill' style='width:0%;'></div>
                </div>
                <span id='uploadProgressText'>Procesando...</span>
            </div>
            <div id='csvPreviewControls' style='display:none; margin-top:10px;'>
                <div style='margin-bottom:10px;'>
                    <input type='checkbox' id='selectAllDuplicatesForOverwrite' onchange='updateSelectAllCheckbox("uploadDataPreview", false, true)'>
                    <label for='selectAllDuplicatesForOverwrite'>Marcar todos los duplicados para sobrescribir (no marcados = ignorar)</label>
                </div>
                <div style='margin-bottom:10px;'>
                    <input type='checkbox' id='selectAllNonDuplicates' onchange='toggleAllNonDuplicates(this, false)'>
                    <label for='selectAllNonDuplicates'>Seleccionar todos los no duplicados</label>
                </div>
                <div id='csvNewCatalogMessage' style='display:none; background:#fff3cd; padding:8px; border:1px solid #ffc107; border-radius:4px; margin-bottom:10px;'>⚠️ Se detectaron valores nuevos en los catálogos. Se insertarán al confirmar la carga.</div>
            </div>
            <div id='uploadPreviewMessages' style='display:none; max-height:120px; overflow-y:auto; margin-top:10px;'></div>
            <div id='uploadDataPreview' style='display:none; margin-top:15px;'></div>
            <div id='paginationControls' style='display:none; justify-content:center; gap:10px; margin-top:10px;'>
                <button class='btn btn-outline' id='prevPageBtn' disabled>Anterior</button>
                <span>
                    Página <input type='number' id='previewPageInput' value='1' min='1' style="width:56px; padding:4px; text-align:center; border:1px solid #ced4da; border-radius:4px;">
                    de <span id='previewTotalPages'>1</span>
                </span>
                <button class='btn btn-outline' id='nextPageBtn' disabled>Siguiente</button>
            </div>
            <div style='display:flex; gap:12px; margin-top:24px;'>
                <button class='btn btn-danger' style='flex:1;' onclick='closeModal("modalConfirmUpload")'>Cancelar</button>
                <button class='btn btn-success' style='flex:1' id='confirmUploadBtn' onclick='confirmUpload()' disabled>✓ Confirmar Carga</button>
            </div>
        </div>
    </div>
</div>

<div class='modal-overlay' id='modalFolderUpload'>
    <div class='modal-content'>
        <div class='modal-header'>
            <h2>Cargar Carpeta (ZIP)</h2>
            <button class='close-button' onclick='closeModal("modalFolderUpload")'>&times;</button>
        </div>
        <div>
            <p><strong>Instrucciones:</strong> Selecciona un archivo ZIP que contenga la siguiente estructura:</p>
            <ul>
                <li><code>conjunto_de_datos/</code> (con archivos CSV de datos)</li>
                <li><code>diccionario_de_datos/</code> (con archivos CSV de diccionario, formato antiguo o nuevo)</li>
                <li><code>metadatos/</code> (opcional, con archivo <code>metadatos.txt</code>)</li>
            </ul>
            <p>El sistema detectará automáticamente el formato del diccionario.</p>
            <div class='file-input-wrapper'>
                <label class='custom-file-upload'>
                    <input type='file' id='zipFileInput' accept='.zip' onchange='handleZipUpload(this)'>
                    Seleccionar archivo ZIP
                </label>
            </div>
            <div id='zipProgress' style='display:none; margin-top:10px;'>
                <div class='progress-bar'>
                    <div id='zipProgressFill' class='progress-bar-fill' style='width:0%;'></div>
                </div>
                <span id='zipProgressText'>Procesando...</span>
            </div>
            <div id='folderLogMessages' style='max-height:250px; overflow-y:auto; background:#f8f9fa; padding:10px; border-radius:6px; border:1px solid var(--border); margin-top:10px; display:none;'>
                <pre id='folderLogContent' style='margin:0; font-size:0.85em; white-space:pre-wrap; word-break:break-word;'></pre>
            </div>
            <div id='zipPreviewContainer' style='display:none; margin-top:20px;'>
                <h3>Previsualización de datos extraídos</h3>
                <p style='font-size:0.85em; color:var(--danger); font-weight:500;'>Filas en rojo = duplicados. Las no duplicadas se insertarán.</p>
                <div style='margin-bottom:10px;'>
                    <input type='checkbox' id='selectAllDuplicatesForOverwriteZip' onchange='updateSelectAllCheckbox("zipPreviewTable", true, true)'>
                    <label for='selectAllDuplicatesForOverwriteZip'>Marcar todos los duplicados para sobrescribir (no marcados = ignorar)</label>
                </div>
                <div style='margin-bottom:10px;'>
                    <input type='checkbox' id='selectAllNonDuplicatesForOverwriteZip' onchange='toggleAllNonDuplicates(this, true)'>
                    <label for='selectAllNonDuplicatesForOverwriteZip'>Seleccionar todos los no duplicados</label>
                </div>
                <div id='zipNewCatalogMessage' style='display:none; background:#fff3cd; padding:8px; border:1px solid #ffc107; border-radius:4px; margin-bottom:10px;'>⚠️ Se detectaron valores nuevos en los catálogos. Se insertarán al confirmar la carga.</div>
                <p>Filas a agregar: <span id='rowCount'></span></p>
                <div id='zipPreviewTable'></div>
                <div id='zipPaginationControls' style='display:none; justify-content:center; gap:10px; margin-top:10px;'>
                    <button class='btn btn-outline' id='zipPrevPageBtn' disabled>Anterior</button>
                    <span>
                        Página <input type='number' id='zipPageInput' value='1' min='1' style="width:56px; padding:4px; text-align:center; border:1px solid #ced4da; border-radius:4px;">
                        de <span id='zipTotalPages'>1</span>
                    </span>
                    <button class='btn btn-outline' id='zipNextPageBtn' disabled>Siguiente</button>
                </div>
            </div>
            <div style='display:flex; gap:12px; margin-top:24px;'>
                <button class='btn btn-danger' style='flex:1;' onclick='closeModal("modalFolderUpload")'>Cancelar</button>
                <button class='btn btn-success' style='flex:1;' id='confirmFolderBtn' onclick='confirmFolderUpload()' disabled>✓ Confirmar carga</button>
            </div>
        </div>
    </div>
</div>

<div id='toast-container'></div>
<div id="loadingSpinner" style="display:none; position:fixed; top:0; left:0; width:100%; height:100%; background:rgba(255,255,255,0.7); z-index:9999; justify-content:center; align-items:center;">
    <div style="background:white; padding:20px 40px; border-radius:8px; box-shadow:0 4px 12px rgba(0,0,0,0.15); display:flex; align-items:center; gap:12px;">
        <div class="spinner-border" role="status" style="width:2rem; height:2rem; border:4px solid #E72000; border-top-color:transparent; border-radius:50%; animation: spin 0.8s linear infinite;"></div>
        <span>Cargando datos...</span>
    </div>
</div>
<style>
    @keyframes spin { to { transform: rotate(360deg); } }
    .spinner-border {
        display: inline-block;
        width: 2rem;
        height: 2rem;
        vertical-align: text-bottom;
        border: 4px solid currentColor;
        border-right-color: transparent;
        border-radius: 50%;
        animation: spinner-border 0.75s linear infinite;
    }
    @keyframes spinner-border { to { transform: rotate(360deg); } }
</style>
"""

### JS

In [10]:
JS = """
console.log('Gestor de Variables iniciado.');

// ============================================================
// VARIABLES GLOBALES (asegurar que existen)
// ============================================================

const previewItemsPerPage = 10;
let allFolderData = [];           // Todos los datos (padres + hijos) sin paginar
let totalParents = 0;             // Número total de padres
let parentsPerPage = 10;          // Tamaño de página (se puede ajustar)
let currentParentPage = 1;        // Página actual de padres
let sortField = 'id';
let sortDir = 'asc';
let totalRecords = 0;
let totalPages = 1;
let isLoading = false;
let previewToken = null;
let currentUploadToken = null;
let originalData = [], filteredData = [];
let currentSort = [];
let currentFileToUpload = null;
let allPreviewData = [];
let previewCurrentPage = 1;
let currentPage = 1;
let itemsPerPage = 10;
let _currentOtraOpcionCancelCallback = null;
let folderPreviewData = [];
let folderGlobalMetadata = null;
let currentZipFilename = null;
let isProcessing = false;
let overwriteSet = new Set();
let currentSearchTerm = '';

const COLUMN_WIDTHS = {
    id: '80px',
    proceso: '160px',
    eje: '160px',
    tema: '180px',
    nombre: '280px',
    institucion: '180px',
    cobertura: '160px',
    periodicidad: '160px',
    liga_web: '190px',
    fuente: '160px',
    año: '110px',
    valor: '130px',
    estado: '160px'
};

function showSpinner(show) {
    const spinner = document.getElementById('loadingSpinner');
    if (spinner) {
        spinner.style.display = show ? 'flex' : 'none';
    }
}

function formatNumero(val, columna) {
    if (val === null || val === undefined || val === '') return '';
    const upperVal = String(val).toUpperCase();
    if (upperVal === 'NSS' || upperVal === 'NA') return val;
    const num = Number(val);
    if (!isNaN(num) && Number.isFinite(num)) {
        if (columna === 'año') return String(Math.round(num));
        else return num.toFixed(1);
    }
    return val;
}

function escapeHtml(unsafe) {
    if (unsafe == null) return '';
    return String(unsafe)
        .replace(/&/g, '&amp;')
        .replace(/</g, '&lt;')
        .replace(/>/g, '&gt;')
        .replace(/\\"/g, '&quot;')
        .replace(/'/g, '&#039;')
        .replace(/`/g, '&#96;');
}

// ============================================================
// FUNCIONES DE MODALES Y UTILIDADES
// ============================================================

function openModal(id) {
    document.getElementById(id).style.display = 'flex';
    if (id === 'modalConfirmUpload') {
        document.getElementById('confirmUploadBtn').disabled = true;
    }
    if (id === 'modalFolderUpload') {
        document.getElementById('confirmFolderBtn').disabled = true;
        resetFolderModal();
    }
    if (id === 'modalUserManagement') {
        loadUsers();
    }
}

function closeModal(id) {
    document.getElementById(id).style.display = 'none';
    if (id === 'modalFolderUpload') {
        resetFolderModal();
    }
    if (id === 'modalConfirmUpload') {
        resetCsvModal();
    }
    loadData();
    document.getElementById('confirmUploadBtn').disabled = true;
    document.getElementById('confirmFolderBtn').disabled = true;
}

function showToast(msg) {
    const c = document.getElementById('toast-container');
    const t = document.createElement('div');
    t.className = 'toast';
    t.textContent = msg;
    c.appendChild(t);
    setTimeout(() => { t.style.opacity = '0'; setTimeout(() => t.remove(), 500); }, 3000);
}

function handleCloseOtraOpcionModal(callback = null) {
    const cancelCb = _currentOtraOpcionCancelCallback;
    _currentOtraOpcionCancelCallback = null;
    closeModal('modalOtraOpcion');
    if (typeof callback === 'function') callback();
    else if (typeof cancelCb === 'function') cancelCb();
}

function resetFolderModal() {
    folderPreviewData = [];
    folderGlobalMetadata = null;
    currentZipFilename = null;
    previewToken = null;
    window._previewToken = null;
    currentUploadToken = null;
    overwriteSet = new Set();
    isProcessing = false;
    previewCurrentPage = 1;
    const previewContainer = document.getElementById('zipPreviewContainer');
    if (previewContainer) previewContainer.style.display = 'none';
    const logContainer = document.getElementById('folderLogMessages');
    if (logContainer) logContainer.style.display = 'none';
    const progressDiv = document.getElementById('zipProgress');
    if (progressDiv) progressDiv.style.display = 'none';
    const fileInput = document.getElementById('zipFileInput');
    if (fileInput) fileInput.value = '';
    document.getElementById('rowCount').textContent = '0';
    document.getElementById('zipPreviewTable').innerHTML = '';
    document.getElementById('folderLogContent').textContent = '';
    document.getElementById('confirmFolderBtn').disabled = true;
}

function resetCsvModal() {
    document.getElementById('csvFileInputModal').value = '';
    document.getElementById('uploadDataPreview').style.display = 'none';
    document.getElementById('paginationControls').style.display = 'none';
    document.getElementById('uploadPreviewMessages').innerHTML = '';
    document.getElementById('uploadPreviewMessages').style.display = 'none';
    document.getElementById('csvPreviewControls').style.display = 'none';
    const progressDiv = document.getElementById('uploadProgress');
    if (progressDiv) progressDiv.style.display = 'none';
    allPreviewData = [];
    currentFileToUpload = null;
    currentUploadToken = null;
    document.getElementById('confirmUploadBtn').disabled = true;
    previewCurrentPage = 1;
    isProcessing = false;
}

function autoResize(el) {
    if (!el || el.tagName !== 'TEXTAREA') return;
    const minH = parseFloat(getComputedStyle(el).minHeight) || 24;
    el.style.height = 'auto';
    const h = Math.max(el.scrollHeight, minH);
    el.style.height = h + 'px';
}

function syncRowHeights(tr) {
    if (!tr || tr.classList.contains('hidden')) return;
    const cells = tr.querySelectorAll('td');
    if (cells.length === 0) return;
    cells.forEach(td => {
        td.style.height = 'auto';
        td.style.minHeight = 'auto';
    });
    tr.style.height = 'auto';
    const firstTd = cells[0];
    const style = getComputedStyle(firstTd);
    const tdPadding = parseFloat(style.paddingTop) + parseFloat(style.paddingBottom);
    let maxHeight = 0;
    cells.forEach(td => {
        const control = td.querySelector('textarea.edit-control') || td.querySelector('div.edit-control');
        if (control) {
            control.style.height = 'auto';
            void control.offsetHeight;
            const contentHeight = control.scrollHeight;
            const totalHeight = contentHeight + tdPadding;
            if (totalHeight > maxHeight) maxHeight = totalHeight;
        } else {
            const h = td.scrollHeight;
            if (h > maxHeight) maxHeight = h;
        }
    });
    if (maxHeight < 28) maxHeight = 28;
    const finalHeight = maxHeight + 2;
    cells.forEach(td => {
        td.style.height = finalHeight + 'px';
        td.style.minHeight = finalHeight + 'px';
    });
    tr.style.height = finalHeight + 'px';
    tr.querySelectorAll('textarea.edit-control, div.edit-control').forEach(ctrl => {
        ctrl.style.height = '100%';
        ctrl.style.minHeight = '100%';
    });
}

function autoResizeModal(el) {
    if (!el || el.tagName !== 'TEXTAREA') return;
    const minH = parseFloat(getComputedStyle(el).minHeight) || 32;
    const curH = el.offsetHeight;
    el.style.height = 'auto';
    const newH = Math.max(el.scrollHeight, minH);
    const lineH = parseFloat(getComputedStyle(el).lineHeight) || 20;
    const linesCur = Math.round(curH / lineH);
    const linesNew = Math.round(newH / lineH);
    if (Math.abs(newH - curH) > 5 || linesNew !== linesCur) {
        el.style.height = newH + 'px';
    } else {
        el.style.height = curH + 'px';
    }
}

function updateTemaOptions(tr, ejeValue, currentTema) {
    const idx = window.COLUMN_DEFINITIONS.findIndex(c => c.keyName === 'tema');
    if (idx === -1) return;
    const td = tr.cells[idx + 1];
    if (!td) return;
    const container = td.querySelector('div');
    if (!container) return;
    const sel = container.querySelector('select');
    if (!sel) return;
    if (!ejeValue) {
        sel.innerHTML = '<option value="">-seleccionar-</option>';
        return;
    }
    const ejeNum = ejeValue.charAt(0);
    const temaDef = window.COLUMN_DEFINITIONS.find(c => c.keyName === 'tema');
    let html = '<option value="">-seleccionar-</option>';
    (temaDef ? temaDef.options : []).forEach(opt => {
        if (opt.startsWith(ejeNum)) html += `<option value="${opt}" ${opt===currentTema?'selected':''}>${opt}</option>`;
    });
    html += '<option value="_OTRA_">Otra...</option>';
    sel.innerHTML = html;
}

async function openOpcionesModal(colDef, rowInfo, rowIndex, isZip, onDone, onCancel) {
    _currentOtraOpcionCancelCallback = onCancel || function() {};
    const modal = document.getElementById('modalOtraOpcion');
    const input = document.getElementById('otraOpcionInput');
    const btn = document.getElementById('btnConfirmOtra');
    document.getElementById('modalOtraTitle').textContent = 'Agregar ' + colDef.displayName;
    const tempVal = rowInfo._temp_new_values && rowInfo._temp_new_values[colDef.keyName] ? rowInfo._temp_new_values[colDef.keyName] : '';
    input.value = tempVal;
    modal.style.display = 'flex';
    input.focus();
    requestAnimationFrame(() => setTimeout(() => autoResizeModal(input), 50));
    const newBtn = btn.cloneNode(true);
    btn.parentNode.replaceChild(newBtn, btn);
    newBtn.onclick = async () => {
        const newVal = input.value.trim();
        if (!newVal) {
            handleCloseOtraOpcionModal(onCancel);
            return;
        }
        function findBestMatch(input, options) {
            if (!input || !options || options.length === 0) return null;
            const inputNorm = input.toLowerCase().trim();
            let best = null;
            let bestScore = 0.7;
            for (let opt of options) {
                const optNorm = opt.toLowerCase().trim();
                if (inputNorm === optNorm) return opt;
                if (optNorm.includes(inputNorm) || inputNorm.includes(optNorm)) {
                    const score = Math.min(optNorm.length, inputNorm.length) / Math.max(optNorm.length, inputNorm.length);
                    if (score > bestScore) {
                        bestScore = score;
                        best = opt;
                    }
                }
            }
            return best;
        }
        const suggested = findBestMatch(newVal, colDef.options || []);
        if (suggested) {
            openSuggestionModal(newVal, suggested,
                async (finalVal) => {
                    if (!colDef.options.includes(finalVal)) {
                        colDef.options.push(finalVal);
                        colDef.options.sort();
                    }
                    rowInfo._temp_new_values = rowInfo._temp_new_values || {};
                    rowInfo._temp_new_values[colDef.keyName] = finalVal;
                    rowInfo._nuevo_en_catalogo[colDef.keyName] = true;
                    if (rowInfo._is_duplicate) {
                        rowInfo._is_duplicate = false;
                        rowInfo._action = 'insert';
                        rowInfo._supabase_matching_id = null;
                    }
                    closeModal('modalSuggestion');
                    handleCloseOtraOpcionModal();
                    if (typeof onDone === 'function') onDone(finalVal);
                    const targetData = isZip ? folderPreviewData : allPreviewData;
                    renderPreviewTable(isZip ? 'zipPreviewTable' : 'uploadDataPreview', targetData, isZip);
                },
                () => {
                    if (!colDef.options.includes(newVal)) {
                        colDef.options.push(newVal);
                        colDef.options.sort();
                    }
                    rowInfo._temp_new_values = rowInfo._temp_new_values || {};
                    rowInfo._temp_new_values[colDef.keyName] = newVal;
                    rowInfo._nuevo_en_catalogo[colDef.keyName] = true;
                    if (rowInfo._is_duplicate) {
                        rowInfo._is_duplicate = false;
                        rowInfo._action = 'insert';
                        rowInfo._supabase_matching_id = null;
                    }
                    closeModal('modalSuggestion');
                    handleCloseOtraOpcionModal();
                    if (typeof onDone === 'function') onDone(newVal);
                    const targetData = isZip ? folderPreviewData : allPreviewData;
                    renderPreviewTable(isZip ? 'zipPreviewTable' : 'uploadDataPreview', targetData, isZip);
                }
            );
            closeModal('modalOtraOpcion');
        } else {
            if (!colDef.options.includes(newVal)) {
                colDef.options.push(newVal);
                colDef.options.sort();
            }
            rowInfo._temp_new_values = rowInfo._temp_new_values || {};
            rowInfo._temp_new_values[colDef.keyName] = newVal;
            rowInfo._nuevo_en_catalogo[colDef.keyName] = true;
            if (rowInfo._is_duplicate) {
                rowInfo._is_duplicate = false;
                rowInfo._action = 'insert';
                rowInfo._supabase_matching_id = null;
            }
            handleCloseOtraOpcionModal();
            if (typeof onDone === 'function') onDone(newVal);
            const targetData = isZip ? folderPreviewData : allPreviewData;
            renderPreviewTable(isZip ? 'zipPreviewTable' : 'uploadDataPreview', targetData, isZip);
        }
    };
    _currentOtraOpcionCancelCallback = () => {
        if (typeof onCancel === 'function') onCancel();
        const confirmBtn = document.getElementById('confirmUploadBtn');
        if (confirmBtn) confirmBtn.disabled = false;
        _currentOtraOpcionCancelCallback = null;
    };
}

function openSuggestionModal(originalInput, suggestedValue, onAccept, onDecline) {
    document.getElementById('suggestionMessage').innerHTML =
        `Tu valor '<strong>${escapeHtml(originalInput)}</strong>' es similar a '<strong>${escapeHtml(suggestedValue)}</strong>'. ¿Deseas usar la sugerencia?`;
    openModal('modalSuggestion');
    document.getElementById('btnAcceptSuggestion').onclick = () => onAccept(suggestedValue);
    document.getElementById('btnDeclineSuggestion').onclick = () => onDecline(originalInput);
}

// ============================================================
// CARGA DE DATOS Y TABLA PRINCIPAL
// ============================================================

async function loadData(page = currentPage, pageSize = itemsPerPage, field = sortField, dir = sortDir, searchTerm = null) {
    if (isLoading) return;
    isLoading = true;
    showSpinner(true);
    const term = (searchTerm !== null) ? searchTerm : currentSearchTerm;
    try {
        const params = new URLSearchParams({
            page: page,
            page_size: pageSize,
            sort_field: field,
            sort_dir: dir
        });
        if (term) {
            params.append('search', term);
        }
        const r = await fetch(`/api/variables?${params.toString()}`, { cache: 'no-cache' });
        if (!r.ok) {
            const errorText = await r.text();
            throw new Error(`Error ${r.status}: ${errorText}`);
        }
        const result = await r.json();
        const parents = result.data || [];
        totalRecords = result.total || 0;
        totalPages = result.total_pages || 1;
        currentPage = result.page || 1;
        itemsPerPage = result.page_size || pageSize;
        currentSearchTerm = term;

        // --- OBTENER HIJOS PARA ESTOS PADRES ---
        let allChildren = [];
        if (parents.length > 0) {
            const parentIds = parents.map(p => p.id);
            try {
                const childResp = await fetch(`/api/variables/children?ids=${parentIds.join(',')}`);
                if (childResp.ok) {
                    allChildren = await childResp.json();
                } else {
                    console.warn('No se pudieron obtener hijos:', childResp.status);
                }
            } catch (e) {
                console.warn('Error al obtener hijos:', e);
            }
        }

        // --- COMBINAR PADRES + HIJOS CON FLAGS ---
        const normalized = [];
        parents.forEach(parent => {
            // Asegurar que parent tenga _is_parent
            parent._is_parent = true;
            parent._children = allChildren.filter(c => c.parent_id === parent.id).map(c => c.id);
            normalized.push(parent);
        });
        allChildren.forEach(child => {
            child._is_child = true;
            normalized.push(child);
        });

        originalData = normalized;
        filteredData = normalized;

        renderHeaders();
        renderTable();
        updateSortIndicators();
        setTimeout(() => { initColumnResize(); }, 100);
    } catch (e) {
        console.error('Error cargando datos:', e);
        showToast('Error al cargar datos: ' + e.message);
    } finally {
        isLoading = false;
        showSpinner(false);
    }
}

function updateSortIndicators() {
    document.querySelectorAll('#headerRow th').forEach(th => {
        th.classList.remove('sorted-asc', 'sorted-desc');
        const key = th.dataset.key;
        if (key) {
            const sort = currentSort.find(s => s.key === key);
            if (sort) th.classList.add(sort.dir === 'asc' ? 'sorted-asc' : 'sorted-desc');
        }
        const orderSpan = th.querySelector('.sort-order');
        if (orderSpan) orderSpan.textContent = '';
    });
    currentSort.forEach((s, idx) => {
        const th = document.querySelector(`#headerRow th[data-key="${s.key}"]`);
        if (th) {
            let orderSpan = th.querySelector('.sort-order');
            if (!orderSpan) {
                orderSpan = document.createElement('span');
                orderSpan.className = 'sort-order';
                const div = th.querySelector('.th-content');
                if (div) div.appendChild(orderSpan);
            }
            orderSpan.textContent = ` ${idx + 1}`;
        }
    });
}

function handleSearch() {
    const q = (document.getElementById('searchInput')?.value || '').trim();
    currentSearchTerm = q;
    currentPage = 1;
    loadData(currentPage, itemsPerPage, sortField, sortDir, q);
}

function setSort(key, dir, event) {
    const shift = event && event.shiftKey;
    if (!shift) {
        if (sortField === key && sortDir === dir) {
            sortField = 'id';
            sortDir = 'asc';
        } else {
            sortField = key;
            sortDir = dir;
        }
    } else {
        sortField = key;
        sortDir = dir;
    }
    currentPage = 1;
    loadData(currentPage, itemsPerPage, sortField, sortDir, currentSearchTerm);
}

function toggleAllCheckboxes(master) {
    const checkboxes = document.querySelectorAll('.row-selector');
    checkboxes.forEach(cb => {
        cb.checked = master.checked;
    });
}

function toggleAllParents() {
    const allParentRows = document.querySelectorAll('#dataTable tbody tr:not(.child-row)');
    let anyExpanded = false;
    const firstParent = allParentRows[0];
    if (firstParent) {
        const firstBtn = firstParent.querySelector('.expand-btn');
        if (firstBtn && firstBtn.textContent === '▼') anyExpanded = true;
    }
    const newState = anyExpanded ? 'collapse' : 'expand';
    allParentRows.forEach(parentRow => {
        const btn = parentRow.querySelector('.expand-btn');
        if (btn) {
            const parentId = btn.dataset.parentId;
            const children = document.querySelectorAll(`tr[data-parent="${parentId}"]`);
            if (newState === 'expand') {
                children.forEach(child => child.classList.remove('hidden'));
                btn.textContent = '▼';
            } else {
                children.forEach(child => child.classList.add('hidden'));
                btn.textContent = '▶';
            }
            children.forEach(child => {
                if (!child.classList.contains('hidden')) {
                    requestAnimationFrame(() => syncRowHeights(child));
                }
            });
            requestAnimationFrame(() => syncRowHeights(parentRow));
        }
    });
}

function renderHeaders() {
    let hr = document.getElementById('headerRow');
    if (!hr) {
        const table = document.getElementById('dataTable');
        if (!table) {
            console.error('No se encontró la tabla #dataTable');
            return;
        }
        let thead = table.querySelector('thead');
        if (!thead) {
            thead = document.createElement('thead');
            table.prepend(thead);
        }
        hr = document.createElement('tr');
        hr.id = 'headerRow';
        thead.appendChild(hr);
    }
    hr.innerHTML = '';
    const thExpand = document.createElement('th');
    thExpand.style.width = '30px';
    thExpand.style.minWidth = '30px';
    thExpand.style.maxWidth = '30px';
    thExpand.style.textAlign = 'center';
    thExpand.style.verticalAlign = 'middle';
    thExpand.style.cursor = 'pointer';
    thExpand.innerHTML = `<span class="expand-all-btn" title="Expandir/Colapsar todos los padres" onclick="toggleAllParents()">⇕</span>`;
    hr.appendChild(thExpand);
    const thCheck = document.createElement('th');
    thCheck.style.width = '40px';
    thCheck.style.minWidth = '40px';
    thCheck.style.maxWidth = '40px';
    thCheck.style.textAlign = 'center';
    thCheck.style.verticalAlign = 'middle';
    const cbMaster = document.createElement('input');
    cbMaster.type = 'checkbox';
    cbMaster.id = 'selectAllRows';
    cbMaster.addEventListener('change', function(e) {
        toggleAllCheckboxes(this);
    });
    thCheck.appendChild(cbMaster);
    hr.appendChild(thCheck);
    window.COLUMN_DEFINITIONS.forEach((col) => {
        const th = document.createElement('th');
        th.dataset.key = col.keyName;
        const sort = currentSort.find(s => s.key === col.keyName);
        if (sort) th.classList.add(sort.dir === 'asc' ? 'sorted-asc' : 'sorted-desc');
        th.innerHTML = `<div class="th-content">
            <span class="th-label">${escapeHtml(col.displayName)}</span>
            <div class="sort-indicators">
                <span class="tri-up" onclick="setSort('${col.keyName}','asc', event)">▲</span>
                <span class="tri-down" onclick="setSort('${col.keyName}','desc', event)">▼</span>
            </div>
        </div>`;
        const width = COLUMN_WIDTHS[col.keyName] || 'auto';
        th.style.width = width;
        th.style.minWidth = width;
        th.style.maxWidth = width;
        hr.appendChild(th);
    });
}

async function deleteSelectedRows() {
    const selected = document.querySelectorAll('.row-selector:checked');
    if (selected.length === 0) { showToast('⚠️ Selecciona al menos una fila.'); return; }
    const ids = Array.from(selected).map(cb => cb.dataset.id);
    if (!confirm(`¿Eliminar ${ids.length} fila(s)?`)) return;
    try {
        const resp = await fetch('/api/variables/bulk-delete', {
            method: 'DELETE',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({ ids })
        });
        const data = await resp.json();
        if (resp.ok) {
            originalData = originalData.filter(row => !ids.includes(row.id));
            handleSearch();
            showToast(`✅ ${data.deleted} fila(s) eliminada(s).`);
        } else {
            showToast('Error: ' + (data.error || 'Algo salió mal'));
        }
    } catch (e) {
        showToast('Error de conexión: ' + e.message);
    }
}

async function duplicateSelected() {
    const selected = document.querySelectorAll('.row-selector:checked');
    if (selected.length === 0) { showToast('⚠️ Selecciona al menos una fila para duplicar.'); return; }
    const ids = Array.from(selected).map(cb => cb.dataset.id);
    try {
        const resp = await fetch('/api/variables/bulk-duplicate', {
            method: 'POST',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({ ids })
        });
        const data = await resp.json();
        if (resp.ok) {
            const newRows = data.inserted || [];
            if (newRows.length) {
                originalData = originalData.concat(newRows);
                handleSearch();
                showToast(`✓ ${newRows.length} fila(s) duplicada(s).`);
            } else {
                showToast('Error: No se recibieron datos duplicados.');
            }
        } else {
            showToast('Error: ' + (data.error || 'Algo salió mal'));
        }
    } catch (e) {
        showToast('Error de conexión: ' + e.message);
    }
}


// Selecciona/deselecciona todos los hijos visibles de un padre
function toggleChildrenSelection(parentCheckbox) {
    const parentRow = parentCheckbox.closest('tr');
    if (!parentRow) return;
    const parentId = parentCheckbox.dataset.id;
    const checked = parentCheckbox.checked;
    // Buscar checkboxes de hijos (ya cargados en el DOM)
    const childCheckboxes = document.querySelectorAll(`.row-selector[data-parent="${parentId}"]`);
    childCheckboxes.forEach(cb => {
        cb.checked = checked;
    });
    // Si los hijos aún no están cargados (expandidos), no hacemos nada porque al cargarlos
    // se sincronizarán con el estado del padre mediante el evento change que ya tienen.
}

function renderTable() {
    const tbody = document.querySelector('#dataTable tbody');
    if (!tbody) return;
    tbody.innerHTML = '';
    document.getElementById('totalPagesSpan').textContent = totalPages;
    document.getElementById('pageInput').value = currentPage;
    document.getElementById('prevPageBtnTable').disabled = currentPage <= 1;
    document.getElementById('nextPageBtnTable').disabled = currentPage >= totalPages;

    // Filtrar padres (con _is_parent === true)
    const parents = filteredData.filter(row => row._is_parent === true);
    const childrenMap = {};
    filteredData.filter(row => row._is_child === true).forEach(child => {
        const parentId = child.parent_id;
        if (parentId) {
            if (!childrenMap[parentId]) childrenMap[parentId] = [];
            childrenMap[parentId].push(child);
        }
    });

    const fragment = document.createDocumentFragment();

    parents.forEach(parentData => {
        // --- Crear fila del padre ---
        const tr = document.createElement('tr');
        tr.dataset.id = parentData.id;
        tr.classList.add('parent-row');
        tr.dataset.loaded = 'true'; // ya tenemos los hijos

        // Celda de expansión
        const tdExpand = document.createElement('td');
        tdExpand.style.cssText = 'width:30px; min-width:30px; max-width:30px; text-align:center; vertical-align:middle; padding:2px 4px;';
        const btn = document.createElement('span');
        btn.className = 'expand-btn';
        btn.innerHTML = '▸';
        btn.dataset.parentId = parentData.id;
        btn.title = 'Expandir/Colapsar hijos';
        tdExpand.appendChild(btn);
        tr.appendChild(tdExpand);

        // Celda de checkbox
        const tdCheck = document.createElement('td');
        tdCheck.className = 'row-checkbox';
        tdCheck.style.cssText = 'width:40px; min-width:40px; max-width:40px; text-align:center; vertical-align:middle; padding:2px 4px;';
        const checkbox = document.createElement('input');
        checkbox.type = 'checkbox';
        checkbox.className = 'row-selector';
        checkbox.dataset.id = parentData.id;
        checkbox.dataset.parent = 'true';
        checkbox.addEventListener('change', function() {
            toggleChildrenSelection(this);
        });
        tdCheck.appendChild(checkbox);
        tr.appendChild(tdCheck);

        // Resto de celdas (datos)
        window.COLUMN_DEFINITIONS.forEach((col) => {
            const td = document.createElement('td');
            td.style.cssText = 'vertical-align:top; padding:2px 4px; min-height:28px; height:100%;';
            const key = col.keyName;
            const widthMap = {
                id: '90px',
                proceso: '160px',
                eje: '160px',
                tema: '180px',
                nombre: '280px',
                institucion: '180px',
                cobertura: '120px',
                periodicidad: '140px',
                liga_web: '180px',
                fuente: '160px',
                año: '90px',
                valor: '90px'
            };
            const w = widthMap[key] || 'auto';
            td.style.width = w;
            td.style.minWidth = w;
            td.style.maxWidth = w;

            let val = (parentData[key] !== undefined && parentData[key] !== null) ? String(parentData[key]) : '';
            if (key === 'valor' && val !== '') val = formatNumero(val);

            if (key === 'id') {
                const input = document.createElement('input');
                input.type = 'text';
                input.className = 'read-only-input';
                input.value = val;
                input.disabled = true;
                input.style.cssText = 'width:100%; text-align:left; height:100%; min-height:28px; padding:2px 4px; box-sizing:border-box; background:transparent; border:none; outline:none; font-family:inherit; font-size:0.8em; vertical-align:top;';
                td.appendChild(input);
            } else if (col.type === 'select') {
                const container = document.createElement('div');
                container.style.cssText = 'position:relative; width:100%; height:100%; min-height:28px; display:flex; align-items:flex-start; cursor:pointer;';
                const displayDiv = document.createElement('div');
                displayDiv.className = 'edit-control';
                displayDiv.textContent = val;
                displayDiv.style.cssText = 'width:100%; min-height:28px; height:100%; padding:2px 6px; line-height:1.4; box-sizing:border-box; border:1px solid transparent; border-radius:0; background:transparent; outline:none; font-family:inherit; font-size:0.8em; white-space:normal; word-wrap:break-word; cursor:pointer; display:flex; align-items:flex-start; flex:1; text-align:left; z-index:2; pointer-events:none; transition:border-color 0.2s, box-shadow 0.2s, background 0.2s;';
                displayDiv.title = val;
                const select = document.createElement('select');
                select.className = 'edit-control';
                select.style.cssText = 'position:absolute; top:0; left:0; width:100%; height:100%; min-height:28px; padding:2px 6px; line-height:1.4; box-sizing:border-box; border:none; background:transparent; outline:none; font-family:inherit; font-size:0.8em; vertical-align:top; overflow:visible; white-space:nowrap; text-overflow:clip; opacity:0; z-index:1; cursor:pointer;';
                let htmlOpts = '<option value="">-seleccionar-</option>';
                (col.options || []).forEach(o => {
                    htmlOpts += `<option value="${o}" ${o===val?'selected':''}>${o}</option>`;
                });
                htmlOpts += '<option value="_OTRA_">Otra...</option>';
                select.innerHTML = htmlOpts;
                select.dataset.colKey = key;
                select.dataset.rowId = parentData.id;
                container.appendChild(displayDiv);
                container.appendChild(select);
                td.appendChild(container);
            } else {
                const textarea = document.createElement('textarea');
                textarea.className = 'edit-control';
                textarea.value = val;
                textarea.style.cssText = 'width:100%; height:auto; min-height:28px; padding:2px 6px; line-height:1.4; box-sizing:border-box; border:1px solid transparent; background:transparent; outline:none; font-family:inherit; font-size:0.8em; resize:none; vertical-align:top; white-space:pre-wrap; word-break:break-word; transition:border-color 0.2s, box-shadow 0.2s;';
                const placeholderMap = {
                    nombre: 'Insertar nombre',
                    liga_web: 'Insertar liga web',
                    fuente: 'Insertar fuente',
                    año: 'Insertar año',
                    valor: 'Insertar valor'
                };
                if (placeholderMap[key] && (!val || val.trim() === '')) {
                    textarea.placeholder = placeholderMap[key];
                }
                textarea.dataset.colKey = key;
                textarea.dataset.rowId = parentData.id;
                td.appendChild(textarea);
            }
            tr.appendChild(td);
        });

        fragment.appendChild(tr);

        // --- Crear filas hijas (si existen) ---
        const parentId = parentData.id;
        const childList = childrenMap[parentId] || [];
        if (childList.length > 0) {
            tr.dataset.childrenIds = JSON.stringify(childList.map(c => c.id));
            childList.forEach(childData => {
                const childTr = createChildRow(childData, parentId);
                childTr.classList.add('hidden');
                fragment.appendChild(childTr);
            });
        }
    });

    tbody.appendChild(fragment);

    // Sincronizar alturas y eventos
    requestAnimationFrame(() => {
        const allRows = tbody.querySelectorAll('tr');
        allRows.forEach(tr => syncRowHeights(tr));
        setTimeout(() => {
            allRows.forEach(tr => syncRowHeights(tr));
        }, 50);
    });

    tbody.removeEventListener('click', handleTbodyClick);
    tbody.addEventListener('click', handleTbodyClick);
    tbody.removeEventListener('change', handleTbodyChange);
    tbody.addEventListener('change', handleTbodyChange);
}

function handleTbodyClick(e) {
    const target = e.target;
    if (target.classList.contains('expand-btn')) {
        e.stopPropagation();
        const parentId = target.dataset.parentId;
        if (parentId) {
            toggleChildren(parentId, target);
        }
    }
}

function handleTbodyChange(e) {
    const target = e.target;
    if (target.classList.contains('edit-control') || target.tagName === 'SELECT' || target.tagName === 'TEXTAREA') {
        const tr = target.closest('tr');
        if (tr) {
            if (target.tagName === 'SELECT' && target.value === '_OTRA_') {
                const colKey = target.dataset.colKey;
                const colDef = window.COLUMN_DEFINITIONS.find(c => c.keyName === colKey);
                const rowData = originalData.find(r => r.id === tr.dataset.id);
                if (colDef && rowData) {
                    const displayDiv = tr.querySelector(`div[data-colkey="${colKey}"]`);
                    addMainTableOption(colDef, rowData, displayDiv, tr, rowData[colKey], target);
                }
                return;
            }
            saveRow(tr);
        }
    }
}

async function saveRow(tr) {
    const id = tr.dataset.id;
    const data = {};
    const cells = tr.querySelectorAll('td');
    window.COLUMN_DEFINITIONS.forEach((col, idx) => {
        const td = cells[idx + 1];
        if (!td) return;
        let ctrl;
        if (col.type === 'select') ctrl = td.querySelector('select');
        else ctrl = td.querySelector('.edit-control');
        if (ctrl) {
            data[col.keyName] = ctrl.value;
        }
    });
    const resp = await fetch(`/api/variables/${id}`, {
        method: 'PUT',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify(data)
    });
    const result = await resp.json();
    if (resp.ok) {
        const idx = originalData.findIndex(r => r.id === id);
        if (idx !== -1) {
            originalData[idx] = { ...originalData[idx], ...data };
        }
        showToast('✓ Celda actualizada');
    } else {
        showToast('Error al actualizar celda.');
    }
}

async function confirmAddRow() {
    const r = await fetch('/api/variables', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({})
    });
    if (r.ok) {
        const newRow = await r.json();
        originalData.unshift(newRow);
        handleSearch();
        showToast('✓ Fila creada');
    } else {
        const error = await r.json();
        showToast('Error: ' + (error.error || 'No se pudo crear la fila'));
    }
}

function changeItemsPerPage(val) {
    itemsPerPage = parseInt(val);
    currentPage = 1;
    loadData(currentPage, itemsPerPage, sortField, sortDir, currentSearchTerm);
}

function goToPage(page) {
    let p = parseInt(page);
    if (isNaN(p) || p < 1) p = 1;
    if (p > totalPages) p = totalPages;
    if (p !== currentPage) {
        currentPage = p;
        loadData(currentPage, itemsPerPage, sortField, sortDir, currentSearchTerm);
    }
}

function nextPage() {
    if (currentPage < totalPages) {
        currentPage++;
        loadData(currentPage, itemsPerPage, sortField, sortDir, currentSearchTerm);
    }
}

function prevPage() {
    if (currentPage > 1) {
        currentPage--;
        loadData(currentPage, itemsPerPage, sortField, sortDir, currentSearchTerm);
    }
}

function downloadCSV() {
    if (filteredData.length === 0) { showToast('No hay datos filtrados para descargar.'); return; }
    const headers = window.COLUMN_DEFINITIONS.map(c => c.displayName);
    const rows = [];
    rows.push(headers.map(h => `"${h.replace(/"/g, '""')}"`).join(','));
    filteredData.forEach(row => {
        const vals = window.COLUMN_DEFINITIONS.map(c => {
            let v = row[c.keyName] != null ? String(row[c.keyName]) : '';
            if (c.keyName === 'valor' && v !== '') v = formatNumero(v);
            return `"${v.replace(/"/g, '""')}"`;
        });
        rows.push(vals.join(','));
    });
    const csvContent = rows.join('\\n');
    const blob = new Blob(['\uFEFF' + csvContent], { type: 'text/csv;charset=utf-8;' });
    const url = URL.createObjectURL(blob);
    const a = document.createElement('a');
    a.href = url;
    const now = new Date();
    const fechaHora = now.getFullYear() +
                      String(now.getMonth() + 1).padStart(2, '0') +
                      String(now.getDate()).padStart(2, '0') +
                      String(now.getHours()).padStart(2, '0') +
                      String(now.getMinutes()).padStart(2, '0') +
                      String(now.getSeconds()).padStart(2, '0');
    a.download = `variables_${fechaHora}.csv`;
    document.body.appendChild(a);
    a.click();
    document.body.removeChild(a);
    URL.revokeObjectURL(url);
    showToast('✓ CSV descargado');
}

let colResizeData = null;

function initColumnResize() {
    const table = document.getElementById('dataTable');
    if (!table) return;
    const cells = table.querySelectorAll('th, td:not(.row-checkbox)');
    cells.forEach(cell => {
        cell.removeEventListener('mousemove', onColumnMouseMove, true);
        cell.removeEventListener('mouseleave', onColumnMouseLeave, true);
        cell.removeEventListener('mousedown', onColumnMouseDown, true);
        cell.addEventListener('mousemove', onColumnMouseMove, true);
        cell.addEventListener('mouseleave', onColumnMouseLeave, true);
        cell.addEventListener('mousedown', onColumnMouseDown, true);
    });
}

function onColumnMouseMove(e) {
    const cell = e.currentTarget;
    const rect = cell.getBoundingClientRect();
    const isRightEdge = (e.clientX - rect.left) > (rect.width - 20);
    if (isRightEdge) {
        cell.style.cursor = 'col-resize';
        cell.dataset.resizeableCol = 'true';
    } else {
        cell.style.cursor = '';
        cell.dataset.resizeableCol = 'false';
    }
}

function onColumnMouseLeave(e) {
    const cell = e.currentTarget;
    cell.style.cursor = '';
    cell.dataset.resizeableCol = 'false';
}

function onColumnMouseDown(e) {
    const cell = e.currentTarget;
    if (cell.dataset.resizeableCol !== 'true') return;
    const colIndex = cell.cellIndex;
    const startX = e.clientX;
    const startWidth = cell.offsetWidth;
    colResizeData = { cell, colIndex, startX, startWidth, table: document.getElementById('dataTable') };
    document.addEventListener('mousemove', onColumnResizeMove);
    document.addEventListener('mouseup', onColumnResizeUp);
    e.preventDefault();
}

function onColumnResizeMove(e) {
    if (!colResizeData) return;
    const dx = e.clientX - colResizeData.startX;
    const newWidth = Math.max(50, colResizeData.startWidth + dx);
    const colIndex = colResizeData.colIndex;
    const rows = colResizeData.table.querySelectorAll('tr');
    rows.forEach(row => {
        const cell = row.cells[colIndex];
        if (cell) {
            cell.style.width = newWidth + 'px';
            cell.style.minWidth = newWidth + 'px';
            cell.style.maxWidth = newWidth + 'px';
        }
    });
    const th = colResizeData.table.querySelector(`thead th:nth-child(${colIndex + 1})`);
    if (th) {
        th.style.width = newWidth + 'px';
        th.style.minWidth = newWidth + 'px';
        th.style.maxWidth = newWidth + 'px';
    }
}

function onColumnResizeUp() {
    document.removeEventListener('mousemove', onColumnResizeMove);
    document.removeEventListener('mouseup', onColumnResizeUp);
    colResizeData = null;
}

function createChildRow(rowData, parentId) {
    const tr = document.createElement('tr');
    tr.dataset.id = rowData.id;
    tr.dataset.parent = parentId;
    tr.classList.add('child-row', 'hidden');
    const tdExpand = document.createElement('td');
    tdExpand.style.width = '30px';
    tdExpand.style.minWidth = '30px';
    tdExpand.style.maxWidth = '30px';
    tdExpand.style.textAlign = 'center';
    tdExpand.style.verticalAlign = 'middle';
    tdExpand.style.padding = '2px 4px';
    tdExpand.innerHTML = '&nbsp;';
    tdExpand.style.paddingLeft = '20px';
    tr.appendChild(tdExpand);
    const tdCheck = document.createElement('td');
    tdCheck.className = 'row-checkbox';
    tdCheck.style.width = '40px';
    tdCheck.style.minWidth = '40px';
    tdCheck.style.maxWidth = '40px';
    tdCheck.style.textAlign = 'center';
    tdCheck.style.verticalAlign = 'middle';
    tdCheck.style.padding = '2px 4px';
    const checkbox = document.createElement('input');
    checkbox.type = 'checkbox';
    checkbox.className = 'row-selector';
    checkbox.dataset.id = rowData.id;
    checkbox.dataset.parent = parentId;
    tdCheck.appendChild(checkbox);
    tr.appendChild(tdCheck);
    window.COLUMN_DEFINITIONS.forEach((col) => {
        const td = document.createElement('td');
        td.style.verticalAlign = 'top';
        td.style.padding = '2px 4px';
        td.style.minHeight = '28px';
        td.style.height = '100%';
        const key = col.keyName;
        let width = 'auto';
        if (key === 'id') width = '90px';
        else if (key === 'proceso') width = '160px';
        else if (key === 'eje') width = '160px';
        else if (key === 'tema') width = '180px';
        else if (key === 'nombre') width = '280px';
        else if (key === 'institucion') width = '180px';
        else if (key === 'cobertura') width = '120px';
        else if (key === 'periodicidad') width = '140px';
        else if (key === 'liga_web') width = '180px';
        else if (key === 'fuente') width = '160px';
        else if (key === 'año') width = '90px';
        else if (key === 'valor') width = '90px';
        td.style.width = width;
        td.style.minWidth = width;
        td.style.maxWidth = width;
        let val = '';
        if (rowData[col.keyName] !== undefined && rowData[col.keyName] !== null) {
            val = String(rowData[col.keyName]);
        }
        if (col.keyName === 'valor' && val !== '') {
            val = formatNumero(val);
        }
        if (col.keyName === 'id') {
            const input = document.createElement('input');
            input.type = 'text';
            input.className = 'read-only-input';
            input.value = val;
            input.disabled = true;
            input.style.width = '100%';
            input.style.textAlign = 'left';
            input.style.height = '100%';
            input.style.minHeight = '28px';
            input.style.padding = '2px 4px';
            input.style.boxSizing = 'border-box';
            input.style.background = 'transparent';
            input.style.border = 'none';
            input.style.outline = 'none';
            input.style.fontFamily = 'inherit';
            input.style.fontSize = '0.8em';
            input.style.verticalAlign = 'top';
            td.appendChild(input);
        } else if (col.type === 'select') {
            const container = document.createElement('div');
            container.style.position = 'relative';
            container.style.width = '100%';
            container.style.height = '100%';
            container.style.minHeight = '28px';
            container.style.display = 'flex';
            container.style.alignItems = 'flex-start';
            container.style.cursor = 'pointer';
            const displayDiv = document.createElement('div');
            displayDiv.className = 'edit-control';
            displayDiv.textContent = val;
            displayDiv.style.width = '100%';
            displayDiv.style.minHeight = '28px';
            displayDiv.style.height = '100%';
            displayDiv.style.padding = '2px 6px';
            displayDiv.style.lineHeight = '1.4';
            displayDiv.style.boxSizing = 'border-box';
            displayDiv.style.border = '1px solid transparent';
            displayDiv.style.borderRadius = '0';
            displayDiv.style.background = 'transparent';
            displayDiv.style.outline = 'none';
            displayDiv.style.fontFamily = 'inherit';
            displayDiv.style.fontSize = '0.8em';
            displayDiv.style.whiteSpace = 'normal';
            displayDiv.style.wordWrap = 'break-word';
            displayDiv.style.cursor = 'pointer';
            displayDiv.style.display = 'flex';
            displayDiv.style.alignItems = 'flex-start';
            displayDiv.style.flex = '1';
            displayDiv.title = val;
            displayDiv.style.textAlign = 'left';
            displayDiv.style.zIndex = '2';
            displayDiv.style.pointerEvents = 'none';
            displayDiv.style.transition = 'border-color 0.2s, box-shadow 0.2s, background 0.2s';
            const select = document.createElement('select');
            select.className = 'edit-control';
            select.style.position = 'absolute';
            select.style.top = '0';
            select.style.left = '0';
            select.style.width = '100%';
            select.style.height = '100%';
            select.style.minHeight = '28px';
            select.style.padding = '2px 6px';
            select.style.lineHeight = '1.4';
            select.style.boxSizing = 'border-box';
            select.style.border = 'none';
            select.style.background = 'transparent';
            select.style.outline = 'none';
            select.style.fontFamily = 'inherit';
            select.style.fontSize = '0.8em';
            select.style.verticalAlign = 'top';
            select.style.overflow = 'visible';
            select.style.whiteSpace = 'nowrap';
            select.style.textOverflow = 'clip';
            select.style.opacity = '0';
            select.style.zIndex = '1';
            select.style.cursor = 'pointer';
            select.addEventListener('focus', function() {
                displayDiv.style.border = '1px solid var(--primary)';
                displayDiv.style.background = 'white';
                displayDiv.style.boxShadow = '0 0 0 3px rgba(74,85,104,0.15)';
            });
            select.addEventListener('blur', function() {
                setTimeout(() => {
                    displayDiv.style.border = '1px solid transparent';
                    displayDiv.style.background = 'transparent';
                    displayDiv.style.boxShadow = 'none';
                }, 150);
            });
            let htmlOpts = `<option value=\"\">-seleccionar-</option>`;
            (col.options || []).forEach(o => htmlOpts += `<option value=\"${o}\" ${o===val?'selected':''}>${o}</option>`);
            htmlOpts += `<option value=\"_OTRA_\">Otra...</option>`;
            select.innerHTML = htmlOpts;
            container.addEventListener('pointerdown', function(e) {
                e.preventDefault();
                e.stopPropagation();
                select.focus();
                if (select.showPicker) select.showPicker();
                else select.click();
            });
            select.onchange = async function() {
                if (this.value === '_OTRA_') {
                    addMainTableOption(col, rowData, displayDiv, tr, val, this);
                    return;
                }
                const newVal = this.value;
                val = newVal;
                rowData[col.keyName] = newVal;
                displayDiv.textContent = newVal;
                displayDiv.title = newVal;
                displayDiv.style.background = 'transparent';
                if (col.keyName === 'eje') updateTemaOptions(tr, newVal, rowData['tema']);
                saveRow(tr);
                syncRowHeights(tr);
            };
            select.addEventListener('blur', function() {
                const newVal = this.value;
                if (newVal !== val && newVal !== '_OTRA_') {
                    val = newVal;
                    rowData[col.keyName] = newVal;
                    displayDiv.textContent = newVal;
                    displayDiv.title = newVal;
                    saveRow(tr);
                    syncRowHeights(tr);
                }
            });
            container.appendChild(displayDiv);
            container.appendChild(select);
            td.appendChild(container);
        } else {
            const textarea = document.createElement('textarea');
            textarea.className = 'edit-control';
            textarea.value = val;
            textarea.style.width = '100%';
            textarea.style.height = 'auto';
            textarea.style.minHeight = '28px';
            textarea.style.padding = '2px 6px';
            textarea.style.lineHeight = '1.4';
            textarea.style.boxSizing = 'border-box';
            textarea.style.border = '1px solid transparent';
            textarea.style.background = 'transparent';
            textarea.style.outline = 'none';
            textarea.style.fontFamily = 'inherit';
            textarea.style.fontSize = '0.8em';
            textarea.style.resize = 'none';
            textarea.style.verticalAlign = 'top';
            textarea.style.whiteSpace = 'pre-wrap';
            textarea.style.wordBreak = 'break-word';
            textarea.style.transition = 'border-color 0.2s, box-shadow 0.2s';
            const placeholderMap = {
                nombre: 'Insertar nombre',
                liga_web: 'Insertar liga web',
                fuente: 'Insertar fuente',
                año: 'Insertar año',
                valor: 'Insertar valor'
            };
            if (placeholderMap[col.keyName] && (!val || val.trim() === '')) {
                textarea.placeholder = placeholderMap[col.keyName];
            }
            textarea.onfocus = function() {
                const parentTd = this.closest('td');
                if (parentTd) parentTd.classList.add('cell-focused');
                this.style.borderColor = 'var(--primary)';
                this.style.background = 'white';
                this.style.boxShadow = '0 0 0 3px rgba(74,85,104,0.15)';
            };
            textarea.onblur = function() {
                const parentTd = this.closest('td');
                if (parentTd) parentTd.classList.remove('cell-focused');
                this.style.borderColor = 'transparent';
                this.style.background = 'transparent';
                this.style.boxShadow = 'none';
                if (this.value !== val) {
                    val = this.value;
                    rowData[col.keyName] = this.value;
                    saveRow(tr);
                    syncRowHeights(tr);
                }
            };
            textarea.oninput = function() {
                setTimeout(() => syncRowHeights(tr), 0);
            };
            td.appendChild(textarea);
        }
        tr.appendChild(td);
    });
    return tr;
}

function toggleChildren(parentId, btn) {
    const parentRow = document.querySelector(`tr[data-id="${parentId}"]`);
    if (!parentRow) {
        console.warn(`⚠️ No se encontró fila padre con id ${parentId}`);
        return;
    }

    const isExpanded = btn.innerHTML === '▾';
    const children = document.querySelectorAll(`tr[data-parent="${parentId}"]`);

    if (isExpanded) {
        children.forEach(child => child.classList.add('hidden'));
        btn.innerHTML = '▸';
        console.log(`🔽 Colapsados ${children.length} hijos de ${parentId}`);
    } else {
        children.forEach(child => child.classList.remove('hidden'));
        btn.innerHTML = '▾';
        console.log(`🔼 Expandidos ${children.length} hijos de ${parentId}`);
        requestAnimationFrame(() => {
            children.forEach(child => {
                if (!child.classList.contains('hidden')) {
                    syncRowHeights(child);
                }
            });
            syncRowHeights(parentRow);
        });
    }
}

function renderPreviewTable(containerId, data, isZip = false, disablePagination = false, allData = null) {
    const container = document.getElementById(containerId);
    if (!container) {
        console.error(`❌ Contenedor ${containerId} no encontrado`);
        return;
    }
    if (!data || data.length === 0) {
        container.innerHTML = '<p>No hay datos para mostrar.</p>';
        return;
    }

    const fullData = allData || data;
    const columnDefs = window.COLUMN_DEFINITIONS || [
        { displayName: 'ID', keyName: 'id' },
        { displayName: 'Proceso', keyName: 'proceso' },
        { displayName: 'Eje', keyName: 'eje' },
        { displayName: 'Tema', keyName: 'tema' },
        { displayName: 'Nombre', keyName: 'nombre' },
        { displayName: 'Institución', keyName: 'institucion' },
        { displayName: 'Cobertura', keyName: 'cobertura' },
        { displayName: 'Periodicidad', keyName: 'periodicidad' },
        { displayName: 'Liga Web', keyName: 'liga_web' },
        { displayName: 'Fuente', keyName: 'fuente' },
        { displayName: 'Año', keyName: 'año' },
        { displayName: 'Estado', keyName: 'estado' },
        { displayName: 'Valor', keyName: 'valor' }
    ];

    const hasParent = fullData.some(r => r._is_parent === true);
    const hasChild = fullData.some(r => r._is_child === true);
    const useHierarchy = hasParent && hasChild;

    const fragment = document.createDocumentFragment();
    const table = document.createElement('table');
    const thead = document.createElement('thead');
    const tbody = document.createElement('tbody');

    // Cabecera
    const trHead = document.createElement('tr');
    const expandWidth = '40px';
    if (useHierarchy) {
        const th = document.createElement('th');
        th.style.cssText = `width:${expandWidth}; min-width:${expandWidth}; max-width:${expandWidth}; text-align:center; vertical-align:middle;`;
        th.innerHTML = `<span title="Expandir/Colapsar todos" onclick="toggleAllPreviewParents('${containerId}')" style="font-size:1.2em;cursor:pointer;">↕</span>`;
        trHead.appendChild(th);
    } else {
        const th = document.createElement('th');
        th.style.cssText = `width:${expandWidth}; min-width:${expandWidth}; max-width:${expandWidth};`;
        trHead.appendChild(th);
    }
    const actionWidth = '90px';
    const thCheck = document.createElement('th');
    thCheck.style.cssText = `width:${actionWidth}; min-width:${actionWidth}; max-width:${actionWidth}; text-align:center; vertical-align:middle;`;
    thCheck.innerHTML = `<input type="checkbox" id="masterCheckbox_${containerId}" onchange="toggleAllPreviewCheckboxes(this, '${containerId}', ${isZip})">`;
    trHead.appendChild(thCheck);
    columnDefs.forEach(col => {
        const th = document.createElement('th');
        const width = COLUMN_WIDTHS[col.keyName] || 'auto';
        th.style.cssText = `width:${width}; min-width:${width}; max-width:${width};`;
        th.textContent = col.displayName;
        trHead.appendChild(th);
    });
    thead.appendChild(trHead);
    table.appendChild(thead);

    // Construir mapa de hijos por hash del padre (usando fullData)
    const childrenMap = {};
    fullData.forEach(row => {
        if (row._is_child && row._parent_hash) {
            if (!childrenMap[row._parent_hash]) childrenMap[row._parent_hash] = [];
            childrenMap[row._parent_hash].push(row);
        }
    });

    // Renderizar solo los padres (data ya contiene solo padres)
    data.forEach((rowInfo, idx) => {
        const isParent = rowInfo._is_parent === true;
        const rowData = rowInfo.data || rowInfo;
        const isDup = rowInfo._is_duplicate || false;
        const checked = (rowInfo._action === 'insert' || rowInfo._action === 'overwrite');
        const cls = isDup ? 'duplicate-row-preview' : '';
        const hash = rowInfo._hash;

        const tr = document.createElement('tr');
        tr.className = `${cls} parent-row`;
        tr.dataset.idx = rowInfo.original_csv_index;
        tr.dataset.hash = hash;

        // Celda de expansión
        const tdExpand = document.createElement('td');
        tdExpand.style.cssText = `width:${expandWidth}; min-width:${expandWidth}; max-width:${expandWidth}; text-align:center; vertical-align:middle; padding:2px 4px;`;
        if (useHierarchy) {
            const btn = document.createElement('span');
            btn.className = 'expand-btn';
            btn.textContent = '▸';
            btn.dataset.parentHash = hash;
            btn.style.cursor = 'pointer';
            btn.style.fontSize = '1.1em';
            tdExpand.appendChild(btn);
        }
        tr.appendChild(tdExpand);

        // Celda de checkbox con evento para seleccionar hijos
        const tdCheck = document.createElement('td');
        tdCheck.style.cssText = `text-align:center; padding:2px 4px; vertical-align:middle;`;
        const cb = document.createElement('input');
        cb.type = 'checkbox';
        cb.className = 'action-checkbox';
        cb.dataset.hash = hash;
        cb.dataset.iszip = isZip;
        if (checked) cb.checked = true;
        // Evento para seleccionar hijos
        cb.addEventListener('change', function() {
            const parentHash = this.dataset.hash;
            const isChecked = this.checked;
            const children = fullData.filter(r => r._parent_hash === parentHash);
            children.forEach(child => {
                child._action = isChecked ? (child._is_duplicate ? 'overwrite' : 'insert') : 'ignore';
                // Actualizar checkbox en DOM si existe
                const childCb = container.querySelector(`.action-checkbox[data-hash="${child._hash}"]`);
                if (childCb) childCb.checked = isChecked;
            });
        });
        tdCheck.appendChild(cb);
        tr.appendChild(tdCheck);

        // Celdas de datos
        columnDefs.forEach(col => {
            const td = document.createElement('td');
            const width = COLUMN_WIDTHS[col.keyName] || 'auto';
            td.style.cssText = `width:${width}; min-width:${width}; max-width:${width}; padding:2px 4px; vertical-align:top;`;
            let val = (rowData[col.keyName] !== undefined && rowData[col.keyName] !== null) ? String(rowData[col.keyName]) : '';
            if (col.keyName === 'valor' && val !== '' && !isNaN(val)) val = formatNumero(val);
            td.textContent = val;
            tr.appendChild(td);
        });

        tbody.appendChild(tr);
    });

    table.appendChild(tbody);
    fragment.appendChild(table);
    container.innerHTML = '';
    container.appendChild(fragment);

    // Manejar eventos de expansión (MEJORADO)
    if (useHierarchy) {
        container.querySelectorAll('.expand-btn').forEach(btn => {
            btn.addEventListener('click', function(e) {
                e.stopPropagation();
                const parentHash = this.dataset.parentHash;
                const isExpanded = this.textContent === '▾';

                if (isExpanded) {
                    // Colapsar: eliminar todos los hijos
                    const childrenRows = container.querySelectorAll(`tr[data-parent-hash="${parentHash}"]`);
                    childrenRows.forEach(row => row.remove());
                    this.textContent = '▸';
                    const parentRow = container.querySelector(`tr[data-hash="${parentHash}"]`);
                    if (parentRow) requestAnimationFrame(() => adjustPreviewRowHeights(parentRow));
                } else {
                    // Expandir: buscar hijos en childrenMap
                    const children = childrenMap[parentHash] || [];
                    if (children.length === 0) {
                        showToast('Este padre no tiene hijos.');
                        return;
                    }
                    const parentRow = container.querySelector(`tr[data-hash="${parentHash}"]`);
                    if (!parentRow) return;

                    // Eliminar cualquier hijo residual (por si acaso)
                    const existingChildren = container.querySelectorAll(`tr[data-parent-hash="${parentHash}"]`);
                    existingChildren.forEach(row => row.remove());

                    // Insertar todos los hijos
                    let previousSibling = parentRow;
                    children.forEach((childInfo) => {
                        const childRow = createPreviewChildRow(childInfo, parentHash, containerId, isZip);
                        previousSibling.parentNode.insertBefore(childRow, previousSibling.nextSibling);
                        previousSibling = childRow;
                        requestAnimationFrame(() => adjustPreviewRowHeights(childRow));
                    });

                    // Sincronizar checkboxes de hijos con el padre si está marcado
                    const parentCb = parentRow.querySelector('.action-checkbox');
                    if (parentCb && parentCb.checked) {
                        const childCheckboxes = container.querySelectorAll(`tr[data-parent-hash="${parentHash}"] .action-checkbox`);
                        childCheckboxes.forEach(cb => cb.checked = true);
                    }

                    this.textContent = '▾';
                    requestAnimationFrame(() => adjustPreviewRowHeights(parentRow));
                }
            });
        });
    }

    // Mostrar controles y contar filas
    const controlsId = isZip ? 'zipPreviewContainer' : 'csvPreviewControls';
    const controls = document.getElementById(controlsId);
    if (controls) controls.style.display = 'block';

    const rowCountSpan = document.getElementById('rowCount');
    if (rowCountSpan) rowCountSpan.textContent = window._totalRows || fullData.length;

    if (!disablePagination) {
        previewUpdateControls(containerId, isZip, window._totalRows || fullData.length);
    }

    // Ajustar alturas de filas visibles
    const visibleRows = container.querySelectorAll('tbody tr:not(.hidden)');
    requestAnimationFrame(() => {
        visibleRows.forEach(row => adjustPreviewRowHeights(row));
    });
}


// Función auxiliar para crear una fila hija en la previsualización
function createPreviewChildRow(childInfo, parentHash, containerId, isZip) {
    const tr = document.createElement('tr');
    tr.className = 'child-row-preview';
    tr.dataset.parentHash = parentHash;
    tr.dataset.hash = childInfo._hash;

    const rowData = childInfo.data || childInfo;
    const isDup = childInfo._is_duplicate || false;
    const checked = (childInfo._action === 'insert' || childInfo._action === 'overwrite');
    if (isDup) tr.classList.add('duplicate-row-preview');

    // Celda de expansión (vacía con sangría)
    const tdExpand = document.createElement('td');
    tdExpand.style.cssText = 'width:40px; min-width:40px; max-width:40px; padding-left:20px;';
    tdExpand.innerHTML = '&nbsp;';
    tr.appendChild(tdExpand);

    // Celda de checkbox
    const tdCheck = document.createElement('td');
    tdCheck.style.cssText = 'text-align:center; padding:2px 4px; vertical-align:middle;';
    const cb = document.createElement('input');
    cb.type = 'checkbox';
    cb.className = 'action-checkbox';
    cb.dataset.hash = childInfo._hash;
    cb.dataset.iszip = isZip;
    if (checked) cb.checked = true;
    tdCheck.appendChild(cb);
    tr.appendChild(tdCheck);

    // Celdas de datos
    const columnDefs = window.COLUMN_DEFINITIONS || [];
    columnDefs.forEach(col => {
        const td = document.createElement('td');
        const width = COLUMN_WIDTHS[col.keyName] || 'auto';
        td.style.cssText = `width:${width}; min-width:${width}; max-width:${width}; padding:2px 4px; vertical-align:top;`;
        let val = (rowData[col.keyName] !== undefined && rowData[col.keyName] !== null) ? String(rowData[col.keyName]) : '';
        if (col.keyName === 'valor' && val !== '' && !isNaN(val)) val = formatNumero(val);
        td.textContent = val;
        tr.appendChild(td);
    });

    return tr;
}

// Función para expandir/colapsar todos los padres en la previsualización (opcional)
function toggleAllPreviewParents(containerId) {
    const container = document.getElementById(containerId);
    if (!container) return;
    const allParentRows = container.querySelectorAll('tbody tr:not(.child-row-preview)');
    if (allParentRows.length === 0) return;
    let anyExpanded = false;
    const firstParent = allParentRows[0];
    if (firstParent) {
        const btn = firstParent.querySelector('.expand-btn');
        if (btn && btn.textContent === '▼') anyExpanded = true;
    }
    const newState = anyExpanded ? 'collapse' : 'expand';
    allParentRows.forEach(parentRow => {
        const btn = parentRow.querySelector('.expand-btn');
        if (btn) {
            // Simular clic en el botón
            btn.click();
        }
    });
}

function previewPrevPage(containerId, isZip) {
    const token = previewToken || window._previewToken;
    if (!token) return;
    const currentPage = previewCurrentPage || 1;
    if (currentPage > 1) {
        const newPage = currentPage - 1;
        loadPreviewPage(token, newPage, previewItemsPerPage);
    }
}

function previewNextPage(containerId, isZip) {
    const token = previewToken || window._previewToken;
    if (!token) return;
    const totalPages = Math.ceil(window._totalRows / previewItemsPerPage) || 1;
    const currentPage = previewCurrentPage || 1;
    if (currentPage < totalPages) {
        const newPage = currentPage + 1;
        loadPreviewPage(token, newPage, previewItemsPerPage);
    }
}

function previewUpdateControls(containerId, isZip, totalParents) {
    const pageSize = parentsPerPage || previewItemsPerPage;
    const totalPages = Math.ceil(totalParents / pageSize) || 1;
    let currentPage = currentParentPage || 1;
    if (currentPage > totalPages) currentPage = totalPages;
    if (currentPage < 1) currentPage = 1;
    currentParentPage = currentPage;
    previewCurrentPage = currentPage;

    const controlsId = isZip ? 'zipPaginationControls' : 'paginationControls';
    const prevBtnId = isZip ? 'zipPrevPageBtn' : 'prevPageBtn';
    const nextBtnId = isZip ? 'zipNextPageBtn' : 'nextPageBtn';
    const controls = document.getElementById(controlsId);
    const prevBtn = document.getElementById(prevBtnId);
    const nextBtn = document.getElementById(nextBtnId);
    if (!controls) return;
    controls.style.display = 'flex';
    if (prevBtn) prevBtn.disabled = currentPage <= 1;
    if (nextBtn) nextBtn.disabled = currentPage >= totalPages;

    const pageInputId = isZip ? 'zipPageInput' : 'previewPageInput';
    const pageInput = document.getElementById(pageInputId);
    if (pageInput) {
        pageInput.value = currentPage;
        pageInput.min = 1;
        pageInput.max = totalPages;
        // Eliminar listeners anteriores para evitar duplicados
        pageInput.removeEventListener('change', pageInput._changeHandler);
        pageInput.removeEventListener('keyup', pageInput._keyupHandler);
        const changeHandler = function() {
            const newPage = parseInt(this.value);
            if (!isNaN(newPage) && newPage >= 1 && newPage <= totalPages) {
                goToPreviewPage(newPage, containerId, isZip);
            } else {
                this.value = currentPage;
            }
        };
        const keyupHandler = function(e) {
            if (e.key === 'Enter') {
                this.blur(); // dispara change
            }
        };
        pageInput.addEventListener('change', changeHandler);
        pageInput.addEventListener('keyup', keyupHandler);
        pageInput._changeHandler = changeHandler;
        pageInput._keyupHandler = keyupHandler;
    }
    const totalSpanId = isZip ? 'zipTotalPages' : 'previewTotalPages';
    const totalSpan = document.getElementById(totalSpanId);
    if (totalSpan) totalSpan.textContent = totalPages;
}

function updateSelectAllCheckbox(containerId, isZip, disablePagination) {
    const container = document.getElementById(containerId);
    if (!container) return;
    const targetData = folderPreviewData; // todos los datos
    const dups = targetData.filter(r => r._is_duplicate);
    const selAllId = isZip ? 'selectAllDuplicatesForOverwriteZip' : 'selectAllDuplicatesForOverwrite';
    const selAll = document.getElementById(selAllId);
    if (selAll) {
        const allSelected = dups.every(r => overwriteSet.has(r._hash));
        selAll.checked = allSelected;
        // Remover listener antiguo y asignar nuevo
        const newOnChange = () => {
            targetData.forEach(r => {
                if (r._is_duplicate) {
                    const action = selAll.checked ? 'overwrite' : 'ignore';
                    r._action = action;
                    if (action === 'overwrite') {
                        overwriteSet.add(r._hash);
                    } else {
                        overwriteSet.delete(r._hash);
                    }
                }
            });
            // Renderizar solo padres de la página actual
            const start = (currentParentPage - 1) * parentsPerPage;
            const end = Math.min(start + parentsPerPage, totalParents);
            const parents = folderPreviewData.filter(r => r._is_parent == true).slice(start, end);
            renderPreviewTable(containerId, parents, isZip, true, folderPreviewData);
            const masterCb = document.getElementById(`masterCheckbox_${containerId}`);
            if (masterCb) {
                const allCheckboxes = container.querySelectorAll('.action-checkbox');
                const allChecked = Array.from(allCheckboxes).every(cb => cb.checked);
                masterCb.checked = allChecked;
            }
            updateSelectAllNonDuplicatesCheckbox(containerId, isZip);
            previewUpdateControls(containerId, isZip, totalParents);
        };
        selAll.onchange = newOnChange;
    }
}

async function handleFileUploadModal(input) {
    if (isProcessing) {
        showToast('⏳ Ya hay una operación en curso, espera a que termine.');
        input.value = '';
        return;
    }
    const file = input.files[0];
    if (!file) return;
    if (!file.name.endsWith('.csv')) {
        showToast('⚠️ Por favor, sube un archivo CSV.');
        input.value = '';
        return;
    }
    currentFileToUpload = file;
    isProcessing = true;
    document.getElementById('confirmUploadBtn').disabled = true;
    document.getElementById('uploadPreviewMessages').innerHTML = '';
    document.getElementById('uploadDataPreview').innerHTML = '';
    document.getElementById('uploadDataPreview').style.display = 'none';
    document.getElementById('paginationControls').style.display = 'none';
    const progressDiv = document.getElementById('uploadProgress');
    const progressFill = document.getElementById('uploadProgressFill');
    const progressText = document.getElementById('uploadProgressText');
    progressDiv.style.display = 'block';
    progressFill.style.width = '0%';
    progressText.textContent = 'Subiendo archivo...';
    const fd = new FormData();
    fd.append('file', file);
    try {
        let progress = 0;
        const interval = setInterval(() => {
            progress += Math.random() * 15;
            if (progress > 90) progress = 90;
            progressFill.style.width = progress + '%';
            progressText.textContent = `Procesando... ${Math.round(progress)}%`;
        }, 300);
        const r = await fetch('/api/upload?preview=true', { method: 'POST', body: fd });
        clearInterval(interval);
        progressFill.style.width = '100%';
        progressText.textContent = '¡Completado!';
        setTimeout(() => { progressDiv.style.display = 'none'; }, 1000);
        if (r.ok) {
            const data = await r.json();
            document.getElementById('uploadPreviewMessages').innerHTML = data.messages.map(m => `<div>${escapeHtml(m)}</div>`).join('');
            allPreviewData = data.preview_data || [];
            allPreviewData.forEach(row => {
                row._action = row._is_duplicate ? 'ignore' : 'insert';
            });
            previewCurrentPage = 1;
            document.getElementById('uploadDataPreview').style.display = 'block';
            document.getElementById('paginationControls').style.display = 'flex';
            renderPreviewTable('uploadDataPreview', allPreviewData, false);
            document.getElementById('confirmUploadBtn').disabled = false;
        } else {
            const err = await r.json();
            document.getElementById('uploadPreviewMessages').innerHTML = `<div style="color:var(--danger);">Error: ${err.error || 'Algo salió mal'}</div>`;
            allPreviewData = [];
            document.getElementById('confirmUploadBtn').disabled = true;
        }
    } catch (e) {
        document.getElementById('uploadPreviewMessages').innerHTML = `<div style="color:var(--danger);">Error de conexión: ${e.message}</div>`;
        allPreviewData = [];
        document.getElementById('confirmUploadBtn').disabled = true;
    } finally {
        isProcessing = false;
    }
}

async function confirmUpload() {
    if (isProcessing) { showToast('⏳ Ya hay una operación en curso.'); return; }
    if (!currentFileToUpload) { showToast('Error: No hay archivo para cargar.'); return; }
    if (allPreviewData.length === 0) { showToast('No hay datos para confirmar.'); return; }
    isProcessing = true;
    document.getElementById('confirmUploadBtn').disabled = true;

    const actions = allPreviewData.map(row => ({
        original_csv_index: row.original_csv_index,
        action: row._action,
        data: row.data,
        _is_duplicate: row._is_duplicate,
        _supabase_matching_id: row._supabase_matching_id,
        _hash: row._hash,
        _is_parent: row._is_parent || false
    }));

    const fd = new FormData();
    fd.append('file', currentFileToUpload);
    fd.append('actions_for_rows', JSON.stringify(actions));

    try {
        const r = await fetch('/api/upload', { method: 'POST', body: fd });
        const data = await r.json();
        if (data.debug) console.log("%c 📡 Logs del backend (CSV): ", "background: #222; color: #00ffaa; font-size: 14px;", data.debug);
        if (r.ok) {
            showToast(data.message);
            await loadData();
            loadHistory();
            await refreshCatalogOptions();
            closeModal('modalConfirmUpload');
            resetCsvModal();
        } else {
            showToast(`Error: ${data.error || 'Algo salió mal'}`);
        }
    } catch (e) {
        showToast('Error de conexión: ' + e.message);
    } finally {
        isProcessing = false;
    }
}

async function handleZipUpload(input) {
    if (isProcessing) {
        showToast('⏳ Ya hay una operación en curso, espera a que termine.');
        input.value = '';
        return;
    }
    const file = input.files[0];
    if (!file) return;
    if (!file.name.endsWith('.zip')) {
        showToast('⚠️ Por favor, selecciona un archivo ZIP.');
        input.value = '';
        return;
    }

    isProcessing = true;
    overwriteSet = new Set();
    previewToken = null;
    window._previewToken = null;
    document.getElementById('confirmFolderBtn').disabled = true;

    const progressDiv = document.getElementById('zipProgress');
    const progressFill = document.getElementById('zipProgressFill');
    const progressText = document.getElementById('zipProgressText');
    progressDiv.style.display = 'block';
    progressFill.style.width = '0%';
    progressText.textContent = 'Subiendo archivo...';

    const previewContainer = document.getElementById('zipPreviewContainer');
    if (previewContainer) previewContainer.style.display = 'none';
    const logContainer = document.getElementById('folderLogMessages');
    if (logContainer) logContainer.style.display = 'none';

    const formData = new FormData();
    formData.append('file', file);

    try {
        const response = await fetch('/api/process-folder-async', {
            method: 'POST',
            body: formData
        });
        if (!response.ok) {
            const error = await response.json();
            throw new Error(error.error || 'Error al iniciar la tarea');
        }
        const data = await response.json();
        const taskId = data.task_id;

        if (logContainer) {
            logContainer.style.display = 'block';
            document.getElementById('folderLogContent').innerHTML = '⏳ Procesando, esto puede tomar varios minutos...';
        }

        let pollInterval = setInterval(async () => {
            try {
                const statusResp = await fetch(`/api/task-status/${taskId}`);
                if (!statusResp.ok) {
                    clearInterval(pollInterval);
                    showToast('Error al consultar el estado de la tarea.');
                    isProcessing = false;
                    return;
                }
                const statusData = await statusResp.json();

                if (statusData.progress !== undefined) {
                    const progress = Math.min(statusData.progress, 100);
                    progressFill.style.width = progress + '%';
                    progressText.textContent = `Procesando... ${progress}%`;
                }

                if (statusData.messages && statusData.messages.length > 0 && logContainer) {
                    const logContent = document.getElementById('folderLogContent');
                    logContent.innerHTML = statusData.messages.map(m => escapeHtml(m)).join('<br>');
                    logContent.scrollTop = logContent.scrollHeight;
                }

                if (statusData.status === 'completed') {
                    clearInterval(pollInterval);
                    const result = statusData.result;
                    previewToken = result.preview_token;
                    window._previewToken = result.preview_token;
                    window._totalRows = result.total_rows;

                    console.log('Token recibido:', previewToken, 'Total filas:', window._totalRows);
                    folderGlobalMetadata = result.global_metadata || {};
                    currentZipFilename = file.name;

                    if (logContainer && result.messages && result.messages.length > 0) {
                        logContainer.style.display = 'block';
                        document.getElementById('folderLogContent').innerHTML = result.messages.map(m => escapeHtml(m)).join('<br>');
                    }

                    if (previewToken) {
                        const fetchAllResp = await fetch('/api/preview-page', {
                            method: 'POST',
                            headers: { 'Content-Type': 'application/json' },
                            body: JSON.stringify({
                                token: previewToken,
                                page: 1,
                                page_size: 9999
                            })
                        });
                        if (fetchAllResp.ok) {
                            const allData = await fetchAllResp.json();
                            allFolderData = allData.preview_data || [];
                            // Asignar a folderPreviewData para uso en otras funciones
                            folderPreviewData = allFolderData;
                            console.log('📦 allFolderData (primeros 3):', allFolderData.slice(0, 3));
                            console.log('📦 total elementos:', allFolderData.length);

                            // Establecer acciones por defecto
                            allFolderData.forEach(row => {
                                if (row._is_duplicate) {
                                    row._action = 'ignore'; // duplicados no seleccionados por defecto
                                } else {
                                    row._action = 'insert'; // no duplicados seleccionados por defecto
                                }
                                // Inicializar _temp_new_values si no existe
                                if (!row._temp_new_values) row._temp_new_values = {};
                                if (!row._nuevo_en_catalogo) row._nuevo_en_catalogo = {};
                            });

                            totalParents = allFolderData.filter(r => r._is_parent == true).length;
                            console.log('📦 totalParents calculado:', totalParents);
                            currentParentPage = 1;
                            parentsPerPage = 10;

                            if (previewContainer && totalParents > 0) {
                                previewContainer.style.display = 'block';
                                document.getElementById('rowCount').textContent = totalParents;

                                const start = 0;
                                const end = Math.min(parentsPerPage, totalParents);
                                const parents = allFolderData.filter(r => r._is_parent == true).slice(start, end);
                                console.log('📦 padres a renderizar (primeros):', parents.length);

                                renderPreviewTable('zipPreviewTable', parents, true, true, allFolderData);
                                previewUpdateControls('zipPreviewTable', true, totalParents);

                                // Sincronizar checkboxes de selección masiva
                                updateSelectAllNonDuplicatesCheckbox('zipPreviewTable', true);
                                // El checkbox de duplicados se actualiza en updateSelectAllCheckbox

                                document.getElementById('confirmFolderBtn').disabled = false;
                                showToast(`✓ Procesamiento completado: ${totalParents} filas padre extraídas.`);
                            } else {
                                showToast('⚠️ No se encontraron datos para previsualizar. Revisa los logs.');
                                document.getElementById('confirmFolderBtn').disabled = true;
                            }
                        } else {
                            showToast('Error al obtener los datos de previsualización.');
                            document.getElementById('confirmFolderBtn').disabled = true;
                        }
                    } else {
                        showToast('⚠️ No se generó token de previsualización.');
                        document.getElementById('confirmFolderBtn').disabled = true;
                    }

                    progressText.textContent = '¡Procesamiento completado!';
                    isProcessing = false;

                } else if (statusData.status === 'error') {
                    clearInterval(pollInterval);
                    showToast('❌ Error en el procesamiento: ' + (statusData.error || 'Error desconocido'));
                    if (logContainer) {
                        logContainer.style.display = 'block';
                        let html = `<strong>Error:</strong><br>${escapeHtml(statusData.error || '')}`;
                        if (statusData.messages && statusData.messages.length) {
                            html += '<br><br><strong>Logs:</strong><br>' + statusData.messages.map(m => escapeHtml(m)).join('<br>');
                        }
                        document.getElementById('folderLogContent').innerHTML = html;
                    }
                    isProcessing = false;
                    document.getElementById('confirmFolderBtn').disabled = true;
                }
            } catch (e) {
                clearInterval(pollInterval);
                showToast('Error de conexión: ' + e.message);
                isProcessing = false;
            }
        }, 2000);

    } catch (e) {
        showToast('Error: ' + e.message);
        isProcessing = false;
        if (logContainer) {
            logContainer.style.display = 'block';
            document.getElementById('folderLogContent').innerHTML = 'Error: ' + escapeHtml(e.message);
        }
    }
}

async function loadPreviewPage(token, page, pageSize) {
    console.log('loadPreviewPage llamada con token:', token, 'página:', page);
    try {
        const resp = await fetch('/api/preview-page', {
            method: 'POST',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({ token, page, page_size: pageSize })
        });
        if (resp.ok) {
            const data = await resp.json();
            console.log('Respuesta de preview-page:', data);

            if (!allFolderData || allFolderData.length === 0) {
                allFolderData = data.preview_data || [];
            }
            console.log('allFolderData (loadPreviewPage):', allFolderData.length, 'elementos');
            // Filtro tolerante
            totalParents = allFolderData.filter(r => r._is_parent == true).length;
            console.log('totalParents (loadPreviewPage):', totalParents);
            currentParentPage = page;
            parentsPerPage = pageSize;
            previewCurrentPage = page;

            const start = (page - 1) * pageSize;
            const end = Math.min(start + pageSize, totalParents);
            const parents = allFolderData.filter(r => r._is_parent == true).slice(start, end);
            console.log('padres a renderizar (loadPreviewPage):', parents.length);

            renderPreviewTable('zipPreviewTable', parents, true, true, allFolderData);
            previewUpdateControls('zipPreviewTable', true, totalParents);

            document.getElementById('rowCount').textContent = totalParents;
        } else {
            showToast('Error al cargar página.');
        }
    } catch (e) {
        showToast('Error de conexión: ' + e.message);
    }
}

function goToZipPage(page) {
    const totalRows = folderPreviewData.length;
    const pageSize = previewItemsPerPage;
    const totalPages = Math.ceil(totalRows / pageSize) || 1;
    if (page < 1) page = 1;
    if (page > totalPages) page = totalPages;
    previewCurrentPage = page;
    window._currentPage = page;
    renderPreviewTable('zipPreviewTable', folderPreviewData, true, true);
    updatePaginationControls(true);
}

function updatePaginationControls(isZip) {
    const containerId = isZip ? 'zipPreviewTable' : 'uploadDataPreview';
    previewUpdateControls(containerId, isZip);
}

async function confirmFolderUpload() {
    if (isProcessing) {
        showToast('⏳ Ya hay una operación en curso.');
        return;
    }
    const token = previewToken || window._previewToken;
    if (!token) {
        showToast('⚠️ No hay token de previsualización. Carga un ZIP primero.');
        return;
    }
    if (window._totalRows === 0) {
        showToast('⚠️ No hay datos para confirmar.');
        return;
    }
    isProcessing = true;
    document.getElementById('confirmFolderBtn').disabled = true;
    const overwriteHashes = Array.from(overwriteSet);
    try {
        const response = await fetch('/api/confirm-folder', {
            method: 'POST',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({
                token: token,
                overwrite_hashes: overwriteHashes
            })
        });
        const data = await response.json();
        if (data.debug) {
            console.log('🐞 Debug logs del backend:', data.debug);
        }
        if (response.ok) {
            if (data.message && data.message.includes('Error')) {
                showToast('⚠️ ' + data.message);
            } else {
                showToast(data.message || '✓ Carga completada');
            }
            await loadData();
            loadHistory();
            await refreshCatalogOptions();
            closeModal('modalFolderUpload');
            resetFolderModal();
        } else {
            showToast('Error: ' + (data.error || 'Algo salió mal'));
        }
    } catch (e) {
        showToast('Error de conexión: ' + e.message);
    } finally {
        isProcessing = false;
        if (previewToken) {
            document.getElementById('confirmFolderBtn').disabled = false;
        }
    }
}

async function loadHistory() {
    try {
        const r = await fetch('/api/upload-history');
        if (r.ok) {
            const history = await r.json();
            const container = document.getElementById('historyListBody');
            if (history.length === 0) {
                container.innerHTML = '<tr><td colspan="4" style="text-align:center; color:var(--text-muted);">No hay cargas registradas.</td></tr>';
                return;
            }
            let html = '';
            history.forEach(item => {
                let fechaFormateada = '';
                if (item.fecha) {
                    try {
                        const date = new Date(item.fecha);
                        if (!isNaN(date.getTime())) {
                            const dia = String(date.getDate()).padStart(2, '0');
                            const mes = String(date.getMonth() + 1).padStart(2, '0');
                            const año = date.getFullYear();
                            const horas = String(date.getHours()).padStart(2, '0');
                            const minutos = String(date.getMinutes()).padStart(2, '0');
                            const segundos = String(date.getSeconds()).padStart(2, '0');
                            fechaFormateada = `${dia}/${mes}/${año} ${horas}:${minutos}:${segundos}`;
                        } else { fechaFormateada = item.fecha; }
                    } catch { fechaFormateada = item.fecha; }
                }
                const statusClass = item.estado === 'completado' ? 'status-success' :
                                  item.estado === 'error' ? 'status-error' : 'status-pending';
                html += `<tr>
                    <td>${escapeHtml(fechaFormateada)}</td>
                    <td>${escapeHtml(item.archivo || '')}</td>
                    <td>${escapeHtml(item.filas_agregadas || '')}</td>
                    <td class="${statusClass}">${escapeHtml(item.estado || '')}</td>
                </tr>`;
            });
            container.innerHTML = html;
        }
    } catch (e) {
        console.warn('Error al cargar historial:', e);
    }
}

async function retrainModels() {
    const btn = document.querySelector('button[onclick="retrainModels()"]');
    if (!btn) return;
    btn.disabled = true;
    btn.textContent = '⏳ Entrenando...';
    showToast('⏳ Entrenando modelos, esto puede tomar unos segundos...');
    try {
        const r = await fetch('/api/retrain-models', { method: 'POST' });
        const data = await r.json();
        if (r.ok) {
            let msg = '✅ ' + data.message;
            if (data.report) {
                const p = data.report.proceso || {};
                const e = data.report.eje || {};
                const t = data.report.tema || {};
                const precision = `P:${(p.accuracy*100||0).toFixed(1)}% | E:${(e.accuracy*100||0).toFixed(1)}% | T:${(t.accuracy*100||0).toFixed(1)}%`;
                msg += ` (${precision})`;
                if (p.accuracy < 0.7 || e.accuracy < 0.7 || t.accuracy < 0.7) {
                    msg += ' ⚠️ Precisión baja. Considera revisar datos.';
                }
            }
            showToast(msg);
        } else {
            showToast('❌ Error: ' + (data.error || 'Algo salió mal'));
        }
    } catch (e) {
        showToast('❌ Error de conexión: ' + e.message);
    } finally {
        btn.disabled = false;
        btn.textContent = '🔄 Reentrenar Modelos';
    }
}

async function rollbackModels() {
    if (!confirm('⚠️ ¿Restaurar los modelos a la versión anterior? Se perderán los cambios del último reentrenamiento.')) return;
    const r = await fetch('/api/rollback-models', { method: 'POST' });
    const data = await r.json();
    if (r.ok) showToast('✅ ' + data.message);
    else showToast('❌ ' + (data.error || 'No hay backup disponible'));
}

const dropZone = document.getElementById('dropZone');
if (dropZone) {
    ['dragenter', 'dragover', 'dragleave', 'drop'].forEach(evt => {
        dropZone.addEventListener(evt, e => { e.preventDefault(); e.stopPropagation(); }, false);
    });
    dropZone.addEventListener('dragenter', () => dropZone.classList.add('drag-over'), false);
    dropZone.addEventListener('dragover', () => dropZone.classList.add('drag-over'), false);
    dropZone.addEventListener('dragleave', () => dropZone.classList.remove('drag-over'), false);
    dropZone.addEventListener('drop', e => {
        dropZone.classList.remove('drag-over');
        const file = e.dataTransfer.files[0];
        if (file) {
            if (file.name.endsWith('.csv')) {
                openModal('modalConfirmUpload');
                const input = document.getElementById('csvFileInputModal');
                const dataTransfer = new DataTransfer();
                dataTransfer.items.add(file);
                input.files = dataTransfer.files;
                const event = new Event('change', { bubbles: true });
                input.dispatchEvent(event);
            } else if (file.name.endsWith('.zip')) {
                openModal('modalFolderUpload');
                const input = document.getElementById('zipFileInput');
                const dataTransfer = new DataTransfer();
                dataTransfer.items.add(file);
                input.files = dataTransfer.files;
                const event = new Event('change', { bubbles: true });
                input.dispatchEvent(event);
            } else {
                showToast('⚠️ Solo se permiten archivos CSV o ZIP.');
            }
        }
    }, false);
} else {
    console.warn('Elemento #dropZone no encontrado.');
}

async function addMainTableOption(col, rowData, displayDiv, tr, currentVal, selectEl) {
    const modal = document.getElementById('modalOtraOpcion');
    const input = document.getElementById('otraOpcionInput');
    const btn = document.getElementById('btnConfirmOtra');
    document.getElementById('modalOtraTitle').textContent = 'Agregar ' + col.displayName;
    input.value = '';
    modal.style.display = 'flex';
    input.focus();
    const newBtn = btn.cloneNode(true);
    btn.parentNode.replaceChild(newBtn, btn);
    newBtn.onclick = async () => {
        const newVal = input.value.trim();
        if (!newVal) {
            handleCloseOtraOpcionModal();
            return;
        }
        try {
            const resp = await fetch(`/api/catalog/${col.keyName}`, {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ value: newVal })
            });
            const data = await resp.json();
            const applyValue = (finalVal) => {
                if (!col.options.includes(finalVal)) {
                    col.options.push(finalVal);
                    col.options.sort();
                }
                rowData[col.keyName] = finalVal;
                displayDiv.textContent = finalVal;
                displayDiv.title = finalVal;
                if (col.keyName === 'eje') {
                    updateTemaOptions(tr, finalVal, rowData['tema']);
                }
                if (selectEl) {
                    const selectedValue = finalVal;
                    let html = `<option value="">-seleccionar-</option>`;
                    col.options.forEach(o => {
                        html += `<option value="${o}" ${o === selectedValue ? 'selected' : ''}>${o}</option>`;
                    });
                    html += `<option value="_OTRA_">Otra...</option>`;
                    selectEl.innerHTML = html;
                    selectEl.value = selectedValue;
                }
                saveRow(tr);
                syncRowHeights(tr);
                showToast('✓ Opción agregada');
            };
            if (data.status === 'suggestion') {
                handleCloseOtraOpcionModal();
                openSuggestionModal(
                    newVal,
                    data.suggested_value,
                    (finalVal) => { applyValue(finalVal); closeModal('modalSuggestion'); },
                    () => { applyValue(newVal); closeModal('modalSuggestion'); }
                );
                return;
            }
            if (data.status === 'already_exists') {
                handleCloseOtraOpcionModal();
                showToast(`'${data.value}' ya existe en el catálogo.`);
                applyValue(data.value);
                return;
            }
            handleCloseOtraOpcionModal();
            applyValue(newVal);
        } catch (e) {
            handleCloseOtraOpcionModal();
            showToast('Error: ' + e.message);
        }
    };
    _currentOtraOpcionCancelCallback = null;
}

async function refreshCatalogOptions() {
    console.log("🔄 Refrescando opciones de catálogo...");
    for (let col of window.COLUMN_DEFINITIONS) {
        if (col.type === 'select' && col.keyName !== 'id') {
            try {
                const resp = await fetch(`/api/catalog/${col.keyName}`);
                if (resp.ok) {
                    const data = await resp.json();
                    if (data.options) {
                        col.options = data.options.sort();
                        console.log(`   ✅ ${col.keyName}: ${col.options.length} opciones`);
                    }
                } else {
                    console.warn(`   ⚠️ Error al obtener ${col.keyName}: ${resp.status}`);
                }
            } catch (e) {
                console.warn(`   ❌ Error en refreshCatalogOptions para ${col.keyName}:`, e);
            }
        }
    }
    console.log("✅ Catálogos actualizados.");
}

function toggleAllPreviewCheckboxes(master, containerId, isZip) {
    const container = document.getElementById(containerId);
    if (!container) return;
    const checkboxes = container.querySelectorAll('.action-checkbox');
    const checked = master.checked;
    checkboxes.forEach(cb => {
        cb.checked = checked;
        const hash = cb.dataset.hash;
        if (hash) {
            const row = folderPreviewData.find(r => r._hash === hash);
            if (row) {
                row._action = checked ? (row._is_duplicate ? 'overwrite' : 'insert') : 'ignore';
                if (row._is_duplicate) {
                    if (checked) overwriteSet.add(hash);
                    else overwriteSet.delete(hash);
                }
            }
        }
    });
    updateSelectAllCheckbox(containerId, isZip);
    updateSelectAllNonDuplicatesCheckbox(containerId, isZip);
}

function toggleAllNonDuplicates(master, isZip) {
    const checked = master.checked;
    // Actualizar acciones en todos los datos (folderPreviewData)
    folderPreviewData.forEach(row => {
        if (!row._is_duplicate) {
            row._action = checked ? 'insert' : 'ignore';
        }
    });
    // Renderizar solo los padres de la página actual
    const containerId = isZip ? 'zipPreviewTable' : 'uploadDataPreview';
    const start = (currentParentPage - 1) * parentsPerPage;
    const end = Math.min(start + parentsPerPage, totalParents);
    const parents = folderPreviewData.filter(r => r._is_parent == true).slice(start, end);
    renderPreviewTable(containerId, parents, isZip, true, folderPreviewData);
    // Actualizar checkbox maestro
    const masterCb = document.getElementById(`masterCheckbox_${containerId}`);
    if (masterCb) {
        const allCheckboxes = document.querySelectorAll(`#${containerId} .action-checkbox`);
        const allChecked = Array.from(allCheckboxes).every(cb => cb.checked);
        masterCb.checked = allChecked;
    }
    // Actualizar el checkbox de "seleccionar todos los no duplicados" (se mantiene marcado si corresponde)
    updateSelectAllNonDuplicatesCheckbox(containerId, isZip);
    // Actualizar controles de paginación (por si acaso)
    previewUpdateControls(containerId, isZip, totalParents);
}

function updateSelectAllNonDuplicatesCheckbox(containerId, isZip) {
    const targetData = isZip ? folderPreviewData : allPreviewData;
    const nonDups = targetData.filter(r => !r._is_duplicate);
    const allSelected = nonDups.every(r => r._action === 'insert' || r._action === 'overwrite');
    const cbId = isZip ? 'selectAllNonDuplicatesForOverwriteZip' : 'selectAllNonDuplicates';
    const cb = document.getElementById(cbId);
    if (cb) cb.checked = allSelected;
}

function renderPreviewRow(rowInfo, rowData, globalIndex, containerId, isZip, isParent, expandWidth, actionWidth, parentHash = null, parentIdx = null) {
    const isDup = rowInfo._is_duplicate || false;
    const nuevoCatalogo = rowInfo._nuevo_en_catalogo || {};
    const tieneNuevo = Object.values(nuevoCatalogo).some(v => v === true);
    const cls = (isDup ? 'duplicate-row-preview' : '') + (tieneNuevo ? ' has-new-catalog' : '');
    const isChecked = (rowInfo._action === 'insert' || rowInfo._action === 'overwrite');
    let expandHtml = `<td style="width:${expandWidth}; min-width:${expandWidth}; max-width:${expandWidth}; text-align:center; vertical-align:middle; padding:2px 4px;">`;
    if (isParent) {
        expandHtml += `<span class="expand-btn" data-parent-idx="${globalIndex}" style="cursor:pointer; font-size:1.1em;">▸</span>`;
    }
    expandHtml += `</td>`;
    let actionHtml = `<td style="padding:2px 4px; vertical-align:middle; text-align:center; width:${actionWidth}; min-width:${actionWidth}; max-width:${actionWidth};">
        <input type="checkbox" class="action-checkbox" data-index="${globalIndex}" data-iszip="${isZip}" ${isChecked ? 'checked' : ''}>
    </td>`;
    if (typeof window.COLUMN_DEFINITIONS === 'undefined') {
        console.error('❌ window.COLUMN_DEFINITIONS no está definido en renderPreviewRow');
        return `<tr><td colspan="16">Error: COLUMN_DEFINITIONS no definido</td></tr>`;
    }
    const rowSeg = window.COLUMN_DEFINITIONS.map(col => {
        let val = rowData[col.keyName] !== undefined && rowData[col.keyName] !== null ? String(rowData[col.keyName]) : '';
        if (col.keyName === 'valor' && val !== '' && !isNaN(val)) {
            val = formatNumero(val);
        }
        let inputHtml = '';
        if (col.type === 'select' && col.keyName !== 'id') {
            const tempVal = rowInfo._temp_new_values && rowInfo._temp_new_values[col.keyName] ? rowInfo._temp_new_values[col.keyName] : null;
            const isNew = nuevoCatalogo[col.keyName] || false;
            let opts = `<option value="">-seleccionar-</option>`;
            const optionsSet = new Set(col.options || []);
            if (tempVal && !optionsSet.has(tempVal)) {
                opts += `<option value="${escapeHtml(tempVal)}" selected>${escapeHtml(tempVal)}</option>`;
                if (!isNew) rowInfo._nuevo_en_catalogo[col.keyName] = true;
            } else if (!tempVal && isNew && val && !optionsSet.has(val)) {
                opts += `<option value="${escapeHtml(val)}" selected>${escapeHtml(val)}</option>`;
            }
            (col.options || []).forEach(o => {
                let selected = false;
                if (tempVal) selected = (o === tempVal);
                else selected = (o === val && val !== '');
                opts += `<option value="${escapeHtml(o)}" ${selected ? 'selected' : ''}>${escapeHtml(o)}</option>`;
            });
            opts += `<option value="_OTRA_">Otra...</option>`;
            inputHtml = `<select class="preview-edit" data-col="${col.keyName}" data-index="${globalIndex}" data-iszip="${isZip}">${opts}</select>`;
            if (isNew || tempVal) {
                inputHtml += ' <span style="color:orange; font-weight:bold;" title="Valor nuevo en catálogo">⚠️</span>';
            }
        } else if (col.keyName === 'id') {
            inputHtml = `<span class="preview-edit" style="display:block; width:100%;">${escapeHtml(val)}</span>`;
        } else {
            inputHtml = `<textarea class="preview-edit" data-col="${col.keyName}" data-index="${globalIndex}" data-iszip="${isZip}">${escapeHtml(val)}</textarea>`;
        }
        const width = COLUMN_WIDTHS[col.keyName] || 'auto';
        return `<td style="width:${width}; min-width:${width}; max-width:${width}; padding:2px 4px; vertical-align:top;">${inputHtml}</td>`;
    }).join('');
    const childClass = (parentIdx !== null) ? 'child-row-preview hidden' : '';
    const dataParentAttr = (parentIdx !== null) ? `data-parent-idx="${parentIdx}"` : '';
    const dataIdxAttr = isParent ? `data-idx="${globalIndex}"` : '';
    return `<tr class="${cls} ${childClass}" ${dataParentAttr} ${dataIdxAttr} data-hash="${rowInfo._hash}">${expandHtml}${actionHtml}${rowSeg}</tr>`;
}

function goToPreviewPage(page, containerId, isZip) {
    const token = previewToken || window._previewToken;
    if (!token) return;
    const totalPages = Math.ceil(totalParents / parentsPerPage) || 1;
    let p = parseInt(page);
    if (isNaN(p) || p < 1) p = 1;
    if (p > totalPages) p = totalPages;
    if (p !== currentParentPage) {
        loadPreviewPage(token, p, parentsPerPage);
    }
}

function adjustPreviewRowHeights(row) {
    if (!row) return;
    const cells = row.querySelectorAll('td');
    if (cells.length === 0) return;
    cells.forEach(td => {
        td.style.height = 'auto';
        td.style.minHeight = 'auto';
    });
    row.style.height = 'auto';
    const firstTd = cells[0];
    const style = getComputedStyle(firstTd);
    const tdPadding = parseFloat(style.paddingTop) + parseFloat(style.paddingBottom);
    let maxHeight = 0;
    cells.forEach(td => {
        const control = td.querySelector('textarea.preview-edit') || td.querySelector('div.preview-edit');
        if (control) {
            control.style.height = 'auto';
            void control.offsetHeight;
            const contentHeight = control.scrollHeight;
            const totalHeight = contentHeight + tdPadding;
            if (totalHeight > maxHeight) maxHeight = totalHeight;
        } else {
            const h = td.scrollHeight;
            if (h > maxHeight) maxHeight = h;
        }
    });
    if (maxHeight < 28) maxHeight = 28;
    const finalHeight = maxHeight + 2;
    cells.forEach(td => {
        td.style.height = finalHeight + 'px';
        td.style.minHeight = finalHeight + 'px';
    });
    row.style.height = finalHeight + 'px';
    row.querySelectorAll('textarea.preview-edit, div.preview-edit').forEach(ctrl => {
        ctrl.style.height = '100%';
        ctrl.style.minHeight = '100%';
    });
}

function insertChildrenForPreview(parentIdx, containerId, isZip) {
    const container = document.getElementById(containerId);
    if (!container) return;
    const childrenMap = window._previewChildrenMap && window._previewChildrenMap[containerId];
    if (!childrenMap) return;
    const children = childrenMap[parentIdx] || [];
    if (children.length === 0) return;
    const parentRow = container.querySelectorAll(`tr[data-parent="${parentId}"]`);
    if (!parentRow) return;
    const fragment = document.createDocumentFragment();
    children.forEach(child => {
        const tr = renderPreviewRow(child, child.data || child, 0, containerId, isZip, false, '40px', '90px', null, child._parentIdx);
        tr.classList.remove('hidden', 'child-row-preview');
        tr.dataset.inserted = 'true';
        fragment.appendChild(tr);
    });
    parentRow.parentNode.insertBefore(fragment, parentRow.nextSibling);
    const newRows = container.querySelectorAll(`tr[data-parent-idx="${parentIdx}"][data-inserted="true"]`);
    requestAnimationFrame(() => {
        newRows.forEach(row => adjustPreviewRowHeights(row));
        adjustPreviewRowHeights(parentRow);
    });
}

function togglePreviewChildren(parentIdx, containerId, isZip, btn) {
    const container = document.getElementById(containerId);
    if (!container) return;
    const isExpanded = btn.textContent === '▾';
    if (isExpanded) {
        const childrenRows = container.querySelectorAll(`tr[data-parent-idx="${parentIdx}"][data-inserted="true"]`);
        childrenRows.forEach(row => row.remove());
        btn.textContent = '▸';
        const parentRow = container.querySelectorAll(`tr[data-parent="${parentId}"]`);
        if (parentRow) adjustPreviewRowHeights(parentRow);
        return;
    }
    insertChildrenForPreview(parentIdx, containerId, isZip);
    btn.textContent = '▾';
    const parentRow = container.querySelectorAll(`tr[data-parent="${parentId}"]`);
    if (parentRow) adjustPreviewRowHeights(parentRow);
}

// ============================================================
// GESTIÓN DE USUARIOS (solo admin)
// ============================================================

async function loadCurrentUser() {
    try {
        const res = await fetch('/api/current-user');
        if (res.ok) {
            const data = await res.json();
            if (data && data.username) {
                const usernameEl = document.getElementById('currentUsername');
                if (usernameEl) usernameEl.textContent = data.username;
                window._currentUserId = data.id;
                const adminLink = document.getElementById('adminUsersLink');
                if (adminLink) {
                    adminLink.style.display = data.rol === 'admin' ? 'block' : 'none';
                }
            }
        }
    } catch (e) {
        console.warn('Error cargando usuario actual:', e);
    }
}

function toggleUserDropdown() {
    const menu = document.getElementById('userDropdownMenu');
    if (menu.style.display === 'block') {
        menu.style.display = 'none';
    } else {
        menu.style.display = 'block';
    }
}

document.addEventListener('click', function(e) {
    const dropdown = document.querySelector('.user-dropdown');
    if (dropdown && !dropdown.contains(e.target)) {
        document.getElementById('userDropdownMenu').style.display = 'none';
    }
});

async function loadUsers() {
    const tbody = document.getElementById('usersTableBody');
    if (!tbody) return;
    tbody.innerHTML = '<tr><td colspan="3" style="text-align:center; padding:20px; color:var(--text-muted);">Cargando...</td></tr>';
    try {
        const res = await fetch('/api/users');
        if (!res.ok) {
            const err = await res.json();
            throw new Error(err.error || 'Error al cargar usuarios');
        }
        const users = await res.json();
        if (users.length === 0) {
            tbody.innerHTML = '<tr><td colspan="3" style="text-align:center; padding:20px; color:var(--text-muted);">No hay usuarios registrados.</td></tr>';
            return;
        }
        let html = '';
        users.forEach(u => {
            const isCurrent = (u.id === window._currentUserId);
            html += `
                <tr>
                    <td style="padding: 8px 10px;">${escapeHtml(u.username)} ${isCurrent ? ' <span style="font-size:0.7rem; color:var(--primary);">(tú)</span>' : ''}</td>
                    <td style="padding: 8px 10px;">
                        <select class="user-rol-select" data-userid="${u.id}" ${isCurrent ? 'disabled' : ''} style="padding: 4px 8px; border-radius: 4px; border: 1px solid var(--border);">
                            <option value="user" ${u.rol === 'user' ? 'selected' : ''}>Usuario</option>
                            <option value="admin" ${u.rol === 'admin' ? 'selected' : ''}>Administrador</option>
                        </select>
                    </td>
                    <td style="padding: 8px 10px; text-align: center;">
                        <button class="btn btn-outline" style="padding: 4px 12px; font-size: 0.8rem;" onclick="changePassword('${u.id}', '${escapeHtml(u.username)}')">🔑 Cambiar contraseña</button>
                        ${!isCurrent ? `<button class="btn btn-danger" style="padding: 4px 12px; font-size: 0.8rem;" onclick="deleteUser('${u.id}', '${escapeHtml(u.username)}')">🗑️</button>` : ''}
                    </td>
                </tr>
            `;
        });
        tbody.innerHTML = html;
        document.querySelectorAll('.user-rol-select').forEach(sel => {
            sel.addEventListener('change', async function() {
                const userId = this.dataset.userid;
                const newRol = this.value;
                try {
                    const resp = await fetch(`/api/users/${userId}`, {
                        method: 'PUT',
                        headers: { 'Content-Type': 'application/json' },
                        body: JSON.stringify({ rol: newRol })
                    });
                    const data = await resp.json();
                    if (resp.ok) {
                        showToast('✅ Rol actualizado');
                        loadUsers();
                    } else {
                        showToast('❌ Error: ' + (data.error || ''));
                        this.value = this.dataset.oldRol || 'user';
                    }
                } catch(e) {
                    showToast('Error de conexión');
                }
            });
            sel.dataset.oldRol = sel.value;
        });
    } catch (e) {
        tbody.innerHTML = `<tr><td colspan="3" style="text-align:center; padding:20px; color:var(--danger);">Error: ${escapeHtml(e.message)}</td></tr>`;
    }
}

async function createUser() {
    const username = document.getElementById('newUsername').value.trim();
    const password = document.getElementById('newPassword').value.trim();
    const rol = document.getElementById('newRol').value;
    if (!username || !password) {
        showToast('⚠️ Completa usuario y contraseña');
        return;
    }
    if (password.length < 6) {
        showToast('⚠️ La contraseña debe tener al menos 6 caracteres');
        return;
    }
    try {
        const res = await fetch('/api/users', {
            method: 'POST',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({ username, password, rol })
        });
        const data = await res.json();
        if (res.ok) {
            showToast(`✅ Usuario "${username}" creado`);
            document.getElementById('newUsername').value = '';
            document.getElementById('newPassword').value = '';
            loadUsers();
        } else {
            showToast('❌ Error: ' + (data.error || ''));
        }
    } catch(e) {
        showToast('Error de conexión');
    }
}

function changePassword(userId, username) {
    const newPass = prompt(`Nueva contraseña para "${username}" (mínimo 6 caracteres):`);
    if (newPass === null) return;
    if (newPass.length < 6) {
        showToast('⚠️ La contraseña debe tener al menos 6 caracteres');
        return;
    }
    if (!confirm(`¿Estás seguro de cambiar la contraseña de "${username}"?`)) return;
    fetch(`/api/users/${userId}`, {
        method: 'PUT',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ password: newPass })
    })
    .then(res => res.json())
    .then(data => {
        if (res.ok) {
            showToast('✅ Contraseña actualizada');
        } else {
            showToast('❌ Error: ' + (data.error || ''));
        }
    })
    .catch(() => showToast('Error de conexión'));
}

function deleteUser(userId, username) {
    if (!confirm(`¿Eliminar al usuario "${username}"? Esta acción no se puede deshacer.`)) return;
    fetch(`/api/users/${userId}`, {
        method: 'DELETE'
    })
    .then(res => res.json())
    .then(data => {
        if (res.ok) {
            showToast(`✅ Usuario "${username}" eliminado`);
            loadUsers();
        } else {
            showToast('❌ Error: ' + (data.error || ''));
        }
    })
    .catch(() => showToast('Error de conexión'));
}

function changeOwnPassword() {
    const newPass = prompt('Nueva contraseña (mínimo 6 caracteres):');
    if (newPass === null) return;
    if (newPass.length < 6) {
        showToast('⚠️ La contraseña debe tener al menos 6 caracteres');
        return;
    }
    if (!confirm('¿Estás seguro de cambiar tu contraseña?')) return;
    fetch(`/api/users/${window._currentUserId}`, {
        method: 'PUT',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ password: newPass })
    })
    .then(res => res.json())
    .then(data => {
        if (res.ok) {
            showToast('✅ Contraseña actualizada');
        } else {
            showToast('❌ Error: ' + (data.error || ''));
        }
    })
    .catch(() => showToast('Error de conexión'));
}

async function deleteAllData() {
    if (!confirm('⚠️ ¿Estás seguro de que quieres eliminar TODOS los datos de la base de datos? Esta acción no se puede deshacer.')) return;
    if (!confirm('⚠️ Confirmación final: ¿Eliminar todas las variables?')) return;
    try {
        const response = await fetch('/api/variables/delete-all', {
            method: 'DELETE',
            headers: { 'Content-Type': 'application/json' }
        });
        const data = await response.json();
        if (response.ok) {
            showToast(`✅ ${data.deleted} registros eliminados.`);
            await loadData(); // recargar tabla
        } else {
            showToast('❌ Error: ' + (data.error || 'No se pudo eliminar'));
        }
    } catch (e) {
        showToast('Error de conexión: ' + e.message);
    }
}


// ============================================================
// INICIALIZACIÓN
// ============================================================

window.onload = async () => {
    currentSearchTerm = '';
    currentPage = 1;
    itemsPerPage = parseInt(document.getElementById('itemsPerPageSelect').value) || 10;
    sortField = 'id';
    sortDir = 'asc';
    await loadData(currentPage, itemsPerPage, sortField, sortDir, '');
    setTimeout(() => {
        document.querySelectorAll('#dataTable tbody tr').forEach(tr => syncRowHeights(tr));
    }, 500);
    renderHeaders();
    document.getElementById('itemsPerPageSelect').value = itemsPerPage;
    await loadHistory();
    await loadCurrentUser();

    // Event listeners para paginación en vista de previsualización CSV
    const prevPageBtn = document.getElementById('prevPageBtn');
    const nextPageBtn = document.getElementById('nextPageBtn');
    if (prevPageBtn) {
        prevPageBtn.addEventListener('click', function() {
            previewPrevPage('uploadDataPreview', false);
        });
    }
    if (nextPageBtn) {
        nextPageBtn.addEventListener('click', function() {
            previewNextPage('uploadDataPreview', false);
        });
    }
    // Event listeners para paginación en vista de previsualización ZIP
    const zipPrevBtn = document.getElementById('zipPrevPageBtn');
    const zipNextBtn = document.getElementById('zipNextPageBtn');
    if (zipPrevBtn) {
        zipPrevBtn.addEventListener('click', function() {
            previewPrevPage('zipPreviewTable', true);
        });
    }
    if (zipNextBtn) {
        zipNextBtn.addEventListener('click', function() {
            previewNextPage('zipPreviewTable', true);
        });
    }

    // Listener para input de página en ZIP (ya se maneja en previewUpdateControls)
    // Pero por si acaso, también lo manejamos aquí para el input de la primera carga
    const zipPageInput = document.getElementById('zipPageInput');
    if (zipPageInput && !zipPageInput._listenerAdded) {
        // Ya se agregará en previewUpdateControls, pero por si acaso:
        zipPageInput.addEventListener('change', function() {
            const newPage = parseInt(this.value);
            const totalPages = Math.ceil(totalParents / parentsPerPage) || 1;
            if (!isNaN(newPage) && newPage >= 1 && newPage <= totalPages) {
                goToPreviewPage(newPage, 'zipPreviewTable', true);
            } else {
                this.value = currentParentPage;
            }
        });
        zipPageInput._listenerAdded = true;
    }

    // Cerrar modales con Escape
    document.addEventListener('keydown', e => {
        if (e.key === 'Escape') {
            if (document.getElementById('modalOtraOpcion').style.display === 'flex') {
                handleCloseOtraOpcionModal();
            } else if (document.getElementById('modalSuggestion').style.display === 'flex') {
                const btn = document.getElementById('btnDeclineSuggestion');
                if (btn) btn.click();
                closeModal('modalSuggestion');
            } else if (document.getElementById('modalConfirmUpload').style.display === 'flex') {
                closeModal('modalConfirmUpload');
            } else if (document.getElementById('modalFolderUpload').style.display === 'flex') {
                closeModal('modalFolderUpload');
            } else if (document.getElementById('modalUserManagement').style.display === 'flex') {
                closeModal('modalUserManagement');
            }
        }
    });

    // Sincronizar con sidebar
    const activeView = document.querySelector('.view-container.active');
    if (activeView) {
        const viewId = activeView.id.replace('view-', '');
        const item = document.querySelector(`.sidebar-item[data-view="${viewId}"]`);
        if (item) {
            document.querySelectorAll('.sidebar-item').forEach(i => i.classList.remove('active'));
            item.classList.add('active');
        }
    }
};


// ============================================================
// NAVEGACIÓN POR VISTAS
// ============================================================

function toggleViewMenu() {
    const menu = document.getElementById('viewDropdownMenu');
    menu.style.display = menu.style.display === 'block' ? 'none' : 'block';
}

// Cerrar menú al hacer clic fuera
document.addEventListener('click', function(e) {
    const dropdown = document.querySelector('.header-menu .dropdown');
    if (dropdown && !dropdown.contains(e.target)) {
        document.getElementById('viewDropdownMenu').style.display = 'none';
    }
});

// ============================================================
// NAVEGACIÓN POR VISTAS (unificada con sidebar)
// ============================================================

function switchView(view, linkElement) {
    // 1. Ocultar todas las vistas
    document.querySelectorAll('.view-container').forEach(el => el.classList.remove('active'));

    // 2. Mostrar la vista seleccionada
    const targetView = document.getElementById('view-' + view);
    if (targetView) targetView.classList.add('active');

    // 3. Actualizar el texto del botón (si existe)
    const labelMap = {
        'variables': '📊 Variables',
        'users': '👥 Usuarios',
        'history': '📋 Cargas'
    };
    const labelEl = document.getElementById('currentViewLabel');
    if (labelEl) labelEl.textContent = labelMap[view] || view;

    // 4. Marcar el enlace activo en el menú desplegable (si existe)
    document.querySelectorAll('#viewDropdownMenu a').forEach(a => a.classList.remove('active'));
    if (linkElement && linkElement.closest('#viewDropdownMenu')) {
        linkElement.classList.add('active');
    }

    // 5. Cerrar el menú desplegable
    const menu = document.getElementById('viewDropdownMenu');
    if (menu) menu.style.display = 'none';

    // 6. Actualizar el sidebar (cerrarlo y marcar el ítem activo)
    closeSidebar();  // función definida en la sección SIDEBAR
    document.querySelectorAll('.sidebar-item').forEach(item => item.classList.remove('active'));
    if (linkElement && linkElement.closest('.sidebar')) {
        linkElement.classList.add('active');
    } else {
        // Si no se pasó linkElement, buscar por data-view
        const item = document.querySelector(`.sidebar-item[data-view="${view}"]`);
        if (item) item.classList.add('active');
    }

    // 7. Cargar datos específicos de la vista
    if (view === 'users') {
        loadUsers();
    } else if (view === 'history') {
        loadHistory();
    }
}

// ============================================================
// SIDEBAR
// ============================================================

function toggleSidebar() {
    const sidebar = document.getElementById('sidebar');
    const overlay = document.getElementById('sidebarOverlay');
    sidebar.classList.toggle('open');
    overlay.classList.toggle('active');
}

function closeSidebar() {
    document.getElementById('sidebar').classList.remove('open');
    document.getElementById('sidebarOverlay').classList.remove('active');
}

// Event listeners
document.addEventListener('DOMContentLoaded', function() {
    document.getElementById('menuToggleBtn').addEventListener('click', toggleSidebar);
    document.getElementById('sidebarOverlay').addEventListener('click', closeSidebar);
    document.getElementById('closeSidebarBtn').addEventListener('click', closeSidebar);
});

// Al cargar, asegurar que el sidebar tenga el item activo según la vista actual
document.addEventListener('DOMContentLoaded', function() {
    // Si ya hay una vista activa, marcar el item correspondiente
    const activeView = document.querySelector('.view-container.active');
    if (activeView) {
        const viewId = activeView.id.replace('view-', '');
        const item = document.querySelector(`.sidebar-item[data-view="${viewId}"]`);
        if (item) {
            document.querySelectorAll('.sidebar-item').forEach(i => i.classList.remove('active'));
            item.classList.add('active');
        }
    }
});
"""

In [11]:
HTML_TEMPLATE = (
    '<!DOCTYPE html>\n'
    '<html lang="es">\n'
    '<head>\n'
    '    <meta charset="UTF-8">\n'
    '    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n'
    '    <title>Gestor de Variables</title>\n'
    '    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap" rel="stylesheet">\n'
    '    <style>\n'
    + CSS +
    '\n    </style>\n'
    '</head>\n'
    '<body>\n'
    + HTML_BODY +
    '\n    <script>\n'
    + JS +
    '\n    </script>\n'
    '</body>\n'
    '</html>'
)

## Módulo 9: Aplicación y modelos ML

In [12]:
# ============================================================
# MÓDULO 9: APLICACIÓN FLASK Y MODELOS ML
# ============================================================

def normalizar_texto(texto: str) -> str:
    """Normaliza un texto eliminando acentos y caracteres no alfanuméricos."""
    if not texto:
        return ""
    texto = str(texto)
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r'[^a-zA-Z0-9\s]', '', texto)
    return texto.lower().strip()


def puntuar_coincidencias(texto: str, lista_palabras: List[str]) -> int:
    """Calcula puntuación de coincidencia entre texto y palabras clave."""
    texto_norm = normalizar_texto(texto)
    if not texto_norm:
        return 0
    score = 0
    for palabra in lista_palabras:
        if normalizar_texto(palabra) in texto_norm:
            score += 1
    return score


def predecir_por_palabras_clave(texto: str) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    """Predice proceso, eje y tema usando coincidencia de palabras clave."""
    if not texto:
        return None, None, None
    mejor_proceso = None
    mejor_puntaje_proceso = 0
    for categoria, palabras in KEYWORDS['proceso'].items():
        score = puntuar_coincidencias(texto, palabras)
        if score > mejor_puntaje_proceso:
            mejor_puntaje_proceso = score
            mejor_proceso = categoria
    if mejor_puntaje_proceso == 0:
        mejor_proceso = list(KEYWORDS['proceso'].keys())[0]

    mejor_eje = None
    mejor_puntaje_eje = 0
    for categoria, palabras in KEYWORDS['eje'].items():
        score = puntuar_coincidencias(texto, palabras)
        if score > mejor_puntaje_eje:
            mejor_puntaje_eje = score
            mejor_eje = categoria
    if mejor_puntaje_eje == 0:
        mejor_eje = list(KEYWORDS['eje'].keys())[0]

    mejor_tema = None
    mejor_puntaje_tema = 0
    for categoria, palabras in KEYWORDS['tema'].items():
        score = puntuar_coincidencias(texto, palabras)
        if score > mejor_puntaje_tema:
            mejor_puntaje_tema = score
            mejor_tema = categoria
    if mejor_puntaje_tema == 0:
        mejor_tema = list(KEYWORDS['tema'].keys())[0]

    return mejor_proceso, mejor_eje, mejor_tema


def clean_text(text: str) -> str:
    """Limpia texto para modelos ML (elimina acentos, caracteres no alfanuméricos)."""
    if not text:
        return ""
    text = str(text)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.lower().strip()


def load_models() -> None:
    """Carga los modelos ML desde disco si existen."""
    global _models_loaded, _model_proceso, _model_eje, _model_tema
    try:
        if Path(MODEL_PROCESO_PATH).exists() and Path(MODEL_EJE_PATH).exists() and Path(MODEL_TEMA_PATH).exists():
            with open(MODEL_PROCESO_PATH, 'rb') as f:
                _model_proceso = pickle.load(f)
            with open(MODEL_EJE_PATH, 'rb') as f:
                _model_eje = pickle.load(f)
            with open(MODEL_TEMA_PATH, 'rb') as f:
                _model_tema = pickle.load(f)
            _models_loaded = True
            logger.info("✅ Modelos ML cargados correctamente.")
        else:
            _models_loaded = False
            logger.warning("⚠️ Modelos ML no encontrados. Usando fallback por palabras clave.")
    except Exception as e:
        _models_loaded = False
        logger.error(f"❌ Error al cargar modelos: {e}. Usando fallback por palabras clave.")


def train_models() -> Dict:
    """Entrena los modelos ML con los datos de Supabase y guarda los modelos."""
    try:
        for path in [MODEL_PROCESO_PATH, MODEL_EJE_PATH, MODEL_TEMA_PATH]:
            if Path(path).exists():
                shutil.copy2(path, path + '.bak')
                logger.info(f"📦 Backup guardado: {path}.bak")

        res = supabase.from_('variables').select('nombre, proceso, eje, tema').execute()
        if not res.data:
            return {"status": "error", "message": "No hay datos etiquetados para entrenar."}

        df = pd.DataFrame(res.data)
        df = df.dropna(subset=['nombre', 'proceso', 'eje', 'tema'])
        df = df[df['nombre'].str.strip() != '']

        if len(df) < 10:
            return {"status": "error", "message": f"Solo hay {len(df)} filas etiquetadas. Se necesitan al menos 10."}

        def is_consistent(row: pd.Series) -> bool:
            eje_num = re.search(r'Eje\s*(\d+)', str(row['eje']))
            tema_num = re.search(r'^(\d+)\.', str(row['tema']))
            if eje_num and tema_num:
                return eje_num.group(1) == tema_num.group(1)
            return False

        df_consistent = df[df.apply(is_consistent, axis=1)]
        if len(df_consistent) == 0:
            return {"status": "error", "message": "No hay filas con coherencia entre Eje y Tema. Corrige manualmente antes de reentrenar."}

        if len(df_consistent) < len(df):
            logger.warning(f"⚠️ Se descartaron {len(df) - len(df_consistent)} filas inconsistentes.")

        df = df_consistent
        df['nombre_limpio'] = df['nombre'].apply(clean_text)
        X = df['nombre_limpio']

        modelos = {
            'proceso': (df['proceso'], MODEL_PROCESO_PATH),
            'eje': (df['eje'], MODEL_EJE_PATH),
            'tema': (df['tema'], MODEL_TEMA_PATH)
        }
        resultados = {}

        for nombre_col, (y, path) in modelos.items():
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            pipeline = Pipeline([
                ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')),
                ('clf', MultinomialNB())
            ])
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            report = classification_report(y_test, y_pred, output_dict=True)
            accuracy = report['accuracy']
            with open(path, 'wb') as f:
                pickle.dump(pipeline, f)
            resultados[nombre_col] = {
                'accuracy': accuracy,
                'samples': len(y_test)
            }

        load_models()
        return {
            "status": "success",
            "message": f"Modelos reentrenados con {len(df)} filas consistentes.",
            "report": resultados
        }
    except Exception as e:
        return {"status": "error", "message": str(e), "traceback": traceback.format_exc()}

def rellenar_categorias(row_data: Dict, debug_log: Optional[List[str]] = None, force: bool = False) -> Dict:
    """
    Rellena las categorías de una fila usando predicción si faltan.
    Si force=False y ya existen y son válidas, no hace nada.
    """
    if debug_log is None:
        debug_log = []

    # Normalizar año y valor
    for col in ['año', 'valor']:
        if col in row_data and row_data[col] is not None and row_data[col] != '':
            row_data[col] = normalizar_numero_texto(row_data[col], col)
            if force:
                debug_log.append(f"🔢 Normalizado {col}: {row_data[col]}")

    # Asegurar que 'nombre' no esté vacío
    nombre = row_data.get('nombre')
    if not nombre or str(nombre).strip() == '':
        nombre = row_data.get('fuente') or f"Variable {row_data.get('id', 'sin_id')}"
        row_data['nombre'] = nombre
        if force:
            debug_log.append(f"⚠️ Nombre vacío, se usó '{nombre}'")

    # Si no se fuerza y ya están todas las categorías y son válidas, salir
    if not force:
        proc = row_data.get('proceso')
        eje = row_data.get('eje')
        tema = row_data.get('tema')
        if proc and eje and tema:
            if proc in KEYWORDS['proceso'] and eje in KEYWORDS['eje'] and tema in KEYWORDS['tema']:
                return row_data  # ya están correctas

    # Si falta alguna, predecir
    if (not row_data.get('proceso') or not row_data.get('eje') or not row_data.get('tema')):
        proc, eje, tema = predict_categories(nombre)
        if proc and not row_data.get('proceso'):
            row_data['proceso'] = proc
            if force:
                debug_log.append(f"✅ Proceso asignado: '{proc}'")
        if eje and not row_data.get('eje'):
            row_data['eje'] = eje
            if force:
                debug_log.append(f"✅ Eje asignado: '{eje}'")
        if tema and not row_data.get('tema'):
            row_data['tema'] = tema
            if force:
                debug_log.append(f"✅ Tema asignado: '{tema}'")

    # Forzar valores por defecto si aún faltan
    if not row_data.get('proceso'):
        row_data['proceso'] = list(KEYWORDS['proceso'].keys())[0]
    if not row_data.get('eje'):
        row_data['eje'] = list(KEYWORDS['eje'].keys())[0]
    if not row_data.get('tema'):
        row_data['tema'] = list(KEYWORDS['tema'].keys())[0]

    # Normalizar categorías (solo si force o si es necesario)
    for col in ['proceso', 'eje', 'tema']:
        if row_data.get(col):
            original = row_data[col]
            valor_normalizado, _ = normalizar_categoria(original, col, debug=force)
            if valor_normalizado != original:
                row_data[col] = valor_normalizado
                if force:
                    debug_log.append(f"🔁 Normalizado {col}: '{original}' -> '{valor_normalizado}'")

    if force:
        debug_log.append(f"📌 Datos finales: proceso='{row_data.get('proceso')}', eje='{row_data.get('eje')}', tema='{row_data.get('tema')}'")
    return row_data

## Módulo 10: Rutas de la API

In [13]:
# ============================================================
# MÓDULO 10: RUTAS DE LA API
# ============================================================

# Variables para tareas asíncronas
_tasks = {}
_task_lock = threading.Lock()

def _build_search_filter(search_term: str) -> str | None:
    if not search_term:
        return None
    cleaned = re.sub(r'[,.:;]', ' ', search_term)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    if not cleaned:
        return None
    safe = cleaned.replace('%', '\\%').replace('_', '\\_')
    text_cols = [
        'proceso', 'eje', 'tema', 'nombre',
        'institucion', 'cobertura', 'periodicidad', 'liga_web',
        'fuente', 'estado'
    ]
    conditions = [f"{col}.ilike.%{safe}%" for col in text_cols]
    return ','.join(conditions)

def _apply_batch_actions(actions: List[Dict], debug_log: Optional[List[str]] = None) -> Tuple[int, int, List[Dict], List[Dict]]:
    if debug_log is None:
        debug_log = []
    _refresh_caches(force=True)
    existing_ids = _get_all_existing_ids()
    used_ids = set(existing_ids)

    # Separar padres e hijos
    parents = [a for a in actions if a.get('_is_parent', False)]
    children = [a for a in actions if not a.get('_is_parent', False)]

    # Función auxiliar para procesar un grupo (padres o hijos) y devolver filas a upsert
    def process_group(group_actions, is_parent):
        rows_to_upsert = []
        parent_id_map = {}  # solo para padres, para mapear hash -> id
        for action in group_actions:
            action_type = action.get('action', 'insert')
            row_data = action['data'].copy()
            supabase_id = action.get('_supabase_matching_id')

            # Asignar parent_id según el tipo
            if is_parent:
                # Los padres siempre deben tener parent_id = None
                row_data['parent_id'] = None
            else:
                # Hijos: obtener el parent_id del mapa o de la acción
                parent_hash = action.get('_parent_hash')
                parent_id_from_action = action.get('_parent_id')
                if parent_id_from_action is not None:
                    row_data['parent_id'] = parent_id_from_action
                elif parent_hash and parent_hash in parent_id_map:
                    row_data['parent_id'] = parent_id_map[parent_hash]
                else:
                    # Si no se encuentra, se deja None (pero luego se intentará reasignar)
                    row_data['parent_id'] = None

            # Rellenar categorías (solo si faltan o se fuerza)
            row_data = rellenar_categorias(row_data, debug_log, force=False)

            if action_type == 'insert':
                if 'id' not in row_data or not row_data['id']:
                    new_id = generar_siguiente_id(used_ids)
                    row_data['id'] = new_id
                    used_ids.add(new_id)
                rows_to_upsert.append(row_data)
                if is_parent:
                    parent_id_map[action.get('_hash')] = row_data['id']
            elif action_type == 'overwrite' and supabase_id:
                row_data['id'] = supabase_id
                rows_to_upsert.append(row_data)
                if is_parent:
                    parent_id_map[action.get('_hash')] = supabase_id
            else:
                debug_log.append(f"⚠️ Acción '{action_type}' ignorada para {action.get('_hash')}")
        return rows_to_upsert, parent_id_map

    # Procesar padres (primero para tener el mapa de IDs)
    parent_rows, parent_id_map = process_group(parents, True)
    # Procesar hijos (usando el mapa de IDs de padres)
    child_rows, _ = process_group(children, False)

    # Ahora, para los hijos que hayan quedado con parent_id None,
    # intentar reasignar usando el parent_id que tengan en su acción (si existe)
    # y que no se haya podido en el primer paso.
    for i, child_row in enumerate(child_rows):
        if child_row.get('parent_id') is None:
            # Buscar la acción original para este hijo
            # (asumimos que el orden se mantiene)
            child_action = children[i] if i < len(children) else None
            if child_action:
                parent_id_from_action = child_action.get('_parent_id')
                if parent_id_from_action is not None:
                    child_row['parent_id'] = parent_id_from_action
                    debug_log.append(f"🔧 Reasignado parent_id {parent_id_from_action} a hijo {child_row.get('id')}")
                else:
                    # Si no hay parent_id, lo dejamos None (será un padre sin hijos, pero eso no debería ocurrir)
                    debug_log.append(f"⚠️ Hijo sin parent_id: {child_row.get('id')}")

    all_rows_to_upsert = parent_rows + child_rows

    inserted_rows = []
    updated_rows = []
    BATCH_SIZE = 1000  # ← aumentado de 50 a 1000

    for i in range(0, len(all_rows_to_upsert), BATCH_SIZE):
        batch = all_rows_to_upsert[i:i+BATCH_SIZE]
        try:
            res = supabase.from_('variables').upsert(batch, on_conflict='id').execute()
            if res.data:
                for row in res.data:
                    if row['id'] in existing_ids:
                        updated_rows.append(row)
                    else:
                        inserted_rows.append(row)
                        existing_ids.add(row['id'])
                debug_log.append(f"✅ Lote upserted: {len(res.data)} filas")
        except Exception as e:
            debug_log.append(f"❌ Error en upsert: {e}")

    _refresh_caches(force=True)
    total_inserted = len(inserted_rows)
    total_updated = len(updated_rows)

    # Depuración: contar cuántos registros tienen parent_id NULL
    try:
        count_null = supabase.from_('variables').select('count', count='exact').is_('parent_id', 'null').execute().count
        debug_log.append(f"📊 Registros con parent_id NULL después de la inserción: {count_null}")
    except:
        pass

    debug_log.append(f"📊 Resumen: {total_inserted} insertados, {total_updated} actualizados.")
    return total_inserted, total_updated, inserted_rows, updated_rows

def _process_zip_file(zip_bytes: bytes, filename: str, log_messages: List[str]) -> Tuple[pd.DataFrame, Dict, List[Dict]]:
    log_messages.append(f"📦 Procesando ZIP: {filename}")
    with tempfile.TemporaryDirectory() as tmpdir:
        zip_path = os.path.join(tmpdir, 'uploaded.zip')
        with open(zip_path, 'wb') as f:
            f.write(zip_bytes)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(tmpdir)

        conjunto_path = None
        diccionario_path = None
        metadatos_path = None

        for root, dirs, files in os.walk(tmpdir):
            rel_path = os.path.relpath(root, tmpdir)
            if 'conjunto_de_datos' in rel_path or 'conjunto_datos' in rel_path:
                if any(f.endswith('.csv') for f in files):
                    conjunto_path = root
            if 'diccionario_de_datos' in rel_path or 'diccionario_datos' in rel_path:
                if any(f.endswith('.csv') for f in files):
                    diccionario_path = root
            if 'metadatos' in rel_path:
                txt_files = [f for f in files if f.endswith('.txt')]
                if txt_files:
                    metadatos_path = os.path.join(root, txt_files[0])

        if not conjunto_path or not diccionario_path:
            csv_files = [f for f in os.listdir(tmpdir) if f.endswith('.csv')]
            if csv_files:
                if not conjunto_path:
                    conjunto_path = tmpdir
                    log_messages.append("📁 Se usará la raíz como 'conjunto_de_datos'")
                if not diccionario_path:
                    for root, dirs, files in os.walk(tmpdir):
                        if 'diccionario' in os.path.basename(root).lower():
                            if any(f.endswith('.csv') for f in files):
                                diccionario_path = root
                                break
                    if not diccionario_path:
                        diccionario_path = tmpdir
                        log_messages.append("📁 Se usará la raíz como 'diccionario_de_datos'")

        if not conjunto_path or not os.path.isdir(conjunto_path):
            log_messages.append("❌ No se encontró carpeta 'conjunto_de_datos' con CSV")
            raise ValueError("No se encontró la carpeta 'conjunto_de_datos' con archivos CSV")

        if not diccionario_path or not os.path.isdir(diccionario_path):
            log_messages.append("❌ No se encontró carpeta 'diccionario_de_datos' con CSV")
            raise ValueError("No se encontró la carpeta 'diccionario_de_datos' con archivos CSV")

        log_messages.append(f"✅ Carpetas encontradas: conjunto={conjunto_path}, diccionario={diccionario_path}, metadatos={metadatos_path}")

        data_dir = Path(conjunto_path)
        dict_dir = Path(diccionario_path)
        meta_path = Path(metadatos_path) if metadatos_path else None

        new_format = False
        indice_filename = None
        if data_dir.is_dir():
            for f in data_dir.iterdir():
                if f.is_file() and f.suffix == '.csv':
                    f_norm = unicodedata.normalize('NFKD', f.name.lower()).encode('ascii', 'ignore').decode('utf-8')
                    if 'indice' in f_norm:
                        new_format = True
                        indice_filename = f.name
                        log_messages.append(f"📌 Formato nuevo detectado: archivo índice '{indice_filename}'")
                        break

        aggregation_keywords = ['total', 'totales', 'cantidad', 'sum'] if new_format else ['cantidad de', 'total']

        final_df, global_metadata, filtered_row_status = process_data_folders_with_return(
            data_dir=data_dir,
            dict_dir=dict_dir,
            metadatos_path=meta_path,
            new_format=new_format,
            indice_filename=indice_filename,
            aggregation_keywords=aggregation_keywords,
            log_messages=log_messages,
            parallel=True
        )
        return final_df, global_metadata, filtered_row_status

# ======================================================================
# VARIABLES GLOBALES DE LA APLICACIÓN FLASK Y MODELOS
# ======================================================================

app = Flask(__name__)
app.secret_key = os.urandom(24)
Compress(app)
app.config['MAX_CONTENT_LENGTH'] = 200 * 1024 * 1024

# Inicializar LoginManager
login_manager = LoginManager()
login_manager.init_app(app)
login_manager.login_view = 'login'
login_manager.login_message = 'Por favor inicia sesión para acceder.'

@login_manager.user_loader
def load_user(user_id):
    return User.get(user_id)

# ============== PÁGINA DE LOGIN MODERNA ==============
LOGIN_PAGE = '''
<!DOCTYPE html>
<html>
<head>
    <title>Iniciar sesión - Gestor de Variables</title>
    <style>
        * { box-sizing: border-box; margin: 0; padding: 0; }
        body {
            font-family: 'Inter', 'Segoe UI', sans-serif;
            background: #F8F9FA;
            display: flex;
            justify-content: center;
            align-items: center;
            height: 100vh;
            margin: 0;
            padding: 20px;
        }
        .login-container {
            background: white;
            border-radius: 16px;
            box-shadow: 0 8px 30px rgba(0,0,0,0.12);
            padding: 48px 40px;
            width: 100%;
            max-width: 420px;
            transition: transform 0.3s ease;
        }
        .login-container:hover {
            transform: translateY(-2px);
        }
        .login-header {
            text-align: center;
            margin-bottom: 32px;
        }
        .login-header h1 {
            font-size: 1.8rem;
            font-weight: 700;
            color: #2C3E50;
            letter-spacing: -0.02em;
        }
        .login-header h1 span {
            color: #E72000;
            font-weight: 300;
        }
        .login-header p {
            color: #6C757D;
            font-size: 0.95rem;
            margin-top: 6px;
        }
        .form-group {
            margin-bottom: 20px;
        }
        .form-group label {
            display: block;
            font-size: 0.85rem;
            font-weight: 600;
            color: #2C3E50;
            margin-bottom: 6px;
        }
        .input-wrapper {
            position: relative;
            display: flex;
            align-items: center;
            background: #f1f3f5;
            border-radius: 8px;
            transition: background 0.2s, box-shadow 0.2s;
            border: 1px solid transparent;
        }
        .input-wrapper:focus-within {
            background: white;
            border-color: #E72000;
            box-shadow: 0 0 0 3px rgba(231,32,0,0.15);
        }
        .input-wrapper input {
            width: 100%;
            padding: 12px 16px;
            border: none;
            background: transparent;
            font-size: 1rem;
            outline: none;
            font-family: inherit;
            color: #212529;
        }
        .input-wrapper input::placeholder {
            color: #adb5bd;
        }
        .toggle-password {
            position: absolute;
            right: 12px;
            background: none;
            border: none;
            cursor: pointer;
            font-size: 1.2rem;
            color: #6C757D;
            padding: 4px 8px;
            border-radius: 4px;
            transition: color 0.2s;
            line-height: 1;
        }
        .toggle-password:hover {
            color: #212529;
        }
        .error-message {
            background: #fff5f5;
            border-left: 4px solid #E72000;
            padding: 12px 16px;
            border-radius: 6px;
            margin-bottom: 20px;
            color: #E72000;
            font-size: 0.9rem;
            display: flex;
            align-items: center;
            gap: 8px;
        }
        .error-message::before {
            content: "⚠️";
            font-size: 1.2rem;
        }
        .login-btn {
            width: 100%;
            padding: 14px;
            background: #E72000;
            color: white;
            border: none;
            border-radius: 8px;
            font-size: 1rem;
            font-weight: 600;
            cursor: pointer;
            transition: background 0.2s, transform 0.1s;
            font-family: inherit;
            letter-spacing: 0.02em;
        }
        .login-btn:hover {
            background: #cc1c00;
        }
        .login-btn:active {
            transform: scale(0.97);
        }
        .login-footer {
            text-align: center;
            margin-top: 24px;
            font-size: 0.85rem;
            color: #6C757D;
        }
        .login-footer a {
            color: #E72000;
            text-decoration: none;
            font-weight: 500;
        }
        .login-footer a:hover {
            text-decoration: underline;
        }
        @media (max-width: 480px) {
            .login-container { padding: 32px 24px; }
        }
    </style>
</head>
<body>
    <div class="login-container">
        <div class="login-header">
            <h1>Gestor <span>Variables</span></h1>
            <p>Inicia sesión para continuar</p>
        </div>
        <form method="post">
            <div class="form-group">
                <label for="username">Usuario</label>
                <div class="input-wrapper">
                    <input type="text" id="username" name="username" placeholder="ej. admin" required autofocus>
                </div>
            </div>
            <div class="form-group">
                <label for="password">Contraseña</label>
                <div class="input-wrapper">
                    <input type="password" id="password" name="password" placeholder="••••••••" required>
                    <button type="button" class="toggle-password" onclick="togglePasswordVisibility()" aria-label="Mostrar/ocultar contraseña">
                        👁️
                    </button>
                </div>
            </div>
            {% if error %}
            <div class="error-message">{{ error }}</div>
            {% endif %}
            <button type="submit" class="login-btn">Iniciar sesión</button>
        </form>
        <div class="login-footer">
            ¿Olvidaste tu contraseña? Contacta al administrador.
        </div>
    </div>
    <script>
        function togglePasswordVisibility() {
            const input = document.getElementById('password');
            const btn = document.querySelector('.toggle-password');
            if (input.type === 'password') {
                input.type = 'text';
                btn.textContent = '🙈';
            } else {
                input.type = 'password';
                btn.textContent = '👁️';
            }
        }
    </script>
</body>
</html>
'''

@app.route('/login', methods=['GET', 'POST'])
def login():
    if DISABLE_AUTH:
        return redirect('/')
    if request.method == 'POST':
        username = request.form.get('username')
        password = request.form.get('password')
        user = User.find_by_username(username)
        if user and user.check_password(password):
            login_user(user)
            return redirect('/')
        else:
            return render_template_string(LOGIN_PAGE, error='Usuario o contraseña incorrectos')
    return render_template_string(LOGIN_PAGE, error=None)

@app.route('/logout')
@login_required
def logout():
    logout_user()
    return redirect('/login')

# ============================================================
# Decoradores de autenticación
# ============================================================

def optional_login_required(func):
    """
    Decorador que exige autenticación solo si DISABLE_AUTH es False.
    Si el usuario no está autenticado, devuelve un error JSON (401)
    en lugar de redirigir al login.
    """
    if DISABLE_AUTH:
        return func
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if not current_user.is_authenticated:
            # Responder con JSON en lugar de redirigir
            return jsonify({"error": "No autenticado. Inicia sesión para acceder a este recurso."}), 401
        return func(*args, **kwargs)
    return wrapper

def admin_required(func):
    """Decorador que exige rol de administrador."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if DISABLE_AUTH:
            return func(*args, **kwargs)
        if not current_user.is_authenticated or not current_user.is_admin():
            return jsonify({"error": "Acceso denegado: se requiere rol de administrador"}), 403
        return func(*args, **kwargs)
    return wrapper

# ============================================================
# RUTAS DE LA API (protegidas con optional_login_required o admin_required)
# ============================================================

MODEL_PROCESO_PATH = 'modelo_proceso.pkl'
MODEL_EJE_PATH = 'modelo_eje.pkl'
MODEL_TEMA_PATH = 'modelo_tema.pkl'
UMBRAL_CONFIANZA = 0.92

_models_loaded = False
_model_proceso = None
_model_eje = None
_model_tema = None

_preview_cache = {}
_PREVIEW_EXPIRE_SECONDS = 600

def _clean_old_preview_cache() -> None:
    now = time.time()
    expired = [k for k, v in _preview_cache.items() if now - v['timestamp'] > _PREVIEW_EXPIRE_SECONDS]
    for k in expired:
        del _preview_cache[k]

# ============================================================
# RUTAS DE LA API (protegidas con optional_login_required)
# ============================================================

@app.route('/')
@login_required
def indice() -> str:
    try:
        js_col_defs = []
        for name in COLUMN_DISPLAY_NAMES:
            key = remover_acentos(name.lower().replace(" ", "_"))
            if name == "ID": key = "id"
            elif name == "Liga Web": key = "liga_web"
            elif name == "Año": key = "año"
            elif name == "Valor": key = "valor"
            info = {"displayName": name, "keyName": key, "type": "input", "readonly": (key == 'id')}
            if key in DYNAMIC_OPTIONS_COLUMNS:
                info["type"] = "select"
                try:
                    opts_res = supabase.from_(key).select('name').execute()
                    info["options"] = sorted({r['name'] for r in opts_res.data if r.get('name')}) if opts_res.data else []
                except Exception:
                    info["options"] = []
            js_col_defs.append(info)

        headers_html = "".join(f"<th><div class='th-content'>{name}</div></th>" for name in COLUMN_DISPLAY_NAMES)
        headers_html = "<th style='width:40px;'></th>" + headers_html
        script_block = f"<script>window.COLUMN_DEFINITIONS = {json.dumps(js_col_defs).replace('</script>', '<\\\\\\\\\\\\\\/script>')};</script>"
        res_html = HTML_TEMPLATE.replace('<tr id="headerRow"></tr>', f'<tr id="headerRow">{headers_html}</tr>')
        res_html = res_html.replace("</head>", f"{script_block}</head>")
        return render_template_string(res_html)
    except Exception as e:
        logger.error(f"Error al renderizar el índice: {e}")
        traceback.print_exc()
        return f"Error al renderizar el índice: {str(e)}", 500

@app.route('/api/variables', methods=['GET', 'POST'])
@optional_login_required
def api_vars():
    if request.method == 'GET':
        try:
            page = request.args.get('page', 1, type=int)
            page_size = request.args.get('page_size', 10, type=int)
            sort_field = request.args.get('sort_field', 'id')
            sort_dir = request.args.get('sort_dir', 'asc')
            search_term = request.args.get('search', '').strip()

            allowed_fields = EXPECTED_COLUMNS + ['valor']
            if sort_field not in allowed_fields:
                sort_field = 'id'
            if sort_dir not in ['asc', 'desc']:
                sort_dir = 'asc'

            start = (page - 1) * page_size
            end = start + page_size - 1

            query = supabase.from_('variables').select('*', count='exact').is_('parent_id', 'null')
            if search_term:
                filter_conditions = _build_search_filter(search_term)
                if filter_conditions:
                    query = query.or_(filter_conditions)

            if sort_dir == 'asc':
                query = query.order(sort_field)
            else:
                query = query.order(sort_field, desc=True)

            query = query.range(start, end)
            res_parents = query.execute()
            total_parents = res_parents.count if res_parents.count is not None else 0
            parent_data = res_parents.data if res_parents.data else []

            children = []
            if parent_data:
                parent_ids = [row['id'] for row in parent_data]
                child_query = supabase.from_('variables').select('*').in_('parent_id', parent_ids)
                child_query = child_query.order('id', desc=False)
                child_res = child_query.execute()
                children = child_res.data if child_res.data else []

            normalized = []
            for parent in parent_data:
                row = {}
                for k, v in parent.items():
                    if k in ['año', 'valor']:
                        v = normalizar_numero_texto(v, k)
                    row[k] = safe_clean(v)
                row['parent_id'] = None
                row['_is_parent'] = True
                row['_children'] = [c['id'] for c in children if c['parent_id'] == parent['id']]
                normalized.append(row)
                for child in children:
                    if child['parent_id'] == parent['id']:
                        child_row = {}
                        for k, v in child.items():
                            if k in ['año', 'valor']:
                                v = normalizar_numero_texto(v, k)
                            child_row[k] = safe_clean(v)
                        child_row['_is_child'] = True
                        normalized.append(child_row)

            total_pages = (total_parents + page_size - 1) // page_size if total_parents > 0 else 1
            return jsonify({
                'data': normalized,
                'total': total_parents,
                'page': page,
                'page_size': page_size,
                'total_pages': total_pages
            })
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    elif request.method == 'POST':
        try:
            existing_ids = _get_all_existing_ids()
            new_id = generar_siguiente_id()
            data = request.json or {}
            new_row = {
                "id": new_id,
                "proceso": data.get('proceso', ''),
                "eje": data.get('eje', ''),
                "tema": data.get('tema', ''),
                "nombre": data.get('nombre', ''),
                "institucion": data.get('institucion', ''),
                "cobertura": data.get('cobertura', ''),
                "periodicidad": data.get('periodicidad', ''),
                "liga_web": data.get('liga_web', ''),
                "fuente": data.get('fuente', ''),
                "año": normalizar_numero_texto(data.get('año', ''), 'año'),
                "valor": normalizar_numero_texto(data.get('valor', ''), 'valor'),
                "estado": data.get('estado', '')
            }
            if new_row['nombre']:
                proc, eje, tema = predict_categories(new_row['nombre'])
                if proc and not new_row['proceso']:
                    new_row['proceso'] = proc
                if eje and not new_row['eje']:
                    new_row['eje'] = eje
                if tema and not new_row['tema']:
                    new_row['tema'] = tema
            new_row['año'] = normalizar_numero_texto(new_row['año'], 'año')
            new_row['valor'] = normalizar_numero_texto(new_row['valor'], 'valor')
            res = supabase.from_('variables').insert(new_row).execute()
            if res.data:
                _refresh_caches()
                return jsonify(res.data[0]), 201
            else:
                return jsonify({"error": "Error al crear variable"}), 500
        except Exception as e:
            return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

@app.route('/api/variables/<id_val>', methods=['PUT', 'DELETE'])
@optional_login_required
def api_item(id_val):
    if request.method == 'PUT':
        try:
            data = request.json
            app.logger.info(f"📥 Recibido PUT para {id_val}: {data}")
            existing = supabase.from_('variables').select('*').eq('id', id_val).execute()
            if not existing.data:
                return jsonify({"error": "Variable no encontrada"}), 404
            current = existing.data[0]
            cleaned_data = {}
            for k, v in data.items():
                if k in EXPECTED_COLUMNS:
                    cleaned_data[k] = str(v) if not pd.isna(v) else None
            if 'nombre' in cleaned_data and cleaned_data['nombre']:
                nombre_nuevo = cleaned_data['nombre']
                if not current.get('proceso') or current.get('proceso') == '' or \
                   not current.get('eje') or current.get('eje') == '' or \
                   not current.get('tema') or current.get('tema') == '':
                    proc, eje, tema = predict_categories(nombre_nuevo)
                    if proc and (not current.get('proceso') or current.get('proceso') == ''):
                        cleaned_data['proceso'] = proc
                    if eje and (not current.get('eje') or current.get('eje') == ''):
                        cleaned_data['eje'] = eje
                    if tema and (not current.get('tema') or current.get('tema') == ''):
                        cleaned_data['tema'] = tema
            if 'año' in cleaned_data:
                cleaned_data['año'] = normalizar_numero_texto(cleaned_data['año'], 'año')
            if 'valor' in cleaned_data:
                cleaned_data['valor'] = normalizar_numero_texto(cleaned_data['valor'], 'valor')
            res = supabase.from_('variables').update(cleaned_data).eq('id', id_val).execute()
            if res.data:
                _refresh_caches()
                return jsonify({"status": "ok"}), 200
            else:
                return jsonify({"error": "Error al actualizar variable"}), 500
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    elif request.method == 'DELETE':
        try:
            res = supabase.from_('variables').delete().eq('id', id_val).execute()
            if res.data and len(res.data) > 0:
                _refresh_caches()
                return jsonify({"status": "ok"}), 200
            else:
                return jsonify({"error": "Registro no encontrado"}), 404
        except Exception as e:
            return jsonify({"error": str(e)}), 500

@app.route('/api/variables/<id_val>/duplicate', methods=['POST'])
@optional_login_required
def duplicate_var(id_val):
    try:
        res = supabase.from_('variables').select('*').eq('id', id_val).execute()
        if not res.data:
            return jsonify({"error": "Variable no encontrada"}), 404
        original_row = res.data[0]
        duplicated_row = original_row.copy()
        existing_ids = _get_all_existing_ids()
        new_id = generar_siguiente_id()
        duplicated_row['id'] = new_id
        duplicated_row['año'] = normalizar_numero_texto(duplicated_row['año'], 'año')
        duplicated_row['valor'] = normalizar_numero_texto(duplicated_row['valor'], 'valor')
        insert_res = supabase.from_('variables').insert(duplicated_row).execute()
        if insert_res.data:
            _refresh_caches()
            return jsonify(insert_res.data[0]), 201
        else:
            return jsonify({"error": "Error al duplicar variable"}), 500
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/catalog/<table_name>', methods=['POST'])
@optional_login_required
def add_catalog_option(table_name):
    try:
        data = request.json
        value = data.get('value', '').strip()
        if not value:
            return jsonify({"error": "Valor no puede estar vacío"}), 400
        all_options = _get_catalog_options(table_name)
        match_val, score = _find_best_value_match(value, all_options, threshold=0.95)
        if score == 1.0:
            return jsonify({"status": "already_exists", "value": match_val}), 200
        elif score > 0.7:
            return jsonify({"status": "suggestion", "original_input": value, "suggested_value": match_val}), 200
        else:
            res = supabase.from_(table_name).insert({"name": value}).execute()
            if res.data:
                return jsonify({"status": "inserted", "value": res.data[0]['name']}), 201
            else:
                return jsonify({"error": "Error al agregar opción de catálogo"}), 500
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/download')
@optional_login_required
def download_csv():
    try:
        data = _get_all_variables_data()
        df = pd.DataFrame(data)
        if df.empty:
            csv_output = ",".join(COLUMN_DISPLAY_NAMES) + "\n"
        else:
            df_display = df[EXPECTED_COLUMNS].copy()
            display_name_map = {remover_acentos(name.lower().replace(" ", "_")): name for name in COLUMN_DISPLAY_NAMES}
            df_display.rename(columns=display_name_map, inplace=True)
            for col_name in COLUMN_DISPLAY_NAMES:
                if col_name not in df_display.columns:
                    df_display[col_name] = ''
            df_display = df_display[COLUMN_DISPLAY_NAMES]
            csv_output = df_display.to_csv(index=False, encoding='utf-8-sig')
        response = make_response(csv_output)
        now = datetime.datetime.now()
        fecha_hora = now.strftime('%Y%m%d%H%M%S')
        filename = f"variables_{fecha_hora}.csv"
        response.headers["Content-Disposition"] = f"attachment; filename={filename}"
        response.headers["Content-type"] = "text/csv; charset=utf-8"
        return response
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/upload', methods=['POST'])
@optional_login_required
def upload_csv():
    try:
        if 'file' not in request.files:
            return jsonify({"error": "No se encontró el archivo"}), 400
        file = request.files['file']
        if file.filename == '' or not file.filename.endswith('.csv'):
            return jsonify({"error": "Archivo inválido. Debe ser CSV."}), 400

        file_content = file.read()
        file_stream = io.BytesIO(file_content)
        df = None
        messages = []

        try:
            df = pd.read_csv(file_stream, encoding='utf-8-sig', dtype=str, low_memory=False)
            messages.append("CSV procesado correctamente (UTF-8).")
        except UnicodeDecodeError:
            file_stream.seek(0)
            try:
                df = pd.read_csv(file_stream, encoding='latin-1', dtype=str, low_memory=False)
                messages.append("CSV procesado correctamente (Latin-1).")
            except Exception as e:
                raise ValueError(f"No se pudo decodificar el CSV: {e}")
        except Exception as e:
            raise ValueError(f"Error al leer el CSV: {e}")

        app.logger.info("📋 Columnas leídas del CSV:", list(df.columns))

        year_aliases = ['año', 'ano', 'anio', 'Año', 'AÑO']
        year_col_found = None
        for col in df.columns:
            col_clean = remover_acentos(str(col).lower().strip().replace(' ', '_'))
            if col_clean in year_aliases or col_clean == 'año':
                year_col_found = col
                break
        if year_col_found:
            if year_col_found != 'año':
                df.rename(columns={year_col_found: 'año'}, inplace=True)
                messages.append(f"Columna '{year_col_found}' renombrada a 'año' por alias.")
        else:
            messages.append("⚠️ No se encontró columna de año. Se dejará vacía.")
            df['año'] = pd.NA

        rename_map = {}
        for col in df.columns:
            norm_col = _normalize_col_for_matching(col)
            if norm_col in EXPECTED_COLUMNS:
                rename_map[col] = norm_col
        if rename_map:
            df.rename(columns=rename_map, inplace=True)
            messages.append("Nombres de columnas normalizados.")

        _refresh_caches(force=True)
        existing_data = _get_all_variables_data()
        processed_rows, process_msgs = _process_csv_data(df, existing_data, debug=True)
        messages.extend(process_msgs)

        catalog_columns = ['proceso', 'eje', 'tema', 'institucion', 'cobertura', 'periodicidad', 'estado']
        for row_info in processed_rows:
            row_data = row_info['data']
            nuevo_en_catalogo = {}
            for col in catalog_columns:
                if col in row_data and row_data[col]:
                    valor = str(row_data[col]).strip()
                    if valor:
                        if col in ['proceso', 'eje', 'tema']:
                            opciones = list(KEYWORDS[col].keys())
                            match, score = _find_best_value_match(valor, opciones, threshold=1.0)
                            if match is None:
                                nuevo_en_catalogo[col] = True
                        else:
                            opciones = _get_catalog_options(col)
                            match, score = _find_best_value_match(valor, opciones, threshold=1.0)
                            if match is None:
                                nuevo_en_catalogo[col] = True
            row_info['_nuevo_en_catalogo'] = nuevo_en_catalogo

        def priority_key(row):
            is_dup = row.get('_is_duplicate', False)
            has_new = any(row.get('_nuevo_en_catalogo', {}).values())
            return (0 if is_dup else 1 if has_new else 2)
        processed_rows.sort(key=priority_key)

        if request.args.get('preview') == 'true':
            cleaned = _clean_data_for_json(processed_rows)
            return jsonify({"status": "preview", "messages": messages, "preview_data": cleaned}), 200

        actions_str = request.form.get('actions_for_rows', '[]')
        actions = json.loads(actions_str)
        debug_log = []
        inserted, updated, inserted_data, updated_data = _apply_batch_actions(actions, debug_log)
        _refresh_caches()

        try:
            estado = 'completado' if inserted > 0 or updated > 0 else 'error'
            supabase.from_('cargas').insert({
                'fecha': datetime.datetime.now().isoformat(),
                'archivo': file.filename,
                'filas_agregadas': inserted,
                'estado': estado
            }).execute()
        except Exception as e:
            app.logger.info(f"Error al guardar historial: {e}")

        msg = f"Se insertaron {inserted} filas nuevas."
        if updated > 0:
            msg += f" Se actualizaron {updated} filas existentes."

        inserted_clean = _clean_data_for_json(inserted_data)
        updated_clean = _clean_data_for_json(updated_data)
        return jsonify({
            "status": "ok",
            "message": msg,
            "debug": debug_log,
            "inserted": inserted_clean,
            "updated": updated_clean
        }), 200
    except Exception as e:
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

@app.route('/api/process-folder', methods=['POST'])
@optional_login_required
def process_folder():
    log_messages = []
    try:
        if 'file' not in request.files:
            return jsonify({"error": "No se encontró el archivo ZIP"}), 400
        file = request.files['file']
        if file.filename == '' or not file.filename.endswith('.zip'):
            return jsonify({"error": "El archivo debe ser un ZIP"}), 400

        zip_bytes = file.read()
        filename = file.filename

        final_df, global_metadata, filtered_row_status = _process_zip_file(zip_bytes, filename, log_messages)

        if final_df.empty:
            log_messages.append("⚠️ No se generaron filas a partir de los archivos. Verifica que los CSV tengan columnas de totales y que el diccionario esté bien formado.")
            return jsonify({
                "status": "preview",
                "messages": log_messages,
                "preview_data": [],
                "global_metadata": {},
                "zip_filename": filename,
                "total_rows": 0,
                "preview_token": None
            }), 200

        _refresh_caches(force=True)
        existing_data = _get_all_variables_data()

        preview_raw = final_df.to_dict(orient='records')
        df_temp = pd.DataFrame(preview_raw)
        for col in EXPECTED_COLUMNS:
            if col not in df_temp.columns:
                df_temp[col] = ''
        df_temp = df_temp[EXPECTED_COLUMNS]

        processed_rows_flat, dup_messages = _process_csv_data(df_temp, existing_data, debug=True)
        log_messages.extend(dup_messages)

        hash_map = {}
        for flat_row in processed_rows_flat:
            h = flat_row.get('_hash')
            if h:
                hash_map[h] = flat_row

        for row_status in filtered_row_status:
            h = row_status.get('_hash')
            if h and h in hash_map:
                flat_info = hash_map[h]
                row_status['_is_duplicate'] = flat_info.get('_is_duplicate', False)
                row_status['_supabase_matching_id'] = flat_info.get('_supabase_matching_id')
                row_status['_nuevo_en_catalogo'] = flat_info.get('_nuevo_en_catalogo', {})
            else:
                row_status['_is_duplicate'] = False
                row_status['_supabase_matching_id'] = None
                row_status['_nuevo_en_catalogo'] = {}

        log_messages.append("🔍 Muestra de filtered_row_status (3 primeros):")
        for i, r in enumerate(filtered_row_status[:3]):
            log_messages.append(
                f"  {i}: _is_parent={r.get('_is_parent')}, _is_child={r.get('_is_child')}, "
                f"_hash={r.get('_hash')}, _parent_hash={r.get('_parent_hash')}"
            )

        def priority_key(row):
            is_dup = row.get('_is_duplicate', False)
            has_new = any(row.get('_nuevo_en_catalogo', {}).values())
            return (0 if is_dup else 1 if has_new else 2)
        filtered_row_status.sort(key=priority_key)

        preview_token = str(uuid.uuid4())
        _preview_cache[preview_token] = {
            'timestamp': time.time(),
            'processed_rows': filtered_row_status,
            'total_rows': len(filtered_row_status),
            'global_metadata': global_metadata,
            'zip_filename': filename,
            'log_messages': log_messages
        }
        _clean_old_preview_cache()

        preview_data_all = _clean_data_for_json(filtered_row_status)
        global_metadata_clean = _clean_data_for_json(global_metadata)
        log_messages_clean = _clean_data_for_json(log_messages)

        return jsonify({
            "status": "preview",
            "messages": log_messages_clean,
            "preview_data": preview_data_all,
            "global_metadata": global_metadata_clean,
            "zip_filename": filename,
            "preview_token": preview_token,
            "total_rows": len(filtered_row_status),
            "page": 1,
            "page_size": len(filtered_row_status)
        }), 200

    except Exception as e:
        return jsonify({
            "error": str(e),
            "traceback": traceback.format_exc(),
            "messages": log_messages
        }), 500

@app.route('/api/confirm-folder', methods=['POST'])
@optional_login_required
def confirm_folder():
    try:
        data = request.json
        token = data.get('token')
        overwrite_hashes = data.get('overwrite_hashes', [])

        if not token or token not in _preview_cache:
            return jsonify({"error": "Token inválido o expirado"}), 404

        cache_entry = _preview_cache[token]
        processed_rows = cache_entry['processed_rows']
        zip_filename = cache_entry.get('zip_filename', 'carga_carpeta')

        actions = []
        for idx, row_info in enumerate(processed_rows):
            action = 'insert'
            if row_info.get('_is_duplicate', False):
                row_hash = row_info.get('_hash')
                if row_hash and row_hash in overwrite_hashes:
                    action = 'overwrite'
                else:
                    action = 'ignore'
            # 🔧 CAMBIO: incluir _parent_id si existe
            actions.append({
                'original_csv_index': row_info.get('original_csv_index', idx),
                'action': action,
                'data': row_info['data'],
                '_is_duplicate': row_info.get('_is_duplicate', False),
                '_supabase_matching_id': row_info.get('_supabase_matching_id'),
                '_hash': row_info.get('_hash'),
                '_is_parent': row_info.get('_is_parent', False),
                '_parent_id': row_info.get('_parent_id')  # <--- NUEVO
            })

        debug_log = []
        _refresh_caches(force=True)
        inserted, updated, inserted_data, updated_data = _apply_batch_actions(actions, debug_log)
        _refresh_caches()

        estado = 'completado' if inserted > 0 or updated > 0 else 'error'
        try:
            supabase.from_('cargas').insert({
                'fecha': datetime.datetime.now().isoformat(),
                'archivo': zip_filename,
                'filas_agregadas': inserted,
                'estado': estado
            }).execute()
        except Exception as e:
            app.logger.info(f"Error al guardar historial: {e}")

        if inserted == 0 and updated == 0:
            error_detail = ""
            for log in debug_log:
                if "ERROR" in log or "❌" in log:
                    error_detail = f" (Error: {log})"
                    break
            msg = f"No se realizaron cambios. Revisa los logs de depuración{error_detail}."
        else:
            msg = f"Se insertaron {inserted} filas nuevas."
            if updated > 0:
                msg += f" Se actualizaron {updated} filas existentes."

        inserted_clean = _clean_data_for_json(inserted_data)
        updated_clean = _clean_data_for_json(updated_data)

        _preview_cache.pop(token, None)

        return jsonify({
            "status": "ok",
            "message": msg,
            "debug": debug_log,
            "inserted": inserted_clean,
            "updated": updated_clean
        }), 200

    except Exception as e:
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

@app.route('/api/retrain-models', methods=['POST'])
@optional_login_required
def retrain_models():
    try:
        result = train_models()
        if result.get('status') == 'success':
            return jsonify({"status": "ok", "message": result['message'], "report": result.get('report')}), 200
        else:
            return jsonify({"error": result.get('message', 'Error al entrenar')}), 500
    except Exception as e:
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

@app.route('/api/rollback-models', methods=['POST'])
@optional_login_required
def rollback_models():
    try:
        restored = []
        for path in [MODEL_PROCESO_PATH, MODEL_EJE_PATH, MODEL_TEMA_PATH]:
            backup_path = path + '.bak'
            if os.path.exists(backup_path):
                shutil.copy2(backup_path, path)
                restored.append(path)
        if restored:
            load_models()
            return jsonify({"status": "ok", "message": f"Restaurados: {', '.join(restored)}"}), 200
        else:
            return jsonify({"error": "No hay backups disponibles"}), 404
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/delete-models', methods=['DELETE'])
@optional_login_required
def delete_models():
    try:
        deleted = []
        for path in [MODEL_PROCESO_PATH, MODEL_EJE_PATH, MODEL_TEMA_PATH]:
            if os.path.exists(path):
                os.remove(path)
                deleted.append(path)
        global _models_loaded
        _models_loaded = False
        return jsonify({"status": "ok", "message": f"Eliminados: {', '.join(deleted)}"}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/upload-history', methods=['GET'])
@optional_login_required
def upload_history():
    try:
        res = supabase.from_('cargas').select('*').order('fecha', desc=True).limit(50).execute()
        return jsonify(res.data if res.data else [])
    except Exception as e:
        app.logger.info(f"Error al obtener historial: {e}")
        return jsonify([])

@app.route('/api/preview-page', methods=['POST'])
@optional_login_required
def preview_page():
    try:
        data = request.json
        token = data.get('token')
        page = int(data.get('page', 1))
        page_size = int(data.get('page_size', 50))
        if not token or token not in _preview_cache:
            return jsonify({"error": "Token inválido o expirado"}), 404

        cache_entry = _preview_cache[token]
        processed_rows = cache_entry['processed_rows']
        total_rows = len(processed_rows)
        total_pages = (total_rows + page_size - 1) // page_size if total_rows > 0 else 1
        if page < 1:
            page = 1
        if page > total_pages:
            page = total_pages

        start = (page - 1) * page_size
        end = min(start + page_size, total_rows)
        page_data = processed_rows[start:end]

        cleaned_page = _clean_data_for_json(page_data)
        return jsonify({
            "preview_data": cleaned_page,
            "page": page,
            "total_pages": total_pages,
            "total_rows": total_rows,
            "page_size": page_size
        }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/variables/bulk-delete', methods=['DELETE'])
@optional_login_required
def bulk_delete_vars():
    try:
        data = request.json
        ids = data.get('ids', [])
        if not ids:
            return jsonify({"error": "No IDs provided"}), 400
        res = supabase.from_('variables').delete().in_('id', ids).execute()
        _refresh_caches()
        return jsonify({"status": "ok", "deleted": len(res.data) if res.data else 0}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/variables/bulk-duplicate', methods=['POST'])
@optional_login_required
def bulk_duplicate_vars():
    try:
        data = request.json
        ids = data.get('ids', [])
        if not ids:
            return jsonify({"error": "No IDs provided"}), 400
        res = supabase.from_('variables').select('*').in_('id', ids).execute()
        if not res.data:
            return jsonify({"error": "No variables found"}), 404
        duplicated = []
        existing_ids = _get_all_existing_ids()
        used_ids = set(existing_ids)
        for row in res.data:
            new_row = row.copy()
            new_id = generar_siguiente_id()
            new_row['id'] = new_id
            used_ids.add(new_id)
            new_row['año'] = normalizar_numero_texto(new_row['año'], 'año')
            new_row['valor'] = normalizar_numero_texto(new_row['valor'], 'valor')
            duplicated.append(new_row)
        insert_res = supabase.from_('variables').insert(duplicated).execute()
        _refresh_caches()
        return jsonify({"status": "ok", "inserted": insert_res.data if insert_res.data else []}), 201
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/variables/delete-all', methods=['DELETE'])
@optional_login_required
@admin_required
def delete_all_variables():
    """Elimina TODOS los registros de la tabla 'variables' (temporal)."""
    try:
        res = supabase.from_('variables').delete().neq('id', '').execute()
        deleted_count = len(res.data) if res.data else 0
        _refresh_caches(force=True)
        return jsonify({"status": "ok", "deleted": deleted_count}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/catalog/<table_name>', methods=['GET'])
@optional_login_required
def get_catalog_options(table_name):
    try:
        options = _get_catalog_options(table_name)
        return jsonify({"options": options})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/variables/children', methods=['GET'])
@optional_login_required
def get_children():
    try:
        ids_str = request.args.get('ids', '')
        if not ids_str:
            return jsonify([])
        ids = ids_str.split(',')
        res = supabase.from_('variables').select('*').in_('parent_id', ids).execute()
        return jsonify(res.data if res.data else [])
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/process-folder-async', methods=['POST'])
@optional_login_required
def process_folder_async():
    if 'file' not in request.files:
        return jsonify({"error": "No se encontró el archivo ZIP"}), 400
    file = request.files['file']
    if file.filename == '' or not file.filename.endswith('.zip'):
        return jsonify({"error": "El archivo debe ser un ZIP"}), 400

    zip_bytes = file.read()
    filename = file.filename

    task_id = str(uuid.uuid4())
    with _task_lock:
        _tasks[task_id] = {
            'status': 'pending',
            'progress': 0,
            'messages': [],
            'result': None,
            'error': None,
            'filename': filename
        }

    def run_task():
        try:
            log_messages = []
            with _task_lock:
                _tasks[task_id]['status'] = 'running'
                _tasks[task_id]['progress'] = 10

            final_df, global_metadata, filtered_row_status = _process_zip_file(zip_bytes, filename, log_messages)

            if final_df.empty:
                log_messages.append("⚠️ No se generaron filas a partir de los archivos. Verifica que los CSV tengan columnas de totales y que el diccionario esté bien formado.")
                with _task_lock:
                    _tasks[task_id]['status'] = 'completed'
                    _tasks[task_id]['progress'] = 100
                    _tasks[task_id]['result'] = {
                        'preview_token': None,
                        'preview_data': [],
                        'global_metadata': {},
                        'messages': log_messages,
                        'total_rows': 0
                    }
                    _tasks[task_id]['messages'] = log_messages
                return

            _refresh_caches(force=True)
            existing_data = _get_all_variables_data()

            preview_raw = final_df.to_dict(orient='records')
            df_temp = pd.DataFrame(preview_raw)
            for col in EXPECTED_COLUMNS:
                if col not in df_temp.columns:
                    df_temp[col] = ''
            df_temp = df_temp[EXPECTED_COLUMNS]

            processed_rows_flat, dup_messages = _process_csv_data(df_temp, existing_data, debug=True)
            log_messages.extend(dup_messages)

            hash_map = {}
            for flat_row in processed_rows_flat:
                h = flat_row.get('_hash')
                if h:
                    hash_map[h] = flat_row

            for row_status in filtered_row_status:
                h = row_status.get('_hash')
                if h and h in hash_map:
                    flat_info = hash_map[h]
                    row_status['_is_duplicate'] = flat_info.get('_is_duplicate', False)
                    row_status['_supabase_matching_id'] = flat_info.get('_supabase_matching_id')
                    row_status['_nuevo_en_catalogo'] = flat_info.get('_nuevo_en_catalogo', {})
                else:
                    row_status['_is_duplicate'] = False
                    row_status['_supabase_matching_id'] = None
                    row_status['_nuevo_en_catalogo'] = {}

            def priority_key(row):
                is_dup = row.get('_is_duplicate', False)
                has_new = any(row.get('_nuevo_en_catalogo', {}).values())
                return (0 if is_dup else 1 if has_new else 2)
            filtered_row_status.sort(key=priority_key)
            logger.info(f"Ejemplo de filtered_row_status con _is_parent: {filtered_row_status[0] if filtered_row_status else 'empty'}")

            preview_token = str(uuid.uuid4())
            _preview_cache[preview_token] = {
                'timestamp': time.time(),
                'processed_rows': filtered_row_status,
                'total_rows': len(filtered_row_status),
                'global_metadata': global_metadata,
                'zip_filename': filename,
                'log_messages': log_messages
            }
            _clean_old_preview_cache()

            preview_data_all = _clean_data_for_json(filtered_row_status)
            global_metadata_clean = _clean_data_for_json(global_metadata)
            log_messages_clean = _clean_data_for_json(log_messages)

            with _task_lock:
                _tasks[task_id]['status'] = 'completed'
                _tasks[task_id]['progress'] = 100
                _tasks[task_id]['result'] = {
                    'preview_token': preview_token,
                    'global_metadata': global_metadata_clean,
                    'messages': log_messages_clean,
                    'total_rows': len(filtered_row_status)
                }
                _tasks[task_id]['messages'] = log_messages_clean

        except Exception as e:
            import traceback
            error_msg = str(e)
            traceback_str = traceback.format_exc()
            with _task_lock:
                _tasks[task_id]['status'] = 'error'
                _tasks[task_id]['error'] = error_msg
                _tasks[task_id]['messages'].append(traceback_str)
                _tasks[task_id]['progress'] = 100

    thread = threading.Thread(target=run_task)
    thread.start()

    return jsonify({"task_id": task_id}), 202

@app.route('/api/task-status/<task_id>', methods=['GET'])
@optional_login_required
def task_status(task_id):
    with _task_lock:
        if task_id not in _tasks:
            return jsonify({"error": "Task not found"}), 404
        task = _tasks[task_id]
        response = {
            "status": task['status'],
            "progress": task['progress'],
            "messages": task.get('messages', [])[-20:],
        }
        if task['status'] == 'completed':
            response['result'] = task['result']
        elif task['status'] == 'error':
            response['error'] = task['error']
        return jsonify(response)

# Decorador para rutas que solo pueden usar administradores
def admin_required(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if DISABLE_AUTH:
            return func(*args, **kwargs)
        if not current_user.is_authenticated or not current_user.is_admin():
            return jsonify({"error": "Acceso denegado: se requiere rol de administrador"}), 403
        return func(*args, **kwargs)
    return wrapper


@app.route('/api/users', methods=['GET'])
@optional_login_required
@admin_required
def list_users():
    """Lista todos los usuarios (solo admin)"""
    try:
        res = supabase.from_('usuarios').select('id, username, rol').execute()
        return jsonify(res.data if res.data else [])
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/users', methods=['POST'])
@optional_login_required
@admin_required
def create_user():
    """Crea un nuevo usuario (solo admin)"""
    data = request.json
    username = data.get('username', '').strip()
    password = data.get('password', '').strip()
    rol = data.get('rol', 'user')

    if not username or not password:
        return jsonify({"error": "Usuario y contraseña son obligatorios"}), 400
    if len(password) < 6:
        return jsonify({"error": "La contraseña debe tener al menos 6 caracteres"}), 400

    # Verificar si ya existe
    existing = supabase.from_('usuarios').select('id').eq('username', username).execute()
    if existing.data:
        return jsonify({"error": "El nombre de usuario ya existe"}), 400

    hashed = generate_password_hash(password)
    try:
        res = supabase.from_('usuarios').insert({
            'username': username,
            'password_hash': hashed,
            'rol': rol
        }).execute()
        if res.data:
            new_user = res.data[0]
            return jsonify({
                "id": new_user['id'],
                "username": new_user['username'],
                "rol": new_user.get('rol', 'user')
            }), 201
        else:
            return jsonify({"error": "Error al crear usuario"}), 500
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/users/<user_id>', methods=['PUT'])
@optional_login_required
@admin_required
def update_user(user_id):
    """Actualiza un usuario (cambiar contraseña o rol) - solo admin"""
    data = request.json
    updates = {}

    # No permitir cambiar el propio usuario (para evitar autodeshabilitación)
    if user_id == current_user.id:
        return jsonify({"error": "No puedes modificarte a ti mismo desde aquí"}), 403

    # Verificar que el usuario existe
    existing = supabase.from_('usuarios').select('id').eq('id', user_id).execute()
    if not existing.data:
        return jsonify({"error": "Usuario no encontrado"}), 404

    # Cambiar contraseña
    if 'password' in data and data['password']:
        new_pass = data['password'].strip()
        if len(new_pass) < 6:
            return jsonify({"error": "La contraseña debe tener al menos 6 caracteres"}), 400
        updates['password_hash'] = generate_password_hash(new_pass)

    # Cambiar rol
    if 'rol' in data and data['rol'] in ['admin', 'user']:
        updates['rol'] = data['rol']

    if not updates:
        return jsonify({"error": "No se proporcionaron campos para actualizar"}), 400

    try:
        res = supabase.from_('usuarios').update(updates).eq('id', user_id).execute()
        if res.data:
            return jsonify({
                "id": res.data[0]['id'],
                "username": res.data[0]['username'],
                "rol": res.data[0].get('rol', 'user')
            }), 200
        else:
            return jsonify({"error": "Error al actualizar usuario"}), 500
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/users/<user_id>', methods=['DELETE'])
@optional_login_required
@admin_required
def delete_user(user_id):
    """Elimina un usuario (solo admin, no puede eliminarse a sí mismo)"""
    if user_id == current_user.id:
        return jsonify({"error": "No puedes eliminarte a ti mismo"}), 403

    # Verificar que el usuario existe
    existing = supabase.from_('usuarios').select('id').eq('id', user_id).execute()
    if not existing.data:
        return jsonify({"error": "Usuario no encontrado"}), 404

    try:
        res = supabase.from_('usuarios').delete().eq('id', user_id).execute()
        if res.data:
            return jsonify({"message": "Usuario eliminado correctamente"}), 200
        else:
            return jsonify({"error": "Error al eliminar usuario"}), 500
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/current-user', methods=['GET'])
@optional_login_required
def current_user_info():
    if DISABLE_AUTH:
        return jsonify({"id": "disabled", "username": "disabled", "rol": "admin"})
    if current_user.is_authenticated:
        return jsonify({"id": current_user.id, "username": current_user.username, "rol": current_user.rol})
    return jsonify({"error": "No autenticado"}), 401

## Módulo 11: Inicio del servidor Flask con ngrok

In [14]:
# ============================================================
# MÓDULO 11: INICIO DEL SERVIDOR FLASK CON NGROK
# ============================================================

FLASK_PORT = 5000
ngrok.kill()
time.sleep(5)

os.environ['DISABLE_AUTH'] = 'false'   # Cambiar a 'true' para deshabilitar autenticación

def find_available_port(start_port: int) -> int:
    """Encuentra un puerto disponible a partir de start_port."""
    port = start_port
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('0.0.0.0', port))
                return port
            except OSError:
                logger.info(f"Puerto {port} en uso, intentando siguiente...")
                port += 1


available_port = find_available_port(FLASK_PORT)
FLASK_PORT = available_port


def ejecutar_app_flask() -> None:
    """Inicia el servidor Flask."""
    app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False)


hilo_flask = threading.Thread(target=ejecutar_app_flask)
hilo_flask.start()
time.sleep(3)

NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

url_publica = None
MAX_RETRIES = 5
RETRY_DELAY = 10

for attempt in range(MAX_RETRIES):
    try:
        url_publica = ngrok.connect(FLASK_PORT).public_url
        logger.info(f" * Túnel ngrok disponible en: {url_publica} (Flask puerto {FLASK_PORT})")
        print(f"Ngrok Public URL: {url_publica}")
        break
    except PyngrokNgrokHTTPError as e:
        if "ERR_NGROK_334" in str(e) and attempt < MAX_RETRIES - 1:
            logger.warning(f"Advertencia: ngrok ya en línea. Reintentando en {RETRY_DELAY}s... (Intento {attempt+1}/{MAX_RETRIES})")
            time.sleep(RETRY_DELAY)
        else:
            logger.error(f"Error fatal ngrok: {e}")
            raise
    except Exception as e:
        logger.error(f"Error inesperado: {e}")
        raise
else:
    logger.error("Error: No se pudo establecer túnel ngrok.")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Ngrok Public URL: https://unpaid-barterer-erasable.ngrok-free.dev
